# Cox Proportional Hazards Regression Pipeline
## Comprehensive Univariate and Multivariate Analysis

**Version:** 1.3  
**Environment:** Google Colab  
**Last Updated:** December 2025

---

## ⚙️ Key Tunable Parameters Summary

| Parameter | Location | Default | Purpose |
|-----------|----------|---------|--------|
| `PENALIZER` | Cell 1 | `0.1` | Cox model regularization |
| `EPV_MIN_VARS` | Cell 1 | `2` | Min variables in multivariate |
| `EPV_MAX_VARS` | Cell 1 | `4` | Max variables (hard cap) |
| `EPV_DIVISOR` | Cell 1 | `5` | EPV rule (events // divisor) |
| `CV_FOLDS` | Cell 1 | `5` | CV section |
| `CV_REPETITIONS` | Cell 1 | `10` |
| `BASE_SEED` | Cell 1 | `42` |
| `PH2_EPV` | Cell 9.9 | `18` | Events per variable budget |
| `PH2_CV_FOLDS` | Cell 9.9 | `5` | Cross-validation folds |
| `l1_ratio` | Cell 9.9 | `0.1` | Elastic-net mixing ratio |
| `n_groups` | Cell 6 | `3` | Risk stratification groups |
| `VIF_THRESHOLD` | Cell 5 | `5.0` | Multicollinearity threshold |
| `CV_FOLDS` | Cell 10.1 | `5` | QC cross-validation folds |

---

## Table of Contents

1. [Setup and Univariate Analysis](#1-setup-and-univariate-cox-regression-analysis)
2. [Multivariate Model Comparison](#2-multivariate-model-comparison-and-champion-selection)
3. [Enhanced Visualizations](#3-enhanced-multivariate-visualizations-with-cv-stability-analysis)
4. [Quick Diagnostics](#4-quick-diagnostics-check-optional)
5. [Full QC Suite](#5-comprehensive-cox-model-quality-control-qc-suite)
6. [Risk Stratification](#6-risk-group-stratification-and-survival-analysis)
7. [Data Export](#7-comprehensive-data-export)
8. [Patient ID Tracking](#8-patient-id-tracked-risk-stratification-optional)
9. [PH Correction Pipeline](#91-ph-correction-pipeline---environment-setup) (Cells 9.1-9.9)
10. [Champion QC](#101-champion-qc---auto-configuration) (Cells 10.1-10.10)

---

## Requirements

```python
lifelines>=0.27.0      # Survival analysis
scikit-learn>=1.0.0    # Machine learning utilities
pandas>=1.3.0          # Data manipulation
numpy>=1.20.0          # Numerical computing
matplotlib>=3.4.0      # Visualization
seaborn>=0.11.0        # Statistical visualization
openpyxl>=3.0.0        # Excel export
scipy>=1.7.0           # Statistical functions
statsmodels>=0.13.0    # Statistical models
```

---

## Quick Start

1. Run **Cell 1** (Setup) - follow prompts to load your data
2. Run **Cells 2-3** for multivariate model selection
3. Run **Cell 5** for model QC diagnostics
4. Run **Cell 6** for risk stratification
5. Run **Cell 7** to export all results

For PH-corrected penalized models, run Cells 9.1-9.9 after Cell 1.

---


# 1. Setup and Univariate Cox Regression Analysis

## Overview
This cell performs the initial setup and comprehensive univariate Cox proportional hazards regression analysis.

### What this cell does:
1. **Package Installation**: Installs required Python packages (lifelines, scikit-learn, openpyxl, scipy, statsmodels)
2. **Google Drive Integration**: Mounts Google Drive for data input/output
3. **Data Configuration**: Interactive prompts for file path, column selection, and export settings
4. **Univariate Analysis**: Performs Cox regression for each biomarker individually
5. **Statistical Testing**: Log-rank tests with median and quartile stratification
6. **Multivariate Model Building**: Constructs multivariate Cox models using stepwise selection or EPV-capped approaches
7. **Visualization**: Generates Kaplan-Meier curves, forest plots, volcano plots, and correlation heatmaps
8. **Export**: Saves comprehensive results to Excel with multiple sheets

---

## ⚙️ TUNABLE PARAMETERS

### Cox Model Settings
| Parameter | Default | Description |
|-----------|---------|-------------|
| `PENALIZER` | `0.1` | L2 regularization strength (try 0.1-1.0 if unstable) |
| `ROBUST_SE` | `True` | Use robust standard errors in lifelines fits |

---

### 🎯 EPV (Events Per Variable) Settings - IMPORTANT

These parameters control how many variables are included in the multivariate model based on the number of events in your data.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `EPV_MIN_VARS` | `2` | **Minimum** variables to include (floor) |
| `EPV_MAX_VARS` | `4` | **Maximum** variables allowed (hard ceiling) |
| `EPV_DIVISOR` | `5` | EPV rule: max_vars = events // EPV_DIVISOR |

#### How EPV Calculation Works:
```
max_vars = max(EPV_MIN_VARS, min(EPV_MAX_VARS, events // EPV_DIVISOR))
```

#### Example Calculations:
| Events | EPV_DIVISOR=5 | EPV_DIVISOR=10 | EPV_DIVISOR=3 |
|--------|---------------|----------------|---------------|
| 20 | max_vars = 4 | max_vars = 2 | max_vars = 4 |
| 30 | max_vars = 4 | max_vars = 3 | max_vars = 4 |
| 50 | max_vars = 4 | max_vars = 4 | max_vars = 4 |

#### EPV Guidelines:
| Setting | EPV_DIVISOR | Use Case |
|---------|-------------|----------|
| **Conservative** | `10` | Small samples, high-stakes decisions |
| **Standard** | `5` | Default, balanced approach |
| **Aggressive** | `3` | Large samples, exploratory analysis |

> **Tip**: For small datasets (<50 events), use `EPV_DIVISOR = 10` and `EPV_MAX_VARS = 3`

---

### User Input Parameters
| Parameter | Description |
|-----------|-------------|
| `time_unit` | Time unit for survival data (days/months/years) |
| `FILE_PATH` | Path to Excel file with survival data |

---

### Key Outputs:
- `Cox_Regression_Results_{timestamp}.xlsx` - Complete univariate and multivariate results
- Multiple PNG visualizations in `RESULTS_FOLDER/KM_Plots/`

### Multivariate Biomarker Selection Options:
| Option | Description |
|--------|-------------|
| **1. Top N** | Select top N biomarkers by p-value (default) |
| **2. Significant only** | All biomarkers with p < 0.05 |
| **3. Low correlation** | Significant biomarkers with r < 0.7 |
| **4. Manual selection** | Browse ALL biomarkers with pagination, select by number or range (e.g., "1,3,5" or "1-5") |

### Required User Inputs:
- Path to Excel file with survival data
- Selection of duration, event, and biomarker columns
- Time unit (days/months/years)
- Export folder preferences


In [ ]:
# ============================================================================
# UNIVARIATE COX REGRESSION ANALYSIS FOR GOOGLE COLAB
# ============================================================================

# STEP 1: Install required packages (run this cell first)
!pip install scikit-learn -q
!pip install lifelines -q
!pip install openpyxl -q
!pip install scipy -q
!pip install statsmodels -q

# STEP 2: Import libraries
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import k_fold_cross_validation
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
import os
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')

# Global model settings
PENALIZER = 0.1     # try 0.1–1.0 if still unstable
ROBUST_SE = True    # use robust=True in lifelines fits

# ============================================================================
# ⚙️ EPV (Events Per Variable) TUNABLE PARAMETERS
# ============================================================================
EPV_MIN_VARS = 2        # Minimum number of variables to include in multivariate model
EPV_MAX_VARS = 5        # Maximum number of variables (hard cap)
EPV_DIVISOR = 5         # Events per variable rule (events // EPV_DIVISOR)
# Example: With 30 events and EPV_DIVISOR=5, max_vars = min(EPV_MAX_VARS, 30//5) = min(4, 6) = 4
# Increase EPV_DIVISOR for more conservative models (e.g., 10 for EPV=10 rule)
# ============================================================================

# Helper to robustly read attrs across lifelines versions
def _get_attr(obj, names, default=None):
    for n in names:
        if hasattr(obj, n):
            val = getattr(obj, n)
            if val is not None:
                return val
    return default

# STEP 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ============================================================================
# FILE INPUT CONFIGURATION
# ============================================================================

print("=" * 80)
print("📁 DATA INPUT CONFIGURATION")
print("=" * 80)

print("\n📝 Please enter the path to your Excel file in Google Drive")
print("\nExample paths:")
print("  • /content/drive/MyDrive/Baseline univariate COX reg.xlsx")
print("  • /content/drive/MyDrive/data/my_data.xlsx")
print("  • /content/drive/MyDrive/Research/Project1/analysis.xlsx")

FILE_PATH = input("\n🔹 Enter your file path: ").strip()
FILE_PATH = FILE_PATH.strip('"').strip("'")

# ============================================================================
# EXPORT PATH CONFIGURATION
# ============================================================================

print("\n" + "=" * 80)
print("📂 EXPORT PATH CONFIGURATION")
print("=" * 80)

print("\n📝 Where would you like to save the results?")
print("\nOptions:")
print("  1. Same folder as input file (default)")
print("  2. Specific folder in Google Drive")
print("  3. Root of MyDrive")

export_choice = input("\n🔹 Enter your choice (1, 2, or 3) [default: 1]: ").strip()

if export_choice == "2":
    print("\nEnter the folder path where you want to save results:")
    print("Example: /content/drive/MyDrive/Results")
    EXPORT_PATH = input("\n🔹 Export path: ").strip()
elif export_choice == "3":
    EXPORT_PATH = "/content/drive/MyDrive"
else:  # Default to option 1
    EXPORT_PATH = os.path.dirname(FILE_PATH)

# Create results subfolder with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULTS_FOLDER = os.path.join(EXPORT_PATH, f"Cox_Results_{timestamp}")
os.makedirs(RESULTS_FOLDER, exist_ok=True)

print(f"\n✅ Results will be saved to:")
print(f"   {RESULTS_FOLDER}")

# ============================================================================
# VERIFY AND LOAD DATA
# ============================================================================

print("\n" + "=" * 80)
print("📊 LOADING DATA")
print("=" * 80)

# Verify the file exists
if os.path.exists(FILE_PATH):
    print(f"\n✅ File found: {FILE_PATH}")

    try:
        # Load the data
        df = pd.read_excel(FILE_PATH)
        print(f"✅ Data loaded successfully!")
        print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")

    except Exception as e:
        print(f"\n❌ Error loading file: {e}")
        raise
else:
    print(f"\n❌ File not found at: {FILE_PATH}")
    raise FileNotFoundError("Please restart and enter a valid file path")

# ============================================================================
# COLUMN SELECTION
# ============================================================================

print("\n" + "=" * 80)
print("📋 COLUMN SELECTION")
print("=" * 80)

# Display all column names with indices
print("\n📊 Available columns in your data:")
print("-" * 60)
for i, col in enumerate(df.columns, 1):
    print(f"  {i:3}. {col}")

print("\n" + "=" * 80)
print("🔍 IDENTIFY REQUIRED COLUMNS")
print("=" * 80)

# Function to get column selection
def select_column(prompt, columns, example_names=None, allow_skip=False):
    """Helper function to select a column"""
    print(f"\n{prompt}")
    if example_names:
        print(f"Common names: {', '.join(example_names)}")

    if allow_skip:
        print("Enter 0 to skip this column")

    while True:
        try:
            choice = input("\n🔹 Enter column number: ").strip()
            if allow_skip and choice == "0":
                return None

            col_idx = int(choice) - 1
            if 0 <= col_idx < len(columns):
                selected_col = columns[col_idx]
                print(f"   ✓ Selected: '{selected_col}'")
                return selected_col
            else:
                print(f"   ❌ Please enter a number between 1 and {len(columns)}")
        except ValueError:
            print("   ❌ Please enter a valid number")

# Select Patient ID column
print("\n1️⃣ PATIENT ID COLUMN")
print("   This column contains unique identifiers for each patient")
patient_id_col = select_column(
    "Which column contains the Patient ID?",
    df.columns.tolist(),
    example_names=['Probe-ID', 'patient_ID', 'ID', 'Subject_ID', 'PatientID']
)

# Select Survival Time column
print("\n2️⃣ SURVIVAL TIME COLUMN")
print("   This column contains the survival time (in months, days, or years)")
survival_time_col = select_column(
    "Which column contains the Survival Time?",
    df.columns.tolist(),
    example_names=['PFS_days', 'Survival_Time', 'Time', 'OS', 'DFS', 'PFS']
)

# Ask for survival type name
print("\n   What type of survival is this?")
print("   Examples:")
print("   • Overall Survival (OS)")
print("   • Disease-Free Survival (DFS)")
print("   • Progression-Free Survival (PFS)")
print("   • Recurrence-Free Survival (RFS)")
print("   • Event-Free Survival (EFS)")
print("   • Or any custom name")
survival_type_input = input("\n🔹 Enter survival type name [default: Overall Survival]: ").strip()

# Abbreviation logic that preserves acronyms like "PFS"
if survival_type_input:
    survival_type = survival_type_input
    if survival_type_input.isupper() and len(survival_type_input) <= 4:
        survival_abbrev = survival_type_input.upper()
    else:
        survival_abbrev = ''.join(word[0].upper() for word in survival_type.split())
else:
    survival_type = "Overall Survival"
    survival_abbrev = "OS"

print(f"   ✓ Survival type: {survival_type} ({survival_abbrev})")

# Ask for time unit
print("\n   What is the time unit for survival data?")
print("   1. Months")
print("   2. Days")
print("   3. Years")
time_unit_choice = input("\n🔹 Enter choice (1, 2, or 3) [default: 1]: ").strip()
time_unit = "months"  # default
if time_unit_choice == "2":
    time_unit = "days"
elif time_unit_choice == "3":
    time_unit = "years"
print(f"   ✓ Time unit: {time_unit}")

# Select Survival Status column
print("\n3️⃣ SURVIVAL STATUS/EVENT COLUMN")
print("   This column indicates whether the event occurred or patient was censored")
survival_status_col = select_column(
    "Which column contains the Survival Status/Event?",
    df.columns.tolist(),
    example_names=['Survival status', 'Status', 'Event', 'Death', 'event']
)

# Identify unique values in survival status column
unique_status = df[survival_status_col].unique()
print(f"\n   Unique values in {survival_status_col}: {unique_status}")

# Determine how to encode event
if survival_type.upper() == "OVERALL SURVIVAL" or survival_abbrev == "OS":
    print("\n   How is death coded in this column?")
    print("   Please identify which value represents death:")
else:
    print("\n   How is the event coded in this column?")
    print("   Please identify which value represents the event occurrence:")

for i, val in enumerate(unique_status, 1):
    print(f"   {i}. {val}")

death_value = None
while death_value is None:
    try:
        if survival_type.upper() == "OVERALL SURVIVAL" or survival_abbrev == "OS":
            choice = input("\n🔹 Enter the number for death value: ").strip()
        else:
            choice = input("\n🔹 Enter the number for event value: ").strip()
        idx = int(choice) - 1
        if 0 <= idx < len(unique_status):
            death_value = unique_status[idx]
            if survival_type.upper() == "OVERALL SURVIVAL" or survival_abbrev == "OS":
                print(f"   ✓ Death coded as: '{death_value}'")
            else:
                print(f"   ✓ Event coded as: '{death_value}'")
        else:
            print(f"   ❌ Please enter a number between 1 and {len(unique_status)}")
    except ValueError:
        print("   ❌ Please enter a valid number")

# Optional: Ask if there's a specific range of columns for biomarkers
print("\n" + "=" * 80)
print("📊 BIOMARKER COLUMNS")
print("=" * 80)

print("\n4️⃣ BIOMARKER COLUMNS IDENTIFICATION")
print("   Biomarkers are the variables you want to analyze with Cox regression")
print("\nHow are your biomarker columns organized?")
print("   1. All columns after the Patient ID column are biomarkers")
print("   2. Biomarkers are in a specific range of columns")
print("   3. I want to manually select biomarker columns")

biomarker_choice = input("\n🔹 Enter choice (1, 2, or 3) [default: 1]: ").strip()

if biomarker_choice == "2":
    print("\n   Select the FIRST biomarker column:")
    for i, col in enumerate(df.columns, 1):
        print(f"   {i:3}. {col}")

    start_idx = None
    while start_idx is None:
        try:
            choice = input("\n🔹 Enter number for FIRST biomarker column: ").strip()
            start_idx = int(choice) - 1
            if 0 <= start_idx < len(df.columns):
                print(f"   ✓ First biomarker: '{df.columns[start_idx]}'")
            else:
                print(f"   ❌ Please enter a number between 1 and {len(df.columns)}")
                start_idx = None
        except ValueError:
            print("   ❌ Please enter a valid number")

    print("\n   Select the LAST biomarker column:")
    end_idx = None
    while end_idx is None:
        try:
            choice = input("\n🔹 Enter number for LAST biomarker column: ").strip()
            end_idx = int(choice) - 1
            if start_idx <= end_idx < len(df.columns):
                print(f"   ✓ Last biomarker: '{df.columns[end_idx]}'")
            else:
                print(f"   ❌ Please enter a valid number between {start_idx+1} and {len(df.columns)}")
                end_idx = None
        except ValueError:
            print("   ❌ Please enter a valid number")

    biomarker_cols = df.columns[start_idx:end_idx+1].tolist()

elif biomarker_choice == "3":
    print("\n   Enter the column numbers for biomarkers (comma-separated):")
    print("   Example: 4,5,6,7,8")

    biomarker_cols = []
    while not biomarker_cols:
        try:
            choices = input("\n🔹 Enter column numbers: ").strip().split(',')
            for choice in choices:
                idx = int(choice.strip()) - 1
                if 0 <= idx < len(df.columns):
                    biomarker_cols.append(df.columns[idx])
                else:
                    raise ValueError(f"Column number {choice} is out of range")
            print(f"   ✓ Selected {len(biomarker_cols)} biomarker columns")
        except (ValueError, IndexError) as e:
            print(f"   ❌ Error: {e}. Please try again.")
            biomarker_cols = []
else:  # Default option 1
    # All columns after patient ID that aren't the survival columns
    start_idx = df.columns.get_loc(patient_id_col) + 1
    biomarker_cols = df.columns[start_idx:].tolist()

# Remove survival-related columns from biomarker list if present
columns_to_remove = [survival_time_col, survival_status_col, patient_id_col]
biomarker_cols = [col for col in biomarker_cols if col not in columns_to_remove]

# Also remove 'event' column if we're about to create it
if 'event' in biomarker_cols:
    biomarker_cols.remove('event')

print(f"\n✅ Identified {len(biomarker_cols)} biomarker columns")
if len(biomarker_cols) <= 10:
    print("   Biomarkers:", ', '.join(biomarker_cols))
else:
    print(f"   First 5: {', '.join(biomarker_cols[:5])}")
    print(f"   Last 5: {', '.join(biomarker_cols[-5:])}")

# ============================================================================
# DATA PREPARATION
# ============================================================================

print("\n" + "=" * 80)
print("🚀 STARTING ANALYSIS")
print("=" * 80)

# Create event indicator (1 for died/event, 0 for censored)
df['event'] = (df[survival_status_col] == death_value).astype(int)

# Determine terminology based on survival type
if survival_type.upper() == "OVERALL SURVIVAL" or survival_abbrev == "OS":
    event_term = "Deaths"
    censored_term = "Alive (Censored)"
    event_rate_term = "Death Rate (%)"
else:
    event_term = "Events"
    censored_term = "Censored"
    event_rate_term = "Event Rate (%)"

# Coerce biomarker columns to numeric (handle EU commas)
for col in biomarker_cols:
    if df[col].dtype == object:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.'), errors='coerce')

# Dataset summary statistics
dataset_stats = {
    'Total Patients': len(df),
    'Number of Biomarkers': len(biomarker_cols),
    event_term: df['event'].sum(),
    censored_term: len(df) - df['event'].sum(),
    event_rate_term: df['event'].mean() * 100,
    f'Median {survival_abbrev} ({time_unit})': df[survival_time_col].median(),
    f'Mean {survival_abbrev} ({time_unit})': df[survival_time_col].mean(),
    f'Min {survival_abbrev} ({time_unit})': df[survival_time_col].min(),
    f'Max {survival_abbrev} ({time_unit})': df[survival_time_col].max(),
    f'Q1 {survival_abbrev} ({time_unit})': df[survival_time_col].quantile(0.25),
    f'Q3 {survival_abbrev} ({time_unit})': df[survival_time_col].quantile(0.75)
}

print(f"\nDataset Summary:")
print(f"  Number of patients: {dataset_stats['Total Patients']}")
print(f"  Number of biomarkers: {dataset_stats['Number of Biomarkers']}")
print(f"  {event_term}: {dataset_stats[event_term]} ({dataset_stats[event_rate_term]:.1f}%)")
print(f"  Censored: {dataset_stats[censored_term]}")
print(f"  Median {survival_abbrev}: {dataset_stats[f'Median {survival_abbrev} ({time_unit})']:.2f} {time_unit}")
print("=" * 80)

# ============================================================================
# ANALYSIS FUNCTIONS
# ============================================================================

def perform_cox_regression(df, biomarker_col, os_col=survival_time_col, event_col='event'):
    """Perform Cox regression for a single biomarker"""
    try:
        # Prepare data
        data = df[[os_col, event_col, biomarker_col]].dropna()

        if len(data) < 10:
            return None

        # Check if biomarker has variation
        if data[biomarker_col].std() == 0:
            return None

        # Rename biomarker column to avoid special character issues
        data = data.rename(columns={biomarker_col: 'biomarker_value'})

        # Fit Cox model
        cph = CoxPHFitter(penalizer=PENALIZER)
        cph.fit(
            data[[os_col, event_col, 'biomarker_value']],
            duration_col=os_col,
            event_col=event_col,
            formula='biomarker_value',
            robust=ROBUST_SE
        )

        # Extract results (robust AIC/BIC across versions)
        AIC_val = _get_attr(cph, ["AIC_partial_", "AIC_", "AIC"])
        BIC_val = _get_attr(cph, ["BIC_partial_", "BIC_", "BIC"])

        results = {
            'biomarker': biomarker_col,
            'n': len(data),
            'n_events': int(data[event_col].sum()),
            'n_censored': int(len(data) - data[event_col].sum()),
            'coef': cph.params_['biomarker_value'],
            'se_coef': cph.standard_errors_['biomarker_value'],
            'hr': float(np.exp(cph.params_['biomarker_value'])),
            'hr_lower': float(np.exp(cph.confidence_intervals_.iloc[0, 0])),
            'hr_upper': float(np.exp(cph.confidence_intervals_.iloc[0, 1])),
            'p_value': float(cph.summary.loc['biomarker_value', 'p']),
            'z_score': float(cph.summary.loc['biomarker_value', 'z']),
            'concordance': float(cph.concordance_index_),
            'log_likelihood': float(cph.log_likelihood_),
            'AIC': AIC_val,
            'BIC': BIC_val,
            'mean_biomarker': float(data['biomarker_value'].mean()),
            'median_biomarker': float(data['biomarker_value'].median()),
            'std_biomarker': float(data['biomarker_value'].std())
        }
        return results
    except Exception as e:
        return None

def stratify_and_test(df, biomarker_col, cutoff_type='median', os_col=survival_time_col, event_col='event'):
    """Stratify patients by biomarker levels and perform log-rank test"""
    try:
        data = df[[os_col, event_col, biomarker_col]].dropna()

        if len(data) < 10:
            return None

        # Determine cutoff
        if cutoff_type == 'median':
            cutoff = data[biomarker_col].median()
            group_low = data[data[biomarker_col] <= cutoff]
            group_high = data[data[biomarker_col] > cutoff]
            label_low, label_high = 'Low (≤median)', 'High (>median)'
        elif cutoff_type == 'quartiles':
            q25 = data[biomarker_col].quantile(0.25)
            q75 = data[biomarker_col].quantile(0.75)
            group_low = data[data[biomarker_col] <= q25]
            group_high = data[data[biomarker_col] >= q75]
            label_low, label_high = 'Low (Q1)', 'High (Q4)'
            cutoff = f"Q1={q25:.2f}, Q4={q75:.2f}"
        else:
            return None

        if len(group_low) < 5 or len(group_high) < 5:
            return None

        # Perform log-rank test
        lr_result = logrank_test(
            group_low[os_col], group_high[os_col],
            group_low[event_col], group_high[event_col]
        )

        # Calculate hazard ratio (high vs low)
        cph = CoxPHFitter(penalizer=PENALIZER)
        combined_data = pd.concat([
            group_low.assign(group=0),
            group_high.assign(group=1)
        ])
        cph.fit(
            combined_data[[os_col, event_col, 'group']],
            duration_col=os_col, event_col=event_col,
            formula='group',
            robust=ROBUST_SE
        )

        hr = float(np.exp(cph.params_['group']))
        hr_ci_lower = float(np.exp(cph.confidence_intervals_.iloc[0, 0]))
        hr_ci_upper = float(np.exp(cph.confidence_intervals_.iloc[0, 1]))

        return {
            'biomarker': biomarker_col,
            'cutoff_type': cutoff_type,
            'cutoff_value': str(cutoff),
            'n_low': int(len(group_low)),
            'n_high': int(len(group_high)),
            'events_low': int(group_low[event_col].sum()),
            'events_high': int(group_high[event_col].sum()),
            'censored_low': int(len(group_low) - group_low[event_col].sum()),
            'censored_high': int(len(group_high) - group_high[event_col].sum()),
            'median_os_low': float(group_low[os_col].median()),
            'median_os_high': float(group_high[os_col].median()),
            'mean_os_low': float(group_low[os_col].mean()),
            'mean_os_high': float(group_high[os_col].mean()),
            'hr_high_vs_low': hr,
            'hr_ci_lower': hr_ci_lower,
            'hr_ci_upper': hr_ci_upper,
            'logrank_statistic': float(lr_result.test_statistic),
            'logrank_p': float(lr_result.p_value),
            'label_low': label_low,
            'label_high': label_high
        }
    except Exception as e:
        return None

# ============================================================================
# CALCULATE CUTOFF VALUES FOR ALL BIOMARKERS
# ============================================================================

print("\n📊 CALCULATING BIOMARKER CUTOFF VALUES...")

cutoff_data = []
for biomarker in biomarker_cols:
    try:
        biomarker_data = df[biomarker].dropna()
        if len(biomarker_data) >= 10:
            cutoff_info = {
                'biomarker': biomarker,
                'n_samples': int(len(biomarker_data)),
                'min': float(biomarker_data.min()),
                'q1': float(biomarker_data.quantile(0.25)),
                'median': float(biomarker_data.median()),
                'q3': float(biomarker_data.quantile(0.75)),
                'max': float(biomarker_data.max()),
                'mean': float(biomarker_data.mean()),
                'std': float(biomarker_data.std()),
                'iqr': float(biomarker_data.quantile(0.75) - biomarker_data.quantile(0.25)),
                'cv': float(biomarker_data.std() / biomarker_data.mean() * 100) if biomarker_data.mean() != 0 else None
            }
            cutoff_data.append(cutoff_info)
    except Exception:
        continue

cutoff_df = pd.DataFrame(cutoff_data)
print(f"✅ Calculated cutoff values for {len(cutoff_df)} biomarkers")

# ============================================================================
# PERFORM UNIVARIATE COX REGRESSION
# ============================================================================

print("\n📊 Performing univariate Cox regression analysis...")
print("   This may take a minute...")

cox_results = []
for i, biomarker in enumerate(biomarker_cols, 1):
    if i % 20 == 0:
        print(f"   Processing biomarker {i}/{len(biomarker_cols)}...")
    result = perform_cox_regression(df, biomarker)
    if result:
        cox_results.append(result)

# Initialize empty DataFrames
cox_df = pd.DataFrame()
quartile_df = pd.DataFrame()
median_df = pd.DataFrame()

# Process Cox results if available
if len(cox_results) > 0:
    cox_df = pd.DataFrame(cox_results)
    cox_df = cox_df.sort_values('p_value').reset_index(drop=True)
    cox_df['rank'] = range(1, len(cox_df) + 1)
    cox_df['significant_0.05'] = cox_df['p_value'] < 0.05
    cox_df['significant_0.01'] = cox_df['p_value'] < 0.01

    # FDR (Benjamini–Hochberg)
    reject, qvals, _, _ = multipletests(cox_df['p_value'].values, method='fdr_bh')
    cox_df['q_value'] = qvals
    cox_df['significant_FDR_0.10'] = cox_df['q_value'] < 0.10
    cox_df['significant_FDR_0.05'] = cox_df['q_value'] < 0.05

    print(f"\n✅ Cox regression completed for {len(cox_df)} biomarkers")

    # Display top results
    print("\n📈 TOP 20 BIOMARKERS (Univariate Cox Regression)")
    print("=" * 120)
    print(f"{'Rank':<6} {'Biomarker':<40} {'HR':>8} {'95% CI':>20} {'P-value':>10} {'C-index':>10}")
    print("-" * 120)
    n_display = min(20, len(cox_df))
    for i, (_, row) in enumerate(cox_df.head(n_display).iterrows(), 1):
        biomarker_name = row['biomarker'][:39]
        print(f"{i:<6} {biomarker_name:<40} {row['hr']:>8.3f} ({row['hr_lower']:>7.3f}-{row['hr_upper']:>7.3f}) {row['p_value']:>10.4f} {row['concordance']:>10.3f}")
else:
    print("\n⚠️ No Cox regression results available")

# ============================================================================
# QUARTILE ANALYSIS (Q1 vs Q4)
# ============================================================================

print("\n" + "=" * 120)
print("📊 QUARTILE ANALYSIS (Bottom 25% vs Top 25%)")
print("-" * 120)

quartile_results = []
for biomarker in biomarker_cols:
    result = stratify_and_test(df, biomarker, cutoff_type='quartiles')
    if result:
        quartile_results.append(result)

if len(quartile_results) > 0:
    quartile_df = pd.DataFrame(quartile_results)
    quartile_df = quartile_df.sort_values('logrank_p').reset_index(drop=True)
    quartile_df['rank'] = range(1, len(quartile_df) + 1)
    quartile_df['significant_0.05'] = quartile_df['logrank_p'] < 0.05
    quartile_df['significant_0.01'] = quartile_df['logrank_p'] < 0.01

    print(f"\n📈 TOP 20 BIOMARKERS (Q1 vs Q4 Stratification)")
    print("=" * 140)
    print(f"{'Rank':<6} {'Biomarker':<40} {'N (Q1/Q4)':>12} {'Events':>12} {f'Median {survival_abbrev}':>20} {'HR':>8} {'P-value':>10}")
    print(f"{'':6} {'':40} {'':>12} {'(Q1/Q4)':>12} {'(Q1/Q4) ' + time_unit:>20} {'(Q4/Q1)':>8}")
    print("-" * 140)

    n_display = min(20, len(quartile_df))
    for i, (_, row) in enumerate(quartile_df.head(n_display).iterrows(), 1):
        biomarker_name = row['biomarker'][:39]
        n_str = f"{row['n_low']}/{row['n_high']}"
        events_str = f"{row['events_low']}/{row['events_high']}"
        os_str = f"{row['median_os_low']:.1f}/{row['median_os_high']:.1f}"
        print(f"{i:<6} {biomarker_name:<40} {n_str:>12} {events_str:>12} {os_str:>20} {row['hr_high_vs_low']:>8.3f} {row['logrank_p']:>10.4f}")
else:
    print("\n⚠️ No quartile analysis results available")

# ============================================================================
# MEDIAN ANALYSIS
# ============================================================================

print("\n" + "=" * 120)
print("📊 MEDIAN ANALYSIS (Below vs Above Median)")
print("-" * 120)

median_results = []
for biomarker in biomarker_cols:
    result = stratify_and_test(df, biomarker, cutoff_type='median')
    if result:
        median_results.append(result)

if len(median_results) > 0:
    median_df = pd.DataFrame(median_results)
    median_df = median_df.sort_values('logrank_p').reset_index(drop=True)
    median_df['rank'] = range(1, len(median_df) + 1)
    median_df['significant_0.05'] = median_df['logrank_p'] < 0.05
    median_df['significant_0.01'] = median_df['logrank_p'] < 0.01

    print(f"\n📈 TOP 20 BIOMARKERS (Median Stratification)")
    print("=" * 140)
    print(f"{'Rank':<6} {'Biomarker':<40} {'N (Low/High)':>12} {'Events':>12} {f'Median {survival_abbrev}':>20} {'HR':>8} {'P-value':>10}")
    print(f"{'':6} {'':40} {'':>12} {'(Low/High)':>12} {'(Low/High) ' + time_unit:>20} {'(High/Low)':>8}")
    print("-" * 140)

    n_display = min(20, len(median_df))
    for i, (_, row) in enumerate(median_df.head(n_display).iterrows(), 1):
        biomarker_name = row['biomarker'][:39]
        n_str = f"{row['n_low']}/{row['n_high']}"
        events_str = f"{row['events_low']}/{row['events_high']}"
        os_str = f"{row['median_os_low']:.1f}/{row['median_os_high']:.1f}"
        print(f"{i:<6} {biomarker_name:<40} {n_str:>12} {events_str:>12} {os_str:>20} {row['hr_high_vs_low']:>8.3f} {row['logrank_p']:>10.4f}")
else:
    print("\n⚠️ No median analysis results available")

# ============================================================================
# MULTIVARIATE COX REGRESSION ANALYSIS - COMPLETE FIXED VERSION
# Includes: Parameter extraction fix + 10x10 Cross-Validation + Stepwise
# ============================================================================

print("\n" + "=" * 120)
print("📊 MULTIVARIATE COX REGRESSION ANALYSIS")
print("-" * 120)

multivariate_results = None
selected_biomarkers_for_multivariate = []
stepwise_results = None
cv_results = None

if len(cox_df) > 0:
    # Ask if user wants to perform multivariate analysis
    print("\n❓ Do you want to perform multivariate Cox regression analysis?")
    print("   This will identify independent predictors by analyzing multiple biomarkers together")
    print("   1. Yes - Perform multivariate analysis")
    print("   2. No - Skip to visualization")

    multivariate_choice = input("\n🔹 Enter choice (1 or 2) [default: 1]: ").strip()
    if multivariate_choice == "" or multivariate_choice == "1":
        print("\n📊 CONFIGURING MULTIVARIATE ANALYSIS")
        print("-" * 60)

        # Select biomarkers for multivariate analysis
        print("\n🔍 Select biomarkers for multivariate analysis:")
        print("   1. Top N biomarkers from univariate analysis (by p-value)")
        print("   2. Significant biomarkers only (p < 0.05)")
        print("   3. Significant biomarkers with low correlation (r < 0.7)")
        print("   4. Manual selection")

        selection_method = input("\n🔹 Enter choice (1, 2, 3, or 4) [default: 1]: ").strip()

        if selection_method == "2":
            # Significant biomarkers only
            selected_biomarkers_for_multivariate = cox_df[cox_df['p_value'] < 0.05]['biomarker'].tolist()
            if len(selected_biomarkers_for_multivariate) == 0:
                print("   ⚠️ No significant biomarkers found. Using top 5 instead.")
                selected_biomarkers_for_multivariate = cox_df.head(5)['biomarker'].tolist()

        elif selection_method == "3":
            # Significant with low correlation
            sig_biomarkers = cox_df[cox_df['p_value'] < 0.05]['biomarker'].tolist()
            if len(sig_biomarkers) > 0:
                corr_matrix = df[sig_biomarkers].corr().abs()
                selected_biomarkers_for_multivariate = [sig_biomarkers[0]]
                for biomarker in sig_biomarkers[1:]:
                    max_corr = max([corr_matrix.loc[biomarker, s] for s in selected_biomarkers_for_multivariate])
                    if max_corr < 0.7:
                        selected_biomarkers_for_multivariate.append(biomarker)
                    if len(selected_biomarkers_for_multivariate) >= 10:
                        break
            else:
                print("   ⚠️ No significant biomarkers found. Using top 5 instead.")
                selected_biomarkers_for_multivariate = cox_df.head(5)['biomarker'].tolist()

        elif selection_method == "4":
            # Manual selection from ALL biomarkers
            all_biomarkers = cox_df['biomarker'].tolist()
            total_biomarkers = len(all_biomarkers)

            print(f"\n   📋 ALL {total_biomarkers} BIOMARKERS (sorted by p-value):")
            print("-" * 70)

            # Display in pages of 20 for readability
            page_size = 20
            current_page = 0

            while True:
                start_idx = current_page * page_size
                end_idx = min(start_idx + page_size, total_biomarkers)

                print(f"\n   Showing {start_idx + 1}-{end_idx} of {total_biomarkers}:")
                print("-" * 70)
                for j in range(start_idx, end_idx):
                    biomarker = all_biomarkers[j]
                    p_val = cox_df[cox_df['biomarker'] == biomarker]['p_value'].values[0]
                    hr = cox_df[cox_df['biomarker'] == biomarker]['hr'].values[0]
                    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
                    print(f"   {j+1:3}. {biomarker[:35]:<35} HR={hr:.2f}  p={p_val:.4f} {sig}")

                print("-" * 70)

                # Navigation options
                has_more = end_idx < total_biomarkers
                has_prev = current_page > 0

                nav_options = []
                if has_more:
                    nav_options.append("'n' for next page")
                if has_prev:
                    nav_options.append("'p' for previous page")
                nav_options.append("'a' to show all at once")
                nav_options.append("or enter numbers to select")

                print(f"   Options: {', '.join(nav_options)}")

                user_input = input("\n🔹 Enter choice: ").strip().lower()

                if user_input == 'n' and has_more:
                    current_page += 1
                    continue
                elif user_input == 'p' and has_prev:
                    current_page -= 1
                    continue
                elif user_input == 'a':
                    # Show all biomarkers at once
                    print(f"\n   📋 ALL {total_biomarkers} BIOMARKERS:")
                    print("-" * 70)
                    for j, biomarker in enumerate(all_biomarkers):
                        p_val = cox_df[cox_df['biomarker'] == biomarker]['p_value'].values[0]
                        hr = cox_df[cox_df['biomarker'] == biomarker]['hr'].values[0]
                        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
                        print(f"   {j+1:3}. {biomarker[:35]:<35} HR={hr:.2f}  p={p_val:.4f} {sig}")
                    print("-" * 70)
                    print("\n   Enter biomarker numbers (comma-separated, e.g., 1,3,5,7):")
                    user_input = input("\n🔹 Enter numbers: ").strip()

                # Try to parse as selection
                if user_input and user_input not in ['n', 'p', 'a']:
                    try:
                        choices = user_input.split(',')
                        for choice in choices:
                            # Handle ranges like "1-5"
                            if '-' in choice and not choice.startswith('-'):
                                range_parts = choice.split('-')
                                start = int(range_parts[0].strip())
                                end = int(range_parts[1].strip())
                                for idx in range(start, end + 1):
                                    if 1 <= idx <= total_biomarkers:
                                        selected_biomarkers_for_multivariate.append(all_biomarkers[idx - 1])
                                    else:
                                        raise ValueError(f"Number {idx} is out of range (1-{total_biomarkers})")
                            else:
                                idx = int(choice.strip())
                                if 1 <= idx <= total_biomarkers:
                                    selected_biomarkers_for_multivariate.append(all_biomarkers[idx - 1])
                                else:
                                    raise ValueError(f"Number {choice} is out of range (1-{total_biomarkers})")

                        # Remove duplicates while preserving order
                        seen = set()
                        selected_biomarkers_for_multivariate = [x for x in selected_biomarkers_for_multivariate
                                                                if not (x in seen or seen.add(x))]

                        if selected_biomarkers_for_multivariate:
                            print(f"\n   ✅ Selected {len(selected_biomarkers_for_multivariate)} biomarkers:")
                            for bm in selected_biomarkers_for_multivariate:
                                print(f"      • {bm}")
                            break
                    except (ValueError, IndexError) as e:
                        print(f"   ❌ Error: {e}. Please try again.")
                        selected_biomarkers_for_multivariate = []
        else:
            # Default: Top N
            print("\n   How many top biomarkers to include?")
            n_biomarkers = input("🔹 Enter number (3-20) [default: 5]: ").strip()
            try:
                n_biomarkers = int(n_biomarkers)
                n_biomarkers = min(max(n_biomarkers, 3), min(20, len(cox_df)))
            except Exception:
                n_biomarkers = min(5, len(cox_df))
            selected_biomarkers_for_multivariate = cox_df.head(n_biomarkers)['biomarker'].tolist()

        # EPV-aware cap using tunable parameters
        events = int(df['event'].sum())
        max_vars = max(EPV_MIN_VARS, min(EPV_MAX_VARS, events // EPV_DIVISOR))
        print(f"   ℹ️ EPV Settings: MIN_VARS={EPV_MIN_VARS}, MAX_VARS={EPV_MAX_VARS}, DIVISOR={EPV_DIVISOR}")
        print(f"   ℹ️ With {events} events: max_vars = max({EPV_MIN_VARS}, min({EPV_MAX_VARS}, {events}//{EPV_DIVISOR})) = {max_vars}")

        # Trim selected to EPV limit preserving cox_df order
        if len(selected_biomarkers_for_multivariate) > max_vars:
            ordered = [b for b in cox_df['biomarker'].tolist() if b in selected_biomarkers_for_multivariate]
            selected_biomarkers_for_multivariate = ordered[:max_vars]
            print(f"\n⚖️ EPV limit applied: using {len(selected_biomarkers_for_multivariate)} variables (max_vars={max_vars})")

        print(f"\n✅ Selected {len(selected_biomarkers_for_multivariate)} biomarkers for multivariate analysis:")
        for i, b in enumerate(selected_biomarkers_for_multivariate, 1):
            print(f"   {i}. {b}")

        # Perform multivariate Cox regression
        if len(selected_biomarkers_for_multivariate) >= 2:
            print("\n📊 Performing multivariate Cox regression...")
            try:
                multivariate_data = df[[survival_time_col, 'event'] + selected_biomarkers_for_multivariate].dropna()
                if len(multivariate_data) < 20:
                    print("   ⚠️ Insufficient data for multivariate analysis")
                    multivariate_results = None
                else:
                    # Multicollinearity check
                    print("\n📊 Checking for multicollinearity...")
                    corr_matrix = multivariate_data[selected_biomarkers_for_multivariate].corr()
                    high_corr_pairs = []
                    for i in range(len(selected_biomarkers_for_multivariate)):
                        for j in range(i+1, len(selected_biomarkers_for_multivariate)):
                            r = corr_matrix.iloc[i, j]
                            if abs(r) > 0.8:
                                high_corr_pairs.append((selected_biomarkers_for_multivariate[i],
                                                       selected_biomarkers_for_multivariate[j],
                                                       r))
                    if high_corr_pairs:
                        print("\n⚠️ Warning: High correlation detected between biomarkers:")
                        for b1, b2, r in high_corr_pairs[:5]:
                            print(f"   • {b1[:30]} ↔ {b2[:30]}: r={r:.3f}")
                    else:
                        print("   ✓ No high correlations detected (all |r| < 0.8)")

                    # Fit multivariate model (quote names safely)
                    print("\n   Fitting multivariate Cox model...")
                    cph_multi = CoxPHFitter(penalizer=PENALIZER)
                    formula = ' + '.join([f'Q("{b}")' for b in selected_biomarkers_for_multivariate])

                    cph_multi.fit(
                        multivariate_data,
                        duration_col=survival_time_col,
                        event_col='event',
                        formula=formula,
                        robust=ROBUST_SE
                    )

                    # Fixed: Better extraction of AIC and BIC
                    AIC_multi = _get_attr(cph_multi, ["AIC_partial_", "AIC_", "AIC"], None)
                    BIC_multi = _get_attr(cph_multi, ["BIC_partial_", "BIC_", "BIC"], None)

                    multivariate_results = {
                        'model_concordance': float(cph_multi.concordance_index_),
                        'log_likelihood': float(cph_multi.log_likelihood_),
                        'AIC': float(AIC_multi) if AIC_multi is not None else None,
                        'BIC': float(BIC_multi) if BIC_multi is not None else None,
                        'n_samples': int(len(multivariate_data)),
                        'n_events': int(multivariate_data['event'].sum()),
                        'biomarkers': []
                    }

                    # ============================================================================
                    # FIXE<data_path> ROBUST PARAMETER EXTRACTION WITH MULTIPLE FORMAT ATTEMPTS
                    # ============================================================================
                    print(f"\n   📋 Model parameters found: {list(cph_multi.params_.index)}")

                    extraction_success_count = 0
                    for biomarker in selected_biomarkers_for_multivariate:
                        # Try multiple parameter name formats
                        param_name = None
                        possible_names = [
                            f'Q("{biomarker}")',      # Double-quoted format
                            biomarker,                # Direct name
                            f"Q('{biomarker}')",      # Single-quoted format
                        ]

                        # First pass: exact match
                        for pn in possible_names:
                            if pn in cph_multi.params_.index:
                                param_name = pn
                                break

                        # Second pass: fuzzy match if not found
                        if param_name is None:
                            for idx_name in cph_multi.params_.index:
                                # Remove quotes and Q() wrapper to compare
                                cleaned_idx = str(idx_name).replace('Q("', '').replace('")', '').replace("Q('", "").replace("')", "")
                                if cleaned_idx == biomarker:
                                    param_name = idx_name
                                    print(f"   🔍 Fuzzy matched '{biomarker[:30]}' → '{str(param_name)[:30]}'")
                                    break

                        # Extract parameter values if found
                        if param_name is not None:
                            try:
                                # Get confidence interval columns (handle different lifelines versions)
                                ci_cols = cph_multi.confidence_intervals_.columns

                                biomarker_result = {
                                    'name': biomarker,
                                    'coef': float(cph_multi.params_[param_name]),
                                    'se': float(cph_multi.standard_errors_[param_name]),
                                    'hr': float(np.exp(cph_multi.params_[param_name])),
                                    'hr_lower': float(np.exp(cph_multi.confidence_intervals_.loc[param_name, ci_cols[0]])),
                                    'hr_upper': float(np.exp(cph_multi.confidence_intervals_.loc[param_name, ci_cols[1]])),
                                    'p_value': float(cph_multi.summary.loc[param_name, 'p']),
                                    'z': float(cph_multi.summary.loc[param_name, 'z'])
                                }

                                multivariate_results['biomarkers'].append(biomarker_result)
                                extraction_success_count += 1
                                print(f"   ✓ Extracted: {biomarker[:40]} (HR={biomarker_result['hr']:.3f}, p={biomarker_result['p_value']:.4f})")

                            except Exception as e:
                                print(f"   ⚠️ Extraction error for '{biomarker[:30]}': {str(e)[:60]}")
                        else:
                            print(f"   ❌ Parameter NOT FOUND: '{biomarker[:40]}'")

                    # Check if we successfully extracted any results
                    if len(multivariate_results['biomarkers']) == 0:
                        print("\n   ⚠️ WARNING: No biomarker results were extracted!")
                        print("   Attempting alternative extraction method...")

                        # Alternative: extract ALL parameters from the model
                        for param_name in cph_multi.params_.index:
                            try:
                                ci_cols = cph_multi.confidence_intervals_.columns

                                # Try to find corresponding biomarker
                                matching_biomarker = str(param_name).replace('Q("', '').replace('")', '').replace("Q('", "").replace("')", "")

                                biomarker_result = {
                                    'name': matching_biomarker,
                                    'coef': float(cph_multi.params_[param_name]),
                                    'se': float(cph_multi.standard_errors_[param_name]),
                                    'hr': float(np.exp(cph_multi.params_[param_name])),
                                    'hr_lower': float(np.exp(cph_multi.confidence_intervals_.loc[param_name, ci_cols[0]])),
                                    'hr_upper': float(np.exp(cph_multi.confidence_intervals_.loc[param_name, ci_cols[1]])),
                                    'p_value': float(cph_multi.summary.loc[param_name, 'p']),
                                    'z': float(cph_multi.summary.loc[param_name, 'z'])
                                }

                                multivariate_results['biomarkers'].append(biomarker_result)
                                print(f"   ✓ Alt. extracted: {matching_biomarker[:40]} from param '{str(param_name)[:30]}'")
                            except Exception:
                                continue

                    print(f"\n✅ Multivariate Cox regression completed!")
                    print(f"   Successfully extracted: {len(multivariate_results['biomarkers'])}/{len(selected_biomarkers_for_multivariate)} biomarkers")
                    print(f"   Model C-index: {multivariate_results['model_concordance']:.3f}")
                    if multivariate_results['AIC'] is not None:
                        print(f"   AIC: {multivariate_results['AIC']:.2f}")
                    if multivariate_results['BIC'] is not None:
                        print(f"   BIC: {multivariate_results['BIC']:.2f}")
                    print(f"   Events/Samples: {multivariate_results['n_events']}/{multivariate_results['n_samples']}")

                    # Display results table
                    if len(multivariate_results['biomarkers']) > 0:
                        print("\n📊 MULTIVARIATE COX REGRESSION RESULTS")
                        print("=" * 100)
                        print(f"{'Biomarker':<35} {'HR':>10} {'95% CI':>20} {'P-value':>10} {'Signif':>8}")
                        print("-" * 100)
                        for biomarker_result in multivariate_results['biomarkers']:
                            name = biomarker_result['name'][:34]
                            hr = biomarker_result['hr']
                            ci = f"({biomarker_result['hr_lower']:.3f}-{biomarker_result['hr_upper']:.3f})"
                            p_val = biomarker_result['p_value']
                            sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
                            print(f"{name:<35} {hr:>10.3f} {ci:>20} {p_val:>10.4f} {sig:>8}")
                        print("=" * 100)
                    else:
                        print("\n⚠️ No multivariate results to display - parameter extraction failed")
                        multivariate_results = None

                    # ============================================================================
                    # ENHANCED CROSS-VALIDATION: 10-FOLD STRATIFIED, 10 REPETITIONS
                    # ============================================================================
                    if multivariate_results is not None and len(multivariate_results['biomarkers']) > 0:
                        print("\n" + "=" * 100)
                        print("📊 CROSS-VALIDATION")
                        print("-" * 100)
                        print("\n❓ Perform cross-validation to assess model stability?")
                        print("   This will run 10-fold stratified CV with 10 repetitions (100 total runs)")
                        print("   1. Yes - Perform CV")
                        print("   2. No - Skip CV")

                        cv_choice = input("\n🔹 Enter choice (1 or 2) [default: 1]: ").strip()

                        if cv_choice == "" or cv_choice == "1":
                            print("\n📊 Performing enhanced cross-validation...")
                            print("   Configuration: 10-fold stratified CV, 10 repetitions")
                            try:
                                from sklearn.model_selection import StratifiedKFold

                                # CV parameters
                                N_FOLDS = 5
                                N_REPETITIONS = 10
                                BASE_SEED = 42

                                # Prepare data
                                cv_data = multivariate_data.copy()

                                # Storage for all CV scores
                                all_cv_scores = []
                                repetition_means = []
                                repetition_stds = []

                                print(f"   Total CV runs: {N_FOLDS * N_REPETITIONS} ({N_FOLDS} folds × {N_REPETITIONS} repetitions)")
                                print(f"   Dataset: {len(cv_data)} samples, {cv_data['event'].sum()} events")
                                print(f"   Formula: {formula[:60]}{'...' if len(formula) > 60 else ''}")
                                print("\n   Running cross-validation...")

                                # Perform repeated stratified CV
                                for rep in range(N_REPETITIONS):
                                    seed = BASE_SEED + rep
                                    np.random.seed(seed)

                                    # Stratified K-Fold split
                                    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

                                    fold_scores = []
                                    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(cv_data, cv_data['event'])):
                                        try:
                                            # Split data
                                            train_data = cv_data.iloc[train_idx]
                                            test_data = cv_data.iloc[test_idx]

                                            # Check minimum requirements
                                            if len(train_data) < 20 or len(test_data) < 5:
                                                continue
                                            if train_data['event'].sum() < 5 or test_data['event'].sum() < 2:
                                                continue

                                            # Fit model on training data
                                            cph_cv = CoxPHFitter(penalizer=PENALIZER)
                                            cph_cv.fit(
                                                train_data,
                                                duration_col=survival_time_col,
                                                event_col='event',
                                                formula=formula,
                                                robust=ROBUST_SE
                                            )

                                            # Evaluate on test data
                                            c_index = cph_cv.score(test_data, scoring_method="concordance_index")
                                            fold_scores.append(c_index)
                                            all_cv_scores.append(c_index)

                                        except Exception as e:
                                            continue

                                    # Store repetition statistics
                                    if len(fold_scores) > 0:
                                        rep_mean = np.mean(fold_scores)
                                        rep_std = np.std(fold_scores)
                                        repetition_means.append(rep_mean)
                                        repetition_stds.append(rep_std)
                                        print(f"   Rep {rep+1:2d}: {len(fold_scores):2d} folds, C-index = {rep_mean:.4f} ± {rep_std:.4f}")

                                # Calculate overall statistics
                                if len(all_cv_scores) > 0:
                                    cv_results = {
                                        'mean_c_index': np.mean(all_cv_scores),
                                        'std_c_index': np.std(all_cv_scores),
                                        'median_c_index': np.median(all_cv_scores),
                                        'min_c_index': np.min(all_cv_scores),
                                        'max_c_index': np.max(all_cv_scores),
                                        'n_successful_runs': len(all_cv_scores),
                                        'n_folds': N_FOLDS,
                                        'n_repetitions': N_REPETITIONS,
                                        'repetition_mean_of_means': np.mean(repetition_means),
                                        'repetition_std_of_means': np.std(repetition_means)
                                    }

                                    print(f"\n   ✅ Cross-Validation Results:")
                                    print(f"      Successful runs: {cv_results['n_successful_runs']}/{N_FOLDS * N_REPETITIONS}")
                                    print(f"      Mean C-index: {cv_results['mean_c_index']:.4f} ± {cv_results['std_c_index']:.4f}")
                                    print(f"      Median C-index: {cv_results['median_c_index']:.4f}")
                                    print(f"      Range: [{cv_results['min_c_index']:.4f}, {cv_results['max_c_index']:.4f}]")
                                    print(f"      Repetition variability: {cv_results['repetition_std_of_means']:.4f}")

                                    # Compare to training C-index
                                    train_c_index = multivariate_results['model_concordance']
                                    optimism = train_c_index - cv_results['mean_c_index']
                                    print(f"\n   📊 Model Performance Comparison:")
                                    print(f"      Training C-index: {train_c_index:.4f}")
                                    print(f"      CV C-index: {cv_results['mean_c_index']:.4f}")
                                    print(f"      Optimism (bias): {optimism:.4f}")

                                    if optimism > 0.05:
                                        print(f"      ⚠️ Substantial optimism detected (>{0.05:.3f}) - model may be overfitting")
                                    elif optimism > 0.03:
                                        print(f"      ⚡ Moderate optimism detected - consider larger sample or fewer variables")
                                    else:
                                        print(f"      ✓ Low optimism - model appears stable")

                                else:
                                    print("\n   ⚠️ CV failed: no successful runs")
                                    cv_results = None

                            except ImportError:
                                print("\n   ⚠️ CV requires scikit-learn. Installing...")
                                import subprocess
                                subprocess.check_call(['pip', 'install', 'scikit-learn', '-q'])
                                print("   ✓ scikit-learn installed. Please re-run the multivariate analysis section.")
                                cv_results = None
                            except Exception as e:
                                print(f"\n   ⚠️ CV failed: {str(e)[:100]}")
                                import traceback
                                print("\n   Error details:")
                                traceback.print_exc()
                                cv_results = None
                        else:
                            print("\n   ➡️ Skipping cross-validation")
                            cv_results = None

                    # ============================================================================
                    # STEPWISE VARIABLE SELECTION (OPTIONAL)
                    # ============================================================================
                    if multivariate_results is not None and len(multivariate_results['biomarkers']) > 2:
                        print("\n" + "=" * 100)
                        print("📊 STEPWISE VARIABLE SELECTION")
                        print("-" * 100)
                        print("\n❓ Perform stepwise variable selection?")
                        print("   This will identify the optimal set of predictors")
                        print("   1. Yes - Backward elimination (remove non-significant)")
                        print("   2. Yes - Forward selection (add best predictors)")
                        print("   3. No - Skip stepwise")

                        stepwise_choice = input("\n🔹 Enter choice (1, 2, or 3) [default: 3]: ").strip()

                        if stepwise_choice in ("1", "2"):
                            print(f"\n📊 Performing stepwise selection...")

                            if stepwise_choice == "2":
                                # Forward selection (minimize AIC) with EPV cap
                                print("   Method: Forward Selection (minimizing AIC)")
                                selected_vars = []
                                remaining_vars = selected_biomarkers_for_multivariate.copy()
                                best_aic = np.inf

                                while remaining_vars and len(selected_vars) < max_vars:
                                    best_var = None
                                    improved = False

                                    for var in remaining_vars:
                                        temp_vars = selected_vars + [var]
                                        temp_data = df[[survival_time_col, 'event'] + temp_vars].dropna()

                                        if len(temp_data) >= 20:
                                            try:
                                                cph_temp = CoxPHFitter(penalizer=PENALIZER)
                                                formula_temp = ' + '.join([f'Q("{v}")' for v in temp_vars])
                                                cph_temp.fit(temp_data, duration_col=survival_time_col,
                                                           event_col='event', formula=formula_temp, robust=ROBUST_SE)

                                                aic_temp = _get_attr(cph_temp, ["AIC_partial_", "AIC_", "AIC"], np.inf)

                                                if aic_temp < best_aic:
                                                    best_aic = aic_temp
                                                    best_var = var
                                                    improved = True
                                            except Exception:
                                                continue

                                    if improved and best_var:
                                        selected_vars.append(best_var)
                                        remaining_vars.remove(best_var)
                                        print(f"   Added: {best_var[:40]} (AIC={best_aic:.2f})")
                                    else:
                                        break

                                stepwise_results = selected_vars

                            else:  # Backward elimination
                                print("   Method: Backward Elimination (p > 0.10 threshold)")
                                selected_vars = selected_biomarkers_for_multivariate.copy()

                                while len(selected_vars) > 2:
                                    temp_data = df[[survival_time_col, 'event'] + selected_vars].dropna()

                                    if len(temp_data) < 20:
                                        break

                                    try:
                                        cph_temp = CoxPHFitter(penalizer=PENALIZER)
                                        formula_temp = ' + '.join([f'Q("{v}")' for v in selected_vars])
                                        cph_temp.fit(temp_data, duration_col=survival_time_col,
                                                   event_col='event', formula=formula_temp, robust=ROBUST_SE)

                                        # Find worst (highest p-value)
                                        worst_var = None
                                        max_p = -1

                                        for var in selected_vars:
                                            # Try multiple formats
                                            for qn in [f'Q("{var}")', var, f"Q('{var}')'"]:
                                                if qn in cph_temp.summary.index:
                                                    p_val = cph_temp.summary.loc[qn, 'p']
                                                    if p_val > max_p:
                                                        max_p = p_val
                                                        worst_var = var
                                                    break

                                        if max_p > 0.1 and worst_var:
                                            selected_vars.remove(worst_var)
                                            print(f"   Removed: {worst_var[:40]} (p={max_p:.4f})")
                                        else:
                                            break
                                    except Exception as e:
                                        print(f"   Warning: {str(e)[:50]}")
                                        break

                                # Enforce EPV cap if still too many
                                if len(selected_vars) > max_vars:
                                    print(f"   Trimming to EPV cap (max={max_vars})...")
                                    selected_vars = selected_vars[:max_vars]

                                stepwise_results = selected_vars

                            if stepwise_results and len(stepwise_results) > 0:
                                print(f"\n   ✅ Stepwise selection completed!")
                                print(f"   Final model includes {len(stepwise_results)} biomarkers:")
                                for i, var in enumerate(stepwise_results, 1):
                                    print(f"      {i}. {var}")

                                # Fit final stepwise model
                                stepwise_data = df[[survival_time_col, 'event'] + stepwise_results].dropna()
                                if len(stepwise_data) >= 20:
                                    try:
                                        cph_stepwise = CoxPHFitter(penalizer=PENALIZER)
                                        formula_stepwise = ' + '.join([f'Q("{v}")' for v in stepwise_results])
                                        cph_stepwise.fit(stepwise_data, duration_col=survival_time_col,
                                                       event_col='event', formula=formula_stepwise, robust=ROBUST_SE)

                                        print(f"\n   📊 Stepwise Model Performance:")
                                        print(f"      C-index: {cph_stepwise.concordance_index_:.4f}")
                                        AIC_sw = _get_attr(cph_stepwise, ["AIC_partial_", "AIC_", "AIC"], None)
                                        if AIC_sw is not None:
                                            print(f"      AIC: {AIC_sw:.2f}")

                                        # Compare to full model
                                        if multivariate_results['AIC'] is not None and AIC_sw is not None:
                                            aic_diff = multivariate_results['AIC'] - AIC_sw
                                            print(f"      AIC improvement: {aic_diff:.2f} {'(better)' if aic_diff > 0 else '(worse)'}")
                                    except Exception as e:
                                        print(f"   ⚠️ Could not fit stepwise model: {str(e)[:50]}")
                            else:
                                print("\n   ⚠️ Stepwise selection did not produce results")
                        else:
                            print("\n   ➡️ Skipping stepwise selection")
                            stepwise_results = None

            except Exception as e:
                print(f"\n❌ Error in multivariate analysis: {str(e)}")
                import traceback
                print("\n📋 Full error traceback:")
                traceback.print_exc()
                multivariate_results = None
        else:
            print("\n⚠️ Need at least 2 biomarkers for multivariate analysis")

        # Give user time to review multivariate results
        print("\n" + "=" * 120)
        input("📊 Multivariate analysis complete. Press Enter to continue to visualizations...")
    else:
        print("\n➡️ Skipping multivariate analysis, proceeding to visualizations...")
else:
    print("\n⚠️ No univariate results available for multivariate analysis")
    print("   Proceeding to visualizations...")

# ============================================================================
# KAPLAN-MEIER PLOTS WITH PNG EXPORT
# ============================================================================

# Create plots subfolder
plots_folder = os.path.join(RESULTS_FOLDER, "KM_Plots")
os.makedirs(plots_folder, exist_ok=True)

def plot_km_curves(df, biomarker, cutoff_type='median', ax=None, save_path=None):
    """Plot Kaplan-Meier curves for a biomarker"""
    try:
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 8))

        data = df[[survival_time_col, 'event', biomarker]].dropna()

        if cutoff_type == 'median':
            cutoff = data[biomarker].median()
            mask_low = data[biomarker] <= cutoff
            label_low = f'≤ median ({cutoff:.1f})'
            label_high = f'> median ({cutoff:.1f})'
            group_low = data[mask_low]
            group_high = data[~mask_low]
        else:  # quartiles
            q25 = data[biomarker].quantile(0.25)
            q75 = data[biomarker].quantile(0.75)
            mask_low = data[biomarker] <= q25
            mask_high = data[biomarker] >= q75
            data = data[mask_low | mask_high]
            group_low = data[data[biomarker] <= q25]
            group_high = data[data[biomarker] >= q75]
            label_low = f'Q1 (≤{q25:.1f})'
            label_high = f'Q4 (≥{q75:.1f})'

        if len(group_low) < 5 or len(group_high) < 5:
            return None

        kmf = KaplanMeierFitter()

        # Plot low group
        kmf.fit(group_low[survival_time_col], group_low['event'], label=label_low)
        kmf.plot_survival_function(ax=ax, ci_show=True, alpha=0.2, linewidth=2.5)

        # Plot high group
        kmf.fit(group_high[survival_time_col], group_high['event'], label=label_high)
        kmf.plot_survival_function(ax=ax, ci_show=True, alpha=0.2, linewidth=2.5)

        # Log-rank test
        lr_result = logrank_test(
            group_low[survival_time_col], group_high[survival_time_col],
            group_low['event'], group_high['event']
        )

        # Calculate HR
        cph = CoxPHFitter(penalizer=PENALIZER)
        combined_data = pd.concat([
            group_low.assign(group=0),
            group_high.assign(group=1)
        ])
        cph.fit(combined_data[[survival_time_col, 'event', 'group']],
                duration_col=survival_time_col, event_col='event', formula='group', robust=ROBUST_SE)
        hr = float(np.exp(cph.params_['group']))

        # Truncate long biomarker names for title
        biomarker_display = biomarker if len(biomarker) <= 35 else biomarker[:32] + "..."

        ax.set_xlabel(f'Time ({time_unit})', fontsize=12)
        ax.set_ylabel(f'{survival_type} Probability', fontsize=12)
        ax.set_title(f'{biomarker_display}\n{cutoff_type.capitalize()} Stratification', fontsize=14, fontweight='bold')
        ax.legend(loc='best', fontsize=11)

        # Add statistics box
        stats_text = f'Log-rank p = {lr_result.p_value:.4f}\nHR = {hr:.3f}\nn_low = {len(group_low)}, n_high = {len(group_high)}'
        ax.text(0.05, 0.15, stats_text, transform=ax.transAxes, fontsize=10,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
        ax.grid(True, alpha=0.3)

        # Save individual plot if path provided
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')

        return ax
    except Exception:
        return None

# Only create plots if we have results
if len(quartile_df) > 0 or len(median_df) > 0:
    print("\n" + "=" * 120)
    print("📊 GENERATING KAPLAN-MEIER SURVIVAL CURVES")
    print("-" * 120)

    print("\n📈 Generating individual plots for top biomarkers...")

    # Quartile individual plots
    if len(quartile_df) > 0:
        n_plots = min(10, len(quartile_df))
        for i, (_, row) in enumerate(quartile_df.head(n_plots).iterrows(), 1):
            try:
                fig, ax = plt.subplots(figsize=(10, 8))
                safe_name = row['biomarker'].replace('/', '_').replace('\\', '_')[:50]
                save_path = os.path.join(plots_folder, f"KM_Quartile_{i:02d}_{safe_name}.png")
                if plot_km_curves(df, row['biomarker'], 'quartiles', ax, save_path):
                    print(f"   Saved: KM_Quartile_{i:02d}_{safe_name}.png")
                plt.close()
            except Exception:
                continue

    # Median individual plots
    if len(median_df) > 0:
        n_plots = min(10, len(median_df))
        for i, (_, row) in enumerate(median_df.head(n_plots).iterrows(), 1):
            try:
                fig, ax = plt.subplots(figsize=(10, 8))
                safe_name = row['biomarker'].replace('/', '_').replace('\\', '_')[:50]
                save_path = os.path.join(plots_folder, f"KM_Median_{i:02d}_{safe_name}.png")
                if plot_km_curves(df, row['biomarker'], 'median', ax, save_path):
                    print(f"   Saved: KM_Median_{i:02d}_{safe_name}.png")
                plt.close()
            except Exception:
                continue

    # Combined plots - Quartile
    if len(quartile_df) >= 3:
        try:
            print("\n📈 Top 3 Biomarkers - Quartile Stratification (Q1 vs Q4)")
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))
            for i, (_, row) in enumerate(quartile_df.head(3).iterrows()):
                plot_km_curves(df, row['biomarker'], 'quartiles', axes[i])
            plt.suptitle(f'TOP 3 BIOMARKERS - QUARTILE STRATIFICATION (Q1 vs Q4)\n{survival_type} Analysis', fontsize=16, fontweight='bold', y=1.05)
            plt.tight_layout()
            combined_path = os.path.join(plots_folder, "KM_Top3_Quartiles_Combined.png")
            plt.savefig(combined_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"   Saved: KM_Top3_Quartiles_Combined.png")
        except Exception:
            pass

    # Combined plots - Median
    if len(median_df) >= 3:
        try:
            print("\n📈 Top 3 Biomarkers - Median Stratification")
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))
            for i, (_, row) in enumerate(median_df.head(3).iterrows()):
                plot_km_curves(df, row['biomarker'], 'median', axes[i])
            plt.suptitle(f'TOP 3 BIOMARKERS - MEDIAN STRATIFICATION\n{survival_type} Analysis', fontsize=16, fontweight='bold', y=1.05)
            plt.tight_layout()
            combined_path = os.path.join(plots_folder, "KM_Top3_Median_Combined.png")
            plt.savefig(combined_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"   Saved: KM_Top3_Median_Combined.png")
        except Exception:
            pass

# ============================================================================
# CREATE DISTRIBUTION PLOTS WITH CUTOFF VALUES
# ============================================================================

print("\n" + "=" * 120)
print("📊 CREATING BIOMARKER DISTRIBUTION PLOTS")
print("-" * 120)

# Create distribution plots for top biomarkers
if len(cox_df) > 0 or len(quartile_df) > 0:
    # Determine which biomarkers to plot
    if len(cox_df) > 0:
        top_biomarkers_to_plot = cox_df.head(6)['biomarker'].tolist()
    else:
        top_biomarkers_to_plot = quartile_df.head(6)['biomarker'].tolist()

    # Create distribution plots with cutoffs
    print("\n📈 Generating distribution plots for top 6 biomarkers...")

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    for idx, biomarker in enumerate(top_biomarkers_to_plot[:6]):
        try:
            ax = axes[idx]
            biomarker_data = df[biomarker].dropna()

            # Create histogram
            ax.hist(biomarker_data, bins=30, alpha=0.7, color='skyblue', edgecolor='black')

            # Add vertical lines for cutoffs
            median_val = biomarker_data.median()
            q1_val = biomarker_data.quantile(0.25)
            q3_val = biomarker_data.quantile(0.75)

            ax.axvline(median_val, color='red', linestyle='--', linewidth=2, label=f'Median: {median_val:.2f}')
            ax.axvline(q1_val, color='green', linestyle='--', linewidth=2, label=f'Q1: {q1_val:.2f}')
            ax.axvline(q3_val, color='orange', linestyle='--', linewidth=2, label=f'Q3: {q3_val:.2f}')

            # Formatting
            biomarker_display = biomarker if len(biomarker) <= 30 else biomarker[:27] + "..."
            ax.set_title(f'{biomarker_display}', fontsize=12, fontweight='bold')
            ax.set_xlabel('Value', fontsize=10)
            ax.set_ylabel('Frequency', fontsize=10)
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
        except Exception:
            continue

    plt.suptitle('BIOMARKER DISTRIBUTIONS WITH CUTOFF VALUES', fontsize=16, fontweight='bold')
    plt.tight_layout()
    dist_path = os.path.join(plots_folder, "Biomarker_Distributions_Top6.png")
    plt.savefig(dist_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"   Saved: Biomarker_Distributions_Top6.png")

# ============================================================================
# CREATE BOX PLOTS FOR GROUP COMPARISONS
# ============================================================================

if len(quartile_df) > 0:
    print("\n📈 Generating box plots for top biomarkers...")

    # Select top 6 biomarkers from quartile analysis
    top_biomarkers_box = quartile_df.head(6)['biomarker'].tolist()

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    for idx, biomarker in enumerate(top_biomarkers_box[:6]):
        try:
            ax = axes[idx]

            # Prepare data for box plot
            biomarker_data = df[[biomarker, survival_status_col]].dropna()

            # Create separate arrays for event and non-event groups
            event_data = biomarker_data[biomarker_data[survival_status_col] == death_value][biomarker]
            censored_data = biomarker_data[biomarker_data[survival_status_col] != death_value][biomarker]

            # Create box plot without labels (we'll add custom ones)
            bp = ax.boxplot([event_data, censored_data],
                           labels=['', ''],  # Empty labels
                           patch_artist=True,
                           showmeans=True,
                           meanprops=dict(marker='o', markerfacecolor='red', markersize=8))

            # Color the boxes
            colors = ['lightcoral', 'lightgreen']
            for patch, color in zip(bp['boxes'], colors):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)

            # Add custom x-axis labels with integrated sample sizes based on survival type
            ax.set_xticks([1, 2])
            if survival_type.upper() == "OVERALL SURVIVAL" or survival_abbrev == "OS":
                ax.set_xticklabels([f'Died\n(n={len(event_data)})',
                                   f'Alive (Censored)\n(n={len(censored_data)})'],
                                  fontsize=10, fontweight='bold')
            else:
                ax.set_xticklabels([f'Event\n(n={len(event_data)})',
                                   f'Censored\n(n={len(censored_data)})'],
                                  fontsize=10, fontweight='bold')

            # Add horizontal line at median
            overall_median = biomarker_data[biomarker].median()
            ax.axhline(overall_median, color='blue', linestyle=':', alpha=0.5,
                      label=f'Overall Median: {overall_median:.2f}')

            # Formatting
            biomarker_display = biomarker if len(biomarker) <= 30 else biomarker[:27] + "..."
            ax.set_title(f'{biomarker_display}', fontsize=12, fontweight='bold')
            ax.set_ylabel('Value', fontsize=10)
            ax.legend(fontsize=8, loc='upper right')
            ax.grid(True, alpha=0.3, axis='y')

        except Exception:
            continue

    plt.suptitle('BIOMARKER DISTRIBUTIONS BY SURVIVAL STATUS', fontsize=16, fontweight='bold')
    plt.tight_layout()
    box_path = os.path.join(plots_folder, "Box_Plots_by_Survival_Top6.png")
    plt.savefig(box_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"   Saved: Box_Plots_by_Survival_Top6.png")

# ============================================================================
# CREATE SUMMARY VISUALIZATIONS
# ============================================================================

if len(cox_df) > 0:
    print("\n📊 Creating summary visualizations...")

    try:
        # 1. Volcano Plot for Cox Regression
        fig, ax = plt.subplots(figsize=(12, 8))
        ax.scatter(cox_df['coef'], -np.log10(cox_df['p_value']), alpha=0.6, s=30)
        ax.axhline(y=-np.log10(0.05), color='r', linestyle='--', alpha=0.5, label='p=0.05')
        ax.axhline(y=-np.log10(0.01), color='darkred', linestyle='--', alpha=0.5, label='p=0.01')
        ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
        ax.set_xlabel('Cox Coefficient', fontsize=12)
        ax.set_ylabel('-log10(p-value)', fontsize=12)
        ax.set_title('Volcano Plot - Cox Regression Results', fontsize=14, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)

        # Annotate top 5 biomarkers
        for i, (_, row) in enumerate(cox_df.head(5).iterrows()):
            ax.annotate(row['biomarker'][:20],
                        xy=(row['coef'], -np.log10(row['p_value'])),
                        xytext=(5, 5), textcoords='offset points',
                        fontsize=8, alpha=0.7)

        volcano_path = os.path.join(plots_folder, "Volcano_Plot_Cox.png")
        plt.savefig(volcano_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"   Saved: Volcano_Plot_Cox.png")
    except Exception:
        pass

    # 2. Forest Plot for Top Cox Results
    if len(cox_df) >= 5:
        try:
            fig, ax = plt.subplots(figsize=(10, 12))
            n_plot = min(20, len(cox_df))
            top20 = cox_df.head(n_plot).sort_values('hr', ascending=True)

            y_pos = np.arange(len(top20))
            ax.errorbarh(y_pos, top20['hr'],
                         xerr=[top20['hr'] - top20['hr_lower'], top20['hr_upper'] - top20['hr']],
                         fmt='o', markersize=5, capsize=3, capthick=1, elinewidth=1)
            ax.axvline(x=1, color='gray', linestyle='--', alpha=0.5)
            ax.set_yticks(y_pos)
            ax.set_yticklabels([b[:30] for b in top20['biomarker']], fontsize=9)
            ax.set_xlabel('Hazard Ratio (95% CI)', fontsize=12)
            ax.set_title(f'Forest Plot - Top {n_plot} Biomarkers (Cox Regression)', fontsize=14, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='x')

            forest_path = os.path.join(plots_folder, f"Forest_Plot_Top{n_plot}.png")
            plt.savefig(forest_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"   Saved: Forest_Plot_Top{n_plot}.png")
        except Exception:
            pass

# ============================================================================
# CREATE HEATMAP OF TOP BIOMARKER CORRELATIONS (ROBUST) — FIXED
# ============================================================================
print("\n📈 Generating correlation heatmap for top biomarkers...")

# Choose candidates (keep order, unique, up to 15)
if len(cox_df) > 0:
    candidates = list(dict.fromkeys(cox_df['biomarker'].tolist()))
elif len(quartile_df) > 0:
    candidates = list(dict.fromkeys(quartile_df['biomarker'].tolist()))
else:
    candidates = []

top_biomarkers = candidates[:15]
present_cols = [c for c in top_biomarkers if c in df.columns]

if len(present_cols) == 0:
    print("⚠️ None of the top biomarkers are present in the dataframe columns. Skipping heatmap.")
else:
    # Coerce to numeric and diagnose columns
    X_raw = df[present_cols].copy()
    X = X_raw.apply(pd.to_numeric, errors='coerce')

    # Thresholds
    min_rows = max(10, int(0.05 * len(df)))  # need at least 5% of rows or min 10 per column
    diag_rows, keep_cols = [], []
    for col in X.columns:
        s = X[col]
        n_nonnull = s.count()
        nunique = s.nunique(dropna=True)
        zero_var = nunique <= 1
        enough_data = n_nonnull >= min_rows
        reason = []
        if zero_var: reason.append("zero variance")
        if not enough_data: reason.append(f"insufficient non-missing ({n_nonnull} < {min_rows})")
        if not reason: keep_cols.append(col)
        diag_rows.append({
            "biomarker": col,
            "dtype_loaded": str(X_raw[col].dtype),
            "non_missing": int(n_nonnull),
            "unique_non_missing": int(nunique),
            "kept_for_corr": len(reason) == 0,
            "exclusion_reason": "; ".join(reason) if reason else ""
        })

    # Save diagnostics
    diag_df = pd.DataFrame(diag_rows)
    diag_csv = os.path.join(RESULTS_FOLDER, "Correlation_Heatmap_Diagnostics.csv")
    diag_df.to_csv(diag_csv, index=False)

    # Compute pairwise-complete correlation only if enough columns survive
    if len(keep_cols) < 2:
        print("⚠️ Not enough valid numeric biomarkers for a correlation heatmap after cleaning.")
        print(f"   See diagnostics: {diag_csv}")
    else:
        corr = X[keep_cols].corr(min_periods=min_rows)
        corr = corr.dropna(axis=0, how='all').dropna(axis=1, how='all')

        if corr.shape[0] < 2 or corr.shape[1] < 2:
            print("⚠️ Correlation matrix is empty after pairwise checks (not enough overlap).")
            print(f"   See diagnostics: {diag_csv}")
        else:
            # Create the figure *only now* to avoid blank frames
            plt.close('all')
            labels = [b[:25] + '...' if len(b) > 25 else b for b in corr.columns]
            plt.figure(figsize=(14, 12))
            sns.heatmap(
                corr, annot=True, fmt='.2f', cmap='coolwarm',
                vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
                cbar_kws={"shrink": 0.8}, xticklabels=labels, yticklabels=labels
            )
            plt.title('CORRELATION MATRIX - TOP BIOMARKERS (cleaned)', fontsize=16, fontweight='bold')
            plt.xticks(rotation=45, ha='right', fontsize=9)
            plt.yticks(rotation=0, fontsize=9)
            plt.tight_layout()

            heatmap_path = os.path.join(RESULTS_FOLDER, "Correlation_Heatmap_TopBiomarkers.png")
            plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"   Saved: {heatmap_path}")
            print(f"   Diagnostics: {diag_csv}")

# ============================================================================
# SAVE ALL RESULTS TO EXCEL (FIXED MULTIVARIATE EXPORT)
# ============================================================================

print("\n" + "=" * 120)
print("💾 SAVING RESULTS")
print("-" * 120)

# Check if we have any results to save
if len(cox_df) == 0 and len(quartile_df) == 0 and len(median_df) == 0:
    print("\n⚠️ No analysis results to save!")
else:
    # Main results file
    output_file = os.path.join(RESULTS_FOLDER, f"Cox_Regression_Results_{timestamp}.xlsx")

    try:
        with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
            # Dataset Summary
            summary_df = pd.DataFrame(dataset_stats.items(), columns=['Metric', 'Value'])
            summary_df.to_excel(writer, sheet_name='Dataset_Summary', index=False)
            print("   ✓ Dataset_Summary")

            # Add column mapping info
            column_mapping = pd.DataFrame([
                ['Patient ID Column', patient_id_col],
                ['Survival Time Column', survival_time_col],
                ['Survival Type', survival_type],
                ['Survival Abbreviation', survival_abbrev],
                ['Survival Status Column', survival_status_col],
                ['Event Value' if not (survival_type.upper() == "OVERALL SURVIVAL" or survival_abbrev == "OS") else 'Death Value', death_value],
                ['Time Unit', time_unit],
                ['Number of Biomarkers', len(biomarker_cols)]
            ], columns=['Parameter', 'Value'])
            column_mapping.to_excel(writer, sheet_name='Column_Mapping', index=False)
            print("   ✓ Column_Mapping")

            # Biomarker Cutoff Values
            if len(cutoff_df) > 0:
                cutoff_df.to_excel(writer, sheet_name='Biomarker_Cutoffs', index=False)
                print(f"   ✓ Biomarker_Cutoffs ({len(cutoff_df)} biomarkers)")

            # Cox Results
            if len(cox_df) > 0:
                cox_df.to_excel(writer, sheet_name='Cox_Full_Results', index=False)
                print(f"   ✓ Cox_Full_Results ({len(cox_df)} biomarkers)")

                cox_df.head(50).to_excel(writer, sheet_name='Cox_Top50', index=False)
                print("   ✓ Cox_Top50")

            # Quartile Results
            if len(quartile_df) > 0:
                quartile_df.to_excel(writer, sheet_name='Quartile_Full_Results', index=False)
                print(f"   ✓ Quartile_Full_Results ({len(quartile_df)} biomarkers)")

            # Median Results
            if len(median_df) > 0:
                median_df.to_excel(writer, sheet_name='Median_Full_Results', index=False)
                print(f"   ✓ Median_Full_Results ({len(median_df)} biomarkers)")

            # FIXE<data_path> Multivariate Results Export
            if multivariate_results is not None and len(multivariate_results.get('biomarkers', [])) > 0:
                try:
                    mv_df = pd.DataFrame(multivariate_results['biomarkers'])

                    # Add comparison with univariate
                    comparison_data = []
                    for _, row in mv_df.iterrows():
                        biomarker = row['name']
                        uv_row = cox_df[cox_df['biomarker'] == biomarker]
                        comparison_data.append({
                            'biomarker': biomarker,
                            'univariate_hr': float(uv_row['hr'].values[0]) if len(uv_row) > 0 else None,
                            'univariate_p': float(uv_row['p_value'].values[0]) if len(uv_row) > 0 else None,
                            'univariate_q': float(uv_row['q_value'].values[0]) if len(uv_row) > 0 else None,
                            'multivariate_hr': float(row['hr']),
                            'multivariate_p': float(row['p_value']),
                            'multivariate_coef': float(row['coef']),
                            'multivariate_se': float(row['se']),
                            'multivariate_hr_lower': float(row['hr_lower']),
                            'multivariate_hr_upper': float(row['hr_upper']),
                            'multivariate_z': float(row['z']),
                            'independent_predictor': row['p_value'] < 0.05
                        })

                    mv_comparison_df = pd.DataFrame(comparison_data)
                    mv_comparison_df.to_excel(writer, sheet_name='Multivariate_Results', index=False)
                    print(f"   ✓ Multivariate_Results ({len(mv_comparison_df)} biomarkers)")

                    # Model statistics
                    model_stats_data = {
                        'Model': 'Multivariate Cox',
                        'N_Samples': multivariate_results['n_samples'],
                        'N_Events': multivariate_results['n_events'],
                        'N_Variables': len(multivariate_results['biomarkers']),
                        'Concordance_Index': multivariate_results['model_concordance'],
                        'Log_Likelihood': multivariate_results['log_likelihood'],
                        'AIC': multivariate_results.get('AIC', None),
                        'BIC': multivariate_results.get('BIC', None)
                    }

                    # Add CV results if available
                    if cv_results:
                        model_stats_data['CV_Mean_C_Index'] = cv_results.get('mean_c_index', None)
                        model_stats_data['CV_Std_C_Index'] = cv_results.get('std_c_index', None)
                        model_stats_data['CV_K_Folds'] = cv_results.get('k_folds', None)
                        model_stats_data['CV_Median_C_Index'] = cv_results.get('median_c_index', None)
                        model_stats_data['CV_Min_C_Index'] = cv_results.get('min_c_index', None)
                        model_stats_data['CV_Max_C_Index'] = cv_results.get('max_c_index', None)
                        model_stats_data['CV_Successful_Runs'] = cv_results.get('n_successful_runs', None)
                        model_stats_data['CV_Optimism'] = multivariate_results['model_concordance'] - cv_results.get('mean_c_index', 0) if cv_results.get('mean_c_index') else None

                    model_stats_df = pd.DataFrame([model_stats_data])
                    model_stats_df.to_excel(writer, sheet_name='Multivariate_Model_Stats', index=False)
                    print("   ✓ Multivariate_Model_Stats")

                    # Stepwise results if available
                    if stepwise_results:
                        stepwise_df = pd.DataFrame({'Selected_Biomarkers': stepwise_results})
                        stepwise_df['Rank'] = range(1, len(stepwise_results) + 1)
                        stepwise_df.to_excel(writer, sheet_name='Stepwise_Selection', index=False)
                        print(f"   ✓ Stepwise_Selection ({len(stepwise_results)} biomarkers)")

                except Exception as e:
                    print(f"   ⚠️ Error saving multivariate results: {str(e)[:50]}")

        print(f"\n✅ Results saved to:")
        print(f"   {output_file}")

    except Exception as e:
        print(f"\n⚠️ Error saving Excel file: {str(e)}")

print("\n✅ Script completed!")

# 2. Multivariate Model Comparison and Champion Selection

## Overview
Compares EPV-capped and Stepwise multivariate Cox models to select the optimal "champion" model.

### What this cell does:
1. **Model Comparison**: Side-by-side analysis of EPV-capped vs Stepwise models
2. **Visualization**: Forest plots, C-index comparison, Venn diagrams
3. **Champion Selection**: Interactive selection of the best-performing model
4. **Cross-Validation**: 10x5-fold CV for model stability assessment

---

## ⚙️ TUNABLE PARAMETERS

| Parameter | Default | Description | Location |
|-----------|---------|-------------|----------|
| `CV_FOLDS` | `5` | Number of cross-validation folds | CV section
| `CV_REPETITIONS` | `10` |
| `BASE_SEED` | `42` |
| `alpha` | `0.05` | Significance threshold for selection | Stepwise section |

---

### Key Outputs:
- `Model_Comparison_{timestamp}.xlsx`
- Comparison plots in `RESULTS_FOLDER/Model_Comparison/`

### Dependencies:
- Requires Cell 0 to be executed first


In [ ]:
# ============================================================================
# MULTIVARIATE MODEL COMPARISON AND CHAMPION SELECTION (WITH VISUALIZATIONS)
# Standalone cell - Run after main Cox regression analysis
# Shows comprehensive visualizations BEFORE user selects champion model
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from lifelines import CoxPHFitter
import os

print("=" * 120)
print("🏆 MULTIVARIATE MODEL COMPARISON AND CHAMPION SELECTION")
print("=" * 120)

# Check if we have the necessary results from previous analysis
if 'multivariate_results' not in globals() or multivariate_results is None:
    print("\n❌ Error: No multivariate results found from previous analysis")
    print("   Please run the main Cox regression analysis first")
else:
    has_epv_model = multivariate_results is not None and len(multivariate_results.get('biomarkers', [])) > 0
    has_stepwise_model = 'stepwise_results' in globals() and stepwise_results is not None and len(stepwise_results) > 0

    if not has_epv_model:
        print("\n❌ Error: EPV-cap model not available")
    elif not has_stepwise_model:
        print("\n⚠️ No stepwise model found - only EPV-cap model available")
        print("   Saving EPV-cap model as champion by default...")

        # Get EPV model statistics
        epv_train_c_index = multivariate_results['model_concordance']
        epv_n_vars = len(selected_biomarkers_for_multivariate)
        epv_cv_mean = cv_results['mean_c_index'] if 'cv_results' in globals() and cv_results else None
        epv_cv_std = cv_results['std_c_index'] if 'cv_results' in globals() and cv_results else None
        epv_optimism = epv_train_c_index - epv_cv_mean if epv_cv_mean else None
        epv_AIC = multivariate_results.get('AIC')
        epv_BIC = multivariate_results.get('BIC')

        # Save EPV model as champion with complete structure
        champion_model = {
            'type': 'EPV-cap',
            'biomarkers': selected_biomarkers_for_multivariate,
            'formula': ' + '.join([f'Q("{b}")' for b in selected_biomarkers_for_multivariate]),
            'n_variables': epv_n_vars,
            'train_c_index': epv_train_c_index,
            'cv_c_index': epv_cv_mean,
            'cv_std': epv_cv_std,
            'optimism': epv_optimism,
            'AIC': epv_AIC,
            'BIC': epv_BIC,
            'results': multivariate_results,
            'cv_results': cv_results if 'cv_results' in globals() else None
        }

        # Store in globals for other cells
        globals()['champion_model'] = champion_model
        globals()['champion_biomarkers'] = selected_biomarkers_for_multivariate
        globals()['champion_formula'] = champion_model['formula']
        globals()['champion_type'] = 'EPV-cap'

        print(f"\n✅ EPV-cap model saved as champion")
        print(f"   Variables: {champion_model['n_variables']}")
        print(f"   Training C-index: {champion_model['train_c_index']:.4f}")
        if epv_cv_mean:
            print(f"   CV C-index: {epv_cv_mean:.4f} ± {epv_cv_std:.4f}")
        if epv_AIC:
            print(f"   AIC: {epv_AIC:.2f}")

        print("\n📦 Champion model saved to global variables:")
        print("   • champion_model, champion_biomarkers, champion_formula, champion_type")

    else:
        # ============================================================================
        # STEP 1: PERFORM 10x10 STRATIFIED CV ON STEPWISE MODEL
        # ============================================================================

        print("\n" + "=" * 120)
        print("📊 CROSS-VALIDATION FOR STEPWISE MODEL")
        print("-" * 120)

        print(f"\n🔄 Performing 10-fold stratified CV (10 repetitions) for stepwise model...")
        print(f"   Stepwise model has {len(stepwise_results)} variables")

        try:
            from sklearn.model_selection import StratifiedKFold

            # CV parameters
            N_FOLDS = 5
            N_REPETITIONS = 10
            BASE_SEED = 42

            # Prepare stepwise data
            stepwise_data = df[[survival_time_col, 'event'] + stepwise_results].dropna()
            stepwise_formula = ' + '.join([f'Q("{v}")' for v in stepwise_results])

            # Storage for CV scores
            stepwise_cv_scores = []
            stepwise_rep_means = []
            stepwise_rep_stds = []

            print(f"   Dataset: {len(stepwise_data)} samples, {stepwise_data['event'].sum()} events")
            print(f"   Formula: {stepwise_formula[:80]}{'...' if len(stepwise_formula) > 80 else ''}")
            print("\n   Running cross-validation...")

            # Perform repeated stratified CV
            for rep in range(N_REPETITIONS):
                seed = BASE_SEED + rep
                np.random.seed(seed)

                # Stratified K-Fold split
                skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

                fold_scores = []
                for fold_idx, (train_idx, test_idx) in enumerate(skf.split(stepwise_data, stepwise_data['event'])):
                    try:
                        # Split data
                        train_data = stepwise_data.iloc[train_idx]
                        test_data = stepwise_data.iloc[test_idx]

                        # Check minimum requirements
                        if len(train_data) < 20 or len(test_data) < 5:
                            continue
                        if train_data['event'].sum() < 5 or test_data['event'].sum() < 2:
                            continue

                        # Fit model on training data
                        cph_cv = CoxPHFitter(penalizer=PENALIZER)
                        cph_cv.fit(
                            train_data,
                            duration_col=survival_time_col,
                            event_col='event',
                            formula=stepwise_formula,
                            robust=ROBUST_SE
                        )

                        # Evaluate on test data
                        c_index = cph_cv.score(test_data, scoring_method="concordance_index")
                        fold_scores.append(c_index)
                        stepwise_cv_scores.append(c_index)

                    except Exception as e:
                        continue

                # Store repetition statistics
                if len(fold_scores) > 0:
                    rep_mean = np.mean(fold_scores)
                    rep_std = np.std(fold_scores)
                    stepwise_rep_means.append(rep_mean)
                    stepwise_rep_stds.append(rep_std)
                    print(f"   Rep {rep+1:2d}: {len(fold_scores):2d} folds, C-index = {rep_mean:.4f} ± {rep_std:.4f}")

            # Calculate overall statistics
            if len(stepwise_cv_scores) > 0:
                stepwise_cv_results = {
                    'mean_c_index': np.mean(stepwise_cv_scores),
                    'std_c_index': np.std(stepwise_cv_scores),
                    'median_c_index': np.median(stepwise_cv_scores),
                    'min_c_index': np.min(stepwise_cv_scores),
                    'max_c_index': np.max(stepwise_cv_scores),
                    'n_successful_runs': len(stepwise_cv_scores),
                    'n_folds': N_FOLDS,
                    'n_repetitions': N_REPETITIONS,
                    'repetition_mean_of_means': np.mean(stepwise_rep_means),
                    'repetition_std_of_means': np.std(stepwise_rep_means),
                    'coefficient_of_variation': (np.std(stepwise_cv_scores) / np.mean(stepwise_cv_scores)) * 100
                }

                print(f"\n   ✅ Cross-Validation Results (Stepwise):")
                print(f"      Successful runs: {stepwise_cv_results['n_successful_runs']}/{N_FOLDS * N_REPETITIONS}")
                print(f"      Mean C-index: {stepwise_cv_results['mean_c_index']:.4f} ± {stepwise_cv_results['std_c_index']:.4f}")
                print(f"      Median C-index: {stepwise_cv_results['median_c_index']:.4f}")
                print(f"      Range: [{stepwise_cv_results['min_c_index']:.4f}, {stepwise_cv_results['max_c_index']:.4f}]")
                print(f"      Repetition variability: {stepwise_cv_results['repetition_std_of_means']:.4f}")

                # Fit final stepwise model for training C-index
                print("\n   📊 Fitting final stepwise model on full data...")
                cph_stepwise = CoxPHFitter(penalizer=PENALIZER)
                cph_stepwise.fit(
                    stepwise_data,
                    duration_col=survival_time_col,
                    event_col='event',
                    formula=stepwise_formula,
                    robust=ROBUST_SE
                )

                # Extract model statistics
                stepwise_train_c_index = float(cph_stepwise.concordance_index_)
                stepwise_optimism = stepwise_train_c_index - stepwise_cv_results['mean_c_index']
                stepwise_AIC = _get_attr(cph_stepwise, ["AIC_partial_", "AIC_", "AIC"], None)
                stepwise_BIC = _get_attr(cph_stepwise, ["BIC_partial_", "BIC_", "BIC"], None)

                print(f"      Training C-index: {stepwise_train_c_index:.4f}")
                print(f"      CV C-index: {stepwise_cv_results['mean_c_index']:.4f}")
                print(f"      Optimism (bias): {stepwise_optimism:.4f}")

                if stepwise_AIC is not None:
                    print(f"      AIC: {stepwise_AIC:.2f}")
                if stepwise_BIC is not None:
                    print(f"      BIC: {stepwise_BIC:.2f}")

                if stepwise_optimism > 0.05:
                    print(f"      ⚠️ Substantial optimism detected (>{0.05:.3f}) - model may be overfitting")
                elif stepwise_optimism > 0.03:
                    print(f"      ⚡ Moderate optimism detected")
                else:
                    print(f"      ✓ Low optimism - model appears stable")

                # Extract stepwise biomarkers
                stepwise_mv_biomarkers = []
                for biomarker_name in stepwise_results:
                    param_name = None
                    if biomarker_name in cph_stepwise.params_.index:
                        param_name = biomarker_name
                    elif f'Q("{biomarker_name}")' in cph_stepwise.params_.index:
                        param_name = f'Q("{biomarker_name}")'
                    elif f"Q('{biomarker_name}')" in cph_stepwise.params_.index:
                        param_name = f"Q('{biomarker_name}')"

                    if param_name:
                        try:
                            ci_cols = cph_stepwise.confidence_intervals_.columns
                            stepwise_mv_biomarkers.append({
                                'name': biomarker_name,
                                'coef': float(cph_stepwise.params_[param_name]),
                                'se': float(cph_stepwise.standard_errors_[param_name]),
                                'hr': float(np.exp(cph_stepwise.params_[param_name])),
                                'hr_lower': float(np.exp(cph_stepwise.confidence_intervals_.loc[param_name, ci_cols[0]])),
                                'hr_upper': float(np.exp(cph_stepwise.confidence_intervals_.loc[param_name, ci_cols[1]])),
                                'p_value': float(cph_stepwise.summary.loc[param_name, 'p']),
                                'z': float(cph_stepwise.summary.loc[param_name, 'z'])
                            })
                        except Exception as e:
                            continue

                # Store stepwise results
                stepwise_results_dict = {
                    'biomarkers': stepwise_mv_biomarkers,
                    'model_concordance': stepwise_train_c_index,
                    'log_likelihood': float(cph_stepwise.log_likelihood_),
                    'AIC': stepwise_AIC,
                    'BIC': stepwise_BIC,
                    'n_samples': len(stepwise_data),
                    'n_events': int(stepwise_data['event'].sum())
                }

            else:
                print("\n   ❌ CV failed: no successful runs")
                stepwise_cv_results = None

        except Exception as e:
            print(f"\n   ❌ Error during stepwise CV: {str(e)}")
            import traceback
            traceback.print_exc()
            stepwise_cv_results = None

        # ============================================================================
        # STEP 2: COMPARE BOTH MODELS (TEXT COMPARISON)
        # ============================================================================

        print("\n" + "=" * 120)
        print("📊 MODEL COMPARISON")
        print("-" * 120)

        # Gather EPV model statistics
        epv_train_c_index = multivariate_results['model_concordance']
        epv_AIC = multivariate_results.get('AIC')
        epv_BIC = multivariate_results.get('BIC')
        epv_n_vars = len(multivariate_results['biomarkers'])
        epv_cv_mean = cv_results['mean_c_index'] if 'cv_results' in globals() and cv_results else None
        epv_cv_std = cv_results['std_c_index'] if 'cv_results' in globals() and cv_results else None
        epv_optimism = epv_train_c_index - epv_cv_mean if epv_cv_mean else None

        # Add coefficient of variation if missing
        if 'cv_results' in globals() and cv_results and 'coefficient_of_variation' not in cv_results:
            cv_results['coefficient_of_variation'] = (cv_results['std_c_index'] / cv_results['mean_c_index']) * 100

        # Gather Stepwise model statistics
        stepwise_n_vars = len(stepwise_results)
        stepwise_train_c_index_final = stepwise_train_c_index if stepwise_cv_results else None
        stepwise_AIC_final = stepwise_AIC if stepwise_cv_results else None
        stepwise_BIC_final = stepwise_BIC if stepwise_cv_results else None
        stepwise_cv_mean = stepwise_cv_results['mean_c_index'] if stepwise_cv_results else None
        stepwise_cv_std = stepwise_cv_results['std_c_index'] if stepwise_cv_results else None
        stepwise_optimism_final = stepwise_optimism if stepwise_cv_results else None

        # Create comparison table
        print("\n📋 COMPREHENSIVE MODEL COMPARISON")
        print("=" * 120)
        print(f"{'Metric':<35} {'EPV-Cap Model':>20} {'Stepwise Model':>20} {'Difference':>20}")
        print("-" * 120)

        # Number of variables
        print(f"{'Number of Variables':<35} {epv_n_vars:>20} {stepwise_n_vars:>20} {stepwise_n_vars - epv_n_vars:>+20}")

        # Training C-index
        if stepwise_train_c_index_final:
            diff = stepwise_train_c_index_final - epv_train_c_index
            winner = " ✓" if diff > 0 else ""
            print(f"{'Training C-index':<35} {epv_train_c_index:>20.4f} {stepwise_train_c_index_final:>20.4f} {diff:>+19.4f}{winner}")
        else:
            print(f"{'Training C-index':<35} {epv_train_c_index:>20.4f} {'N/A':>20} {'N/A':>20}")

        # CV C-index
        if epv_cv_mean and stepwise_cv_mean:
            diff = stepwise_cv_mean - epv_cv_mean
            winner = " ✓" if diff > 0 else ""
            print(f"{'CV Mean C-index':<35} {epv_cv_mean:>20.4f} {stepwise_cv_mean:>20.4f} {diff:>+19.4f}{winner}")
            print(f"{'CV Std C-index':<35} {epv_cv_std:>20.4f} {stepwise_cv_std:>20.4f} {stepwise_cv_std - epv_cv_std:>+20.4f}")
        elif epv_cv_mean:
            print(f"{'CV Mean C-index':<35} {epv_cv_mean:>20.4f} {'N/A':>20} {'N/A':>20}")
        elif stepwise_cv_mean:
            print(f"{'CV Mean C-index':<35} {'N/A':>20} {stepwise_cv_mean:>20.4f} {'N/A':>20}")

        # Optimism
        if epv_optimism and stepwise_optimism_final:
            diff = abs(stepwise_optimism_final) - abs(epv_optimism)
            winner = " ✓" if diff < 0 else ""  # Lower optimism is better
            print(f"{'Optimism (Bias)':<35} {epv_optimism:>20.4f} {stepwise_optimism_final:>20.4f} {diff:>+19.4f}{winner}")

        # AIC
        if epv_AIC and stepwise_AIC_final:
            diff = stepwise_AIC_final - epv_AIC
            winner = " ✓" if diff < 0 else ""  # Lower AIC is better
            print(f"{'AIC':<35} {epv_AIC:>20.2f} {stepwise_AIC_final:>20.2f} {diff:>+19.2f}{winner}")
        elif epv_AIC:
            print(f"{'AIC':<35} {epv_AIC:>20.2f} {'N/A':>20} {'N/A':>20}")
        elif stepwise_AIC_final:
            print(f"{'AIC':<35} {'N/A':>20} {stepwise_AIC_final:>20.2f} {'N/A':>20}")

        # BIC
        if epv_BIC and stepwise_BIC_final:
            diff = stepwise_BIC_final - epv_BIC
            winner = " ✓" if diff < 0 else ""  # Lower BIC is better
            print(f"{'BIC':<35} {epv_BIC:>20.2f} {stepwise_BIC_final:>20.2f} {diff:>+19.2f}{winner}")
        elif epv_BIC:
            print(f"{'BIC':<35} {epv_BIC:>20.2f} {'N/A':>20} {'N/A':>20}")
        elif stepwise_BIC_final:
            print(f"{'BIC':<35} {'N/A':>20} {stepwise_BIC_final:>20.2f} {'N/A':>20}")

        print("=" * 120)

        # ============================================================================
        # STEP 3: PREPARE MODEL DATA FOR VISUALIZATION
        # ============================================================================

        print("\n" + "=" * 120)
        print("📊 PREPARING MODEL DATA FOR VISUALIZATION")
        print("-" * 120)

        # ----------------------------------------------------------------------------
        # Verify/Extract EPV Model Data
        # ----------------------------------------------------------------------------

        print("\n🔧 Processing EPV Model Data...")

        epv_biomarkers_list = selected_biomarkers_for_multivariate
        epv_results = multivariate_results

        # Check if EPV biomarkers are properly formatted
        if 'biomarkers' in epv_results and len(epv_results['biomarkers']) > 0:
            # Verify data structure
            sample = epv_results['biomarkers'][0]
            if 'p_value' in sample and 'name' in sample:
                print(f"   ✓ EPV model data valid: {len(epv_results['biomarkers'])} biomarkers")
                epv_mv_biomarkers = epv_results['biomarkers']
            else:
                print(f"   ⚠️ EPV biomarkers missing required fields, reconstructing...")
                epv_mv_biomarkers = None
        else:
            print(f"   ⚠️ EPV biomarkers not found, reconstructing...")
            epv_mv_biomarkers = None

        # Reconstruct EPV model if needed
        if epv_mv_biomarkers is None:
            print(f"   🔧 Reconstructing EPV model with {len(epv_biomarkers_list)} variables...")

            try:
                epv_data = df[[survival_time_col, 'event'] + epv_biomarkers_list].dropna()
                epv_formula = ' + '.join([f'Q("{b}")' for b in epv_biomarkers_list])

                cph_epv = CoxPHFitter(penalizer=PENALIZER)
                cph_epv.fit(epv_data, duration_col=survival_time_col, event_col='event',
                           formula=epv_formula, robust=ROBUST_SE)

                print(f"   ✓ EPV model fitted: C-index = {float(cph_epv.concordance_index_):.4f}")

                # Extract biomarkers
                epv_mv_biomarkers = []
                for biomarker_name in epv_biomarkers_list:
                    param_name = None

                    # Try multiple formats
                    if biomarker_name in cph_epv.params_.index:
                        param_name = biomarker_name
                    elif f'Q("{biomarker_name}")' in cph_epv.params_.index:
                        param_name = f'Q("{biomarker_name}")'
                    elif f"Q('{biomarker_name}')" in cph_epv.params_.index:
                        param_name = f"Q('{biomarker_name}')"

                    if param_name:
                        try:
                            ci_cols = cph_epv.confidence_intervals_.columns
                            epv_mv_biomarkers.append({
                                'name': biomarker_name,
                                'coef': float(cph_epv.params_[param_name]),
                                'se': float(cph_epv.standard_errors_[param_name]),
                                'hr': float(np.exp(cph_epv.params_[param_name])),
                                'hr_lower': float(np.exp(cph_epv.confidence_intervals_.loc[param_name, ci_cols[0]])),
                                'hr_upper': float(np.exp(cph_epv.confidence_intervals_.loc[param_name, ci_cols[1]])),
                                'p_value': float(cph_epv.summary.loc[param_name, 'p']),
                                'z': float(cph_epv.summary.loc[param_name, 'z'])
                            })
                        except Exception as e:
                            print(f"      ⚠️ Failed to extract {biomarker_name}: {str(e)}")

                print(f"   ✓ Extracted {len(epv_mv_biomarkers)} EPV biomarkers")

                # Update EPV results
                epv_results['biomarkers'] = epv_mv_biomarkers

            except Exception as e:
                print(f"   ❌ Failed to reconstruct EPV model: {str(e)}")
                import traceback
                traceback.print_exc()
                raise

        # ----------------------------------------------------------------------------
        # Verify Stepwise Model Data
        # ----------------------------------------------------------------------------

        print(f"\n🔧 Processing Stepwise Model Data...")
        print(f"   ✓ Stepwise model data valid: {len(stepwise_mv_biomarkers)} biomarkers")

        # ============================================================================
        # STEP 4: GENERATE VISUAL COMPARISONS (BEFORE USER SELECTION)
        # ============================================================================

        print("\n" + "=" * 120)
        print("📊 GENERATING VISUAL COMPARISONS")
        print("-" * 120)

        comparison_folder = os.path.join(RESULTS_FOLDER, "Model_Comparison")
        os.makedirs(comparison_folder, exist_ok=True)
        print(f"\n📁 Output folder: {comparison_folder}")

        # Prepare biomarker lists and dataframes
        stepwise_biomarkers_list = stepwise_results

        epv_set = set(epv_biomarkers_list)
        stepwise_set = set(stepwise_biomarkers_list)

        # Create dataframes with verified data
        epv_mv_df = pd.DataFrame(epv_mv_biomarkers).sort_values('p_value')
        stepwise_mv_df = pd.DataFrame(stepwise_mv_biomarkers).sort_values('p_value')

        print(f"\n   EPV DataFrame: {len(epv_mv_df)} biomarkers")
        print(f"   Stepwise DataFrame: {len(stepwise_mv_df)} biomarkers")

        # Get CV results
        epv_cv = cv_results if 'cv_results' in globals() and cv_results else None
        stepwise_cv = stepwise_cv_results

        # ----------------------------------------------------------------------------
        # VISUALIZATION 1: SIDE-BY-SIDE FOREST PLOTS
        # ----------------------------------------------------------------------------

        print("\n📈 1. Generating Side-by-Side Forest Plots...")

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, max(10, max(len(epv_biomarkers_list), len(stepwise_biomarkers_list)) * 0.6)))

        # EPV Forest Plot
        for i, (idx, row) in enumerate(epv_mv_df.iterrows()):
            color = 'darkred' if row['p_value'] < 0.001 else 'red' if row['p_value'] < 0.01 else 'orange' if row['p_value'] < 0.05 else 'gray'
            ax1.plot([row['hr_lower'], row['hr_upper']], [i, i], color=color, linewidth=2, alpha=0.7)
            ax1.scatter(row['hr'], i, color=color, s=80, zorder=5)

        ax1.axvline(x=1, color='black', linestyle='--', alpha=0.5)
        ax1.set_yticks(range(len(epv_mv_df)))
        ax1.set_yticklabels([name[:30] + '...' if len(name) > 30 else name for name in epv_mv_df['name']], fontsize=10)
        ax1.set_xlabel('Hazard Ratio', fontsize=12, fontweight='bold')
        ax1.set_title(f'EPV-CAP MODEL\n({len(epv_biomarkers_list)} variables, C={epv_train_c_index:.3f})',
                     fontsize=13, fontweight='bold', color='#2E5090')
        ax1.grid(True, alpha=0.3, axis='x')

        # Stepwise Forest Plot
        for i, (idx, row) in enumerate(stepwise_mv_df.iterrows()):
            color = 'darkred' if row['p_value'] < 0.001 else 'red' if row['p_value'] < 0.01 else 'orange' if row['p_value'] < 0.05 else 'gray'
            ax2.plot([row['hr_lower'], row['hr_upper']], [i, i], color=color, linewidth=2, alpha=0.7)
            ax2.scatter(row['hr'], i, color=color, s=80, zorder=5)

        ax2.axvline(x=1, color='black', linestyle='--', alpha=0.5)
        ax2.set_yticks(range(len(stepwise_mv_df)))
        ax2.set_yticklabels([name[:30] + '...' if len(name) > 30 else name for name in stepwise_mv_df['name']], fontsize=10)
        ax2.set_xlabel('Hazard Ratio', fontsize=12, fontweight='bold')
        ax2.set_title(f'STEPWISE MODEL\n({len(stepwise_biomarkers_list)} variables, C={stepwise_train_c_index_final:.3f})',
                     fontsize=13, fontweight='bold', color='#2E5090')
        ax2.grid(True, alpha=0.3, axis='x')

        # Add legend
        legend_elements = [
            Patch(facecolor='darkred', alpha=0.7, label='p < 0.001'),
            Patch(facecolor='red', alpha=0.7, label='p < 0.01'),
            Patch(facecolor='orange', alpha=0.7, label='p < 0.05'),
            Patch(facecolor='gray', alpha=0.7, label='p ≥ 0.05')
        ]
        fig.legend(handles=legend_elements, loc='lower center', ncol=4,
                  bbox_to_anchor=(0.5, -0.02), title='Significance', frameon=True)

        plt.suptitle('MODEL COMPARISON: EPV-CAP vs STEPWISE (Forest Plots)', fontsize=16, fontweight='bold')
        plt.tight_layout(rect=[0, 0.02, 1, 1])
        forest_comp_path = os.path.join(comparison_folder, "01_EPV_vs_Stepwise_Forest_Comparison.png")
        plt.savefig(forest_comp_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"   ✓ Saved: 01_EPV_vs_Stepwise_Forest_Comparison.png")

        # ----------------------------------------------------------------------------
        # VISUALIZATION 2: C-INDEX COMPARISON
        # ----------------------------------------------------------------------------

        print("\n📈 2. Generating C-Index Comparison...")

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

        models = ['EPV-Cap', 'Stepwise']
        train_c_vals = [epv_train_c_index, stepwise_train_c_index_final]
        colors = ['steelblue', 'coral']

        bars1 = ax1.bar(models, train_c_vals, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
        ax1.set_ylabel('C-index', fontsize=12, fontweight='bold')
        ax1.set_title('TRAINING C-INDEX COMPARISON', fontsize=13, fontweight='bold')
        ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
        ax1.set_ylim([0.4, 1.0])
        ax1.grid(True, alpha=0.3, axis='y')
        ax1.legend()

        for bar, val in zip(bars1, train_c_vals):
            ax1.text(bar.get_x() + bar.get_width()/2., val + 0.01,
                    f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

        # CV comparison
        if epv_cv and stepwise_cv:
            cv_means = [epv_cv['mean_c_index'], stepwise_cv['mean_c_index']]
            cv_stds = [epv_cv['std_c_index'], stepwise_cv['std_c_index']]

            bars2 = ax2.bar(models, cv_means, yerr=cv_stds, color=colors,
                           alpha=0.7, edgecolor='black', linewidth=2, capsize=5)
            ax2.set_ylabel('C-index', fontsize=12, fontweight='bold')
            ax2.set_title('CROSS-VALIDATION C-INDEX COMPARISON', fontsize=13, fontweight='bold')
            ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
            ax2.set_ylim([0.4, 1.0])
            ax2.grid(True, alpha=0.3, axis='y')
            ax2.legend()

            for bar, mean, std in zip(bars2, cv_means, cv_stds):
                ax2.text(bar.get_x() + bar.get_width()/2., mean + std + 0.01,
                        f'{mean:.4f}±{std:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
        else:
            ax2.text(0.5, 0.5, 'CV data not available\nfor both models', ha='center', va='center',
                    fontsize=12, transform=ax2.transAxes)
            ax2.set_xticks([])
            ax2.set_yticks([])

        plt.suptitle('C-INDEX COMPARISON: EPV-CAP vs STEPWISE', fontsize=15, fontweight='bold')
        plt.tight_layout()
        cindex_comp_path = os.path.join(comparison_folder, "02_EPV_vs_Stepwise_CIndex_Comparison.png")
        plt.savefig(cindex_comp_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"   ✓ Saved: 02_EPV_vs_Stepwise_CIndex_Comparison.png")

        # ----------------------------------------------------------------------------
        # VISUALIZATION 3: VARIABLE OVERLAP VENN DIAGRAM
        # ----------------------------------------------------------------------------

        print("\n📈 3. Generating Variable Overlap Venn Diagram...")

        try:
            from matplotlib_venn import venn2

            fig, ax = plt.subplots(figsize=(10, 8))

            only_epv = epv_set - stepwise_set
            only_stepwise = stepwise_set - epv_set
            both = epv_set & stepwise_set

            venn = venn2(subsets=(len(only_epv), len(only_stepwise), len(both)),
                         set_labels=('EPV-Cap', 'Stepwise'),
                         ax=ax)

            if venn.get_patch_by_id('10'):
                venn.get_patch_by_id('10').set_color('lightblue')
                venn.get_patch_by_id('10').set_alpha(0.7)
            if venn.get_patch_by_id('01'):
                venn.get_patch_by_id('01').set_color('lightcoral')
                venn.get_patch_by_id('01').set_alpha(0.7)
            if venn.get_patch_by_id('11'):
                venn.get_patch_by_id('11').set_color('lightgreen')
                venn.get_patch_by_id('11').set_alpha(0.7)

            if venn.get_label_by_id('A'):
                venn.get_label_by_id('A').set_position((-0.4, 0.3))
                venn.get_label_by_id('A').set_fontsize(14)
                venn.get_label_by_id('A').set_fontweight('bold')
            if venn.get_label_by_id('B'):
                venn.get_label_by_id('B').set_position((0.4, 0.3))
                venn.get_label_by_id('B').set_fontsize(14)
                venn.get_label_by_id('B').set_fontweight('bold')

            for text in venn.subset_labels:
                if text:
                    text.set_fontsize(16)
                    text.set_fontweight('bold')

            ax.set_title('VARIABLE OVERLAP: EPV-CAP vs STEPWISE MODEL',
                        fontsize=14, fontweight='bold', pad=20)

            info_text = f"ONLY IN EPV-CAP ({len(only_epv)}):\n"
            for var in sorted(list(only_epv))[:8]:
                info_text += f"  • {var[:35]}\n"
            if len(only_epv) > 8:
                info_text += f"  ... and {len(only_epv)-8} more\n"

            info_text += f"\nONLY IN STEPWISE ({len(only_stepwise)}):\n"
            for var in sorted(list(only_stepwise))[:8]:
                info_text += f"  • {var[:35]}\n"
            if len(only_stepwise) > 8:
                info_text += f"  ... and {len(only_stepwise)-8} more\n"

            info_text += f"\nSHARED BY BOTH ({len(both)}):\n"
            for var in sorted(list(both))[:12]:
                info_text += f"  • {var[:35]}\n"
            if len(both) > 12:
                info_text += f"  ... and {len(both)-12} more"

            ax.text(1.15, 0.5, info_text, transform=ax.transAxes, fontsize=12,
                   verticalalignment='center', fontfamily='monospace',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

            plt.tight_layout()
            venn_path = os.path.join(comparison_folder, "03_EPV_vs_Stepwise_Variable_Overlap.png")
            plt.savefig(venn_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"   ✓ Saved: 03_EPV_vs_Stepwise_Variable_Overlap.png")

        except ImportError:
            print("   ⚠️ matplotlib_venn not available, skipping Venn diagram")
        except Exception as e:
            print(f"   ⚠️ Error creating Venn diagram: {str(e)}")

        # ----------------------------------------------------------------------------
        # VISUALIZATION 4: COMPREHENSIVE SUMMARY TABLE
        # ----------------------------------------------------------------------------

        print("\n📈 4. Generating Comprehensive Summary Table...")

        fig, ax = plt.subplots(figsize=(16, 10))
        ax.axis('off')

        overlap = len(both)

        # Create summary table
        summary_data = []
        summary_data.append(['METRIC', 'EPV-CAP', 'STEPWISE', 'WINNER'])

        summary_data.append([
            'Number of Variables',
            f'{epv_n_vars}',
            f'{stepwise_n_vars}',
            '✓ SW' if stepwise_n_vars < epv_n_vars else '✓ EPV' if epv_n_vars < stepwise_n_vars else 'Tie'
        ])

        winner_train = '✓ SW' if stepwise_train_c_index_final > epv_train_c_index else '✓ EPV' if epv_train_c_index > stepwise_train_c_index_final else 'Tie'
        summary_data.append([
            'Training C-index',
            f'{epv_train_c_index:.4f}',
            f'{stepwise_train_c_index_final:.4f}',
            winner_train
        ])

        if epv_cv and stepwise_cv:
            epv_cv_val = epv_cv['mean_c_index']
            sw_cv_val = stepwise_cv['mean_c_index']
            winner_cv = '✓ SW' if sw_cv_val > epv_cv_val else '✓ EPV' if epv_cv_val > sw_cv_val else 'Tie'
            summary_data.append([
                'CV C-index (mean)',
                f'{epv_cv_val:.4f}',
                f'{sw_cv_val:.4f}',
                winner_cv
            ])

            epv_opt = epv_train_c_index - epv_cv_val
            sw_opt = stepwise_train_c_index_final - sw_cv_val
            summary_data.append([
                'Optimism',
                f'{epv_opt:+.4f}',
                f'{sw_opt:+.4f}',
                '✓ SW' if abs(sw_opt) < abs(epv_opt) else '✓ EPV'
            ])

            if 'coefficient_of_variation' in epv_cv and 'coefficient_of_variation' in stepwise_cv:
                epv_cv_pct = epv_cv['coefficient_of_variation']
                sw_cv_pct = stepwise_cv['coefficient_of_variation']
                summary_data.append([
                    'CV % (lower = better)',
                    f'{epv_cv_pct:.2f}%',
                    f'{sw_cv_pct:.2f}%',
                    '✓ SW' if sw_cv_pct < epv_cv_pct else '✓ EPV'
                ])

        if epv_AIC and stepwise_AIC_final:
            summary_data.append([
                'AIC (lower = better)',
                f'{epv_AIC:.1f}',
                f'{stepwise_AIC_final:.1f}',
                '✓ SW' if stepwise_AIC_final < epv_AIC else '✓ EPV'
            ])

        summary_data.append([
            'Shared variables',
            f'{overlap}',
            f'{overlap}',
            f'{overlap}/{max(epv_n_vars, stepwise_n_vars)}'
        ])

        summary_data.append([
            'Model Complexity',
            'Higher' if epv_n_vars > stepwise_n_vars else 'Lower',
            'Higher' if stepwise_n_vars > epv_n_vars else 'Lower',
            'SW' if stepwise_n_vars < epv_n_vars else 'EPV'
        ])

        table = ax.table(cellText=summary_data, cellLoc='center', loc='upper center',
                        colWidths=[0.35, 0.2, 0.2, 0.15])

        table.auto_set_font_size(False)
        table.set_fontsize(11)
        table.scale(1, 2.5)

        for i in range(4):
            cell = table[(0, i)]
            cell.set_facecolor('#2E5090')
            cell.set_text_props(weight='bold', color='white', fontsize=13)

        for i in range(1, len(summary_data)):
            for j in range(4):
                cell = table[(i, j)]
                if j == 3:
                    if '✓ SW' in summary_data[i][j]:
                        cell.set_facecolor('#FFE5E5')
                    elif '✓ EPV' in summary_data[i][j]:
                        cell.set_facecolor('#E5F2FF')
                else:
                    cell.set_facecolor('#F5F5F5' if i % 2 == 0 else 'white')

        ax.set_title('COMPREHENSIVE MODEL COMPARISON SUMMARY',
                    fontsize=18, fontweight='bold', pad=20)

        interp_text = "KEY FINDINGS:\n"
        interp_text += f"• {abs(epv_n_vars - stepwise_n_vars)} variable difference between models\n"
        interp_text += f"• {overlap} variables ({overlap/max(epv_n_vars, stepwise_n_vars)*100:.1f}%) shared between both models\n"
        interp_text += f"• Training C-index difference: {abs(epv_train_c_index - stepwise_train_c_index_final):.4f}\n"
        if epv_cv and stepwise_cv:
            interp_text += f"• CV C-index difference: {abs(epv_cv_val - sw_cv_val):.4f}\n"
            interp_text += f"• Optimism: EPV={epv_opt:+.4f}, Stepwise={sw_opt:+.4f}"

        ax.text(0.5, 0.35, interp_text, transform=ax.transAxes,
               fontsize=11, ha='center', va='top', weight='bold',
               bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7, pad=1))

        plt.tight_layout()
        summary_path = os.path.join(comparison_folder, "04_EPV_vs_Stepwise_Summary_Table.png")
        plt.savefig(summary_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"   ✓ Saved: 04_EPV_vs_Stepwise_Summary_Table.png")

        print(f"\n✅ All comparison visualizations generated!")

        # ============================================================================
        # STEP 4: PROVIDE RECOMMENDATION
        # ============================================================================

        print("\n" + "=" * 120)
        print("📊 MODEL SELECTION GUIDANCE")
        print("-" * 120)

        recommendation_score = 0
        reasons = []

        # Score based on CV performance
        if epv_cv_mean and stepwise_cv_mean:
            cv_diff = stepwise_cv_mean - epv_cv_mean
            if abs(cv_diff) > 0.02:
                if cv_diff > 0:
                    recommendation_score += 3
                    reasons.append(f"✓ Stepwise has better CV C-index (+{cv_diff:.4f})")
                else:
                    recommendation_score -= 3
                    reasons.append(f"✓ EPV-cap has better CV C-index (+{abs(cv_diff):.4f})")

        # Score based on optimism
        if epv_optimism and stepwise_optimism_final:
            if abs(stepwise_optimism_final) < abs(epv_optimism) - 0.01:
                recommendation_score += 2
                reasons.append(f"✓ Stepwise has lower optimism ({abs(stepwise_optimism_final):.4f} vs {abs(epv_optimism):.4f})")
            elif abs(epv_optimism) < abs(stepwise_optimism_final) - 0.01:
                recommendation_score -= 2
                reasons.append(f"✓ EPV-cap has lower optimism ({abs(epv_optimism):.4f} vs {abs(stepwise_optimism_final):.4f})")

        # Score based on AIC
        if epv_AIC and stepwise_AIC_final:
            aic_diff = epv_AIC - stepwise_AIC_final
            if abs(aic_diff) > 2:
                if aic_diff > 0:
                    recommendation_score += 1
                    reasons.append(f"✓ Stepwise has lower AIC (-{aic_diff:.2f})")
                else:
                    recommendation_score -= 1
                    reasons.append(f"✓ EPV-cap has lower AIC (-{abs(aic_diff):.2f})")

        # Score based on parsimony
        var_diff = epv_n_vars - stepwise_n_vars
        if var_diff > 1:
            recommendation_score += 1
            reasons.append(f"✓ Stepwise is more parsimonious ({stepwise_n_vars} vs {epv_n_vars} variables)")
        elif var_diff < -1:
            recommendation_score -= 1
            reasons.append(f"✓ EPV-cap is more parsimonious ({epv_n_vars} vs {stepwise_n_vars} variables)")

        print("\n🎯 Recommendation based on statistical criteria:")
        if recommendation_score > 2:
            print("   ➡️  STEPWISE MODEL appears superior")
        elif recommendation_score < -2:
            print("   ➡️  EPV-CAP MODEL appears superior")
        else:
            print("   ➡️  Models perform SIMILARLY - consider domain knowledge")

        if reasons:
            print("\n   Key factors:")
            for reason in reasons:
                print(f"      {reason}")

        print("\n   Note: Consider domain expertise and biological plausibility")
        print("         when making final selection!")

        # ============================================================================
        # STEP 5: USER SELECTION (AFTER SEEING VISUALIZATIONS)
        # ============================================================================

        print("\n" + "=" * 120)
        print("🏆 CHAMPION MODEL SELECTION")
        print("-" * 120)

        print("\n📋 Model Summary:")
        print(f"\n   1. EPV-CAP MODEL")
        print(f"      • Variables: {epv_n_vars}")
        print(f"      • Training C-index: {epv_train_c_index:.4f}")
        if epv_cv_mean:
            print(f"      • CV C-index: {epv_cv_mean:.4f} ± {epv_cv_std:.4f}")
        if epv_AIC:
            print(f"      • AIC: {epv_AIC:.2f}")
        print(f"      • Variables: {', '.join(selected_biomarkers_for_multivariate[:3])}...")

        print(f"\n   2. STEPWISE MODEL")
        print(f"      • Variables: {stepwise_n_vars}")
        if stepwise_train_c_index_final:
            print(f"      • Training C-index: {stepwise_train_c_index_final:.4f}")
        if stepwise_cv_mean:
            print(f"      • CV C-index: {stepwise_cv_mean:.4f} ± {stepwise_cv_std:.4f}")
        if stepwise_AIC_final:
            print(f"      • AIC: {stepwise_AIC_final:.2f}")
        print(f"      • Variables: {', '.join(stepwise_results[:3])}...")

        print("\n🔹 Which model would you like to select as champion?")
        print("   1. EPV-Cap Model")
        print("   2. Stepwise Model")

        champion_choice = None
        while champion_choice not in ["1", "2"]:
            champion_choice = input("\n🔹 Enter choice (1 or 2): ").strip()
            if champion_choice not in ["1", "2"]:
                print("   ❌ Invalid choice. Please enter 1 or 2")

        # ============================================================================
        # STEP 6: SAVE CHAMPION MODEL
        # ============================================================================

        if champion_choice == "1":
            print("\n✅ EPV-Cap Model selected as CHAMPION")

            champion_model = {
                'type': 'EPV-cap',
                'biomarkers': selected_biomarkers_for_multivariate,
                'formula': ' + '.join([f'Q("{b}")' for b in selected_biomarkers_for_multivariate]),
                'n_variables': epv_n_vars,
                'train_c_index': epv_train_c_index,
                'cv_c_index': epv_cv_mean,
                'cv_std': epv_cv_std,
                'optimism': epv_optimism,
                'AIC': epv_AIC,
                'BIC': epv_BIC,
                'results': multivariate_results,
                'cv_results': cv_results if 'cv_results' in globals() else None
            }

        else:
            print("\n✅ Stepwise Model selected as CHAMPION")

            champion_model = {
                'type': 'Stepwise',
                'biomarkers': stepwise_results,
                'formula': stepwise_formula,
                'n_variables': stepwise_n_vars,
                'train_c_index': stepwise_train_c_index_final,
                'cv_c_index': stepwise_cv_mean,
                'cv_std': stepwise_cv_std,
                'optimism': stepwise_optimism_final,
                'AIC': stepwise_AIC_final,
                'BIC': stepwise_BIC_final,
                'model_object': cph_stepwise if stepwise_cv_results else None,
                'cv_results': stepwise_cv_results
            }

            # Store stepwise results as multivariate_results for next code
            multivariate_results = stepwise_results_dict
            cv_results = stepwise_cv_results
            globals()['multivariate_results'] = multivariate_results
            globals()['cv_results'] = cv_results

        # Store in globals
        globals()['champion_model'] = champion_model
        globals()['champion_biomarkers'] = champion_model['biomarkers']
        globals()['champion_formula'] = champion_model['formula']
        globals()['champion_type'] = champion_model['type']

        print("\n📦 Champion model saved to global variables:")
        print("   • champion_model (full model details)")
        print("   • champion_biomarkers (list of biomarker names)")
        print("   • champion_formula (Cox formula string)")
        print("   • champion_type (model type: 'EPV-cap' or 'Stepwise')")

        print("\n" + "=" * 120)
        print("🎯 CHAMPION MODEL DETAILS")
        print("-" * 120)
        print(f"\n   Model Type: {champion_model['type']}")
        print(f"   Number of Variables: {champion_model['n_variables']}")
        print(f"   Training C-index: {champion_model['train_c_index']:.4f}")
        if champion_model['cv_c_index']:
            print(f"   CV C-index: {champion_model['cv_c_index']:.4f} ± {champion_model['cv_std']:.4f}")
        if champion_model['optimism']:
            print(f"   Optimism: {champion_model['optimism']:.4f}")
        if champion_model['AIC']:
            print(f"   AIC: {champion_model['AIC']:.2f}")
        if champion_model['BIC']:
            print(f"   BIC: {champion_model['BIC']:.2f}")

        print(f"\n   Selected Biomarkers:")
        for i, biomarker in enumerate(champion_model['biomarkers'], 1):
            print(f"      {i}. {biomarker}")

        # ============================================================================
        # STEP 7: SAVE COMPARISON RESULTS
        # ============================================================================

        if 'RESULTS_FOLDER' in globals():
            print("\n" + "=" * 120)
            print("💾 SAVING MODEL COMPARISON RESULTS")
            print("-" * 120)

            try:
                comparison_file = os.path.join(RESULTS_FOLDER, f"Model_Comparison_{timestamp}.xlsx")

                with pd.ExcelWriter(comparison_file, engine='openpyxl') as writer:
                    # Comparison summary
                    comparison_data = {
                        'Metric': [
                            'Number of Variables',
                            'Training C-index',
                            'CV Mean C-index',
                            'CV Std C-index',
                            'Optimism (Bias)',
                            'AIC',
                            'BIC'
                        ],
                        'EPV-Cap Model': [
                            epv_n_vars,
                            epv_train_c_index,
                            epv_cv_mean if epv_cv_mean else None,
                            epv_cv_std if epv_cv_std else None,
                            epv_optimism if epv_optimism else None,
                            epv_AIC if epv_AIC else None,
                            epv_BIC if epv_BIC else None
                        ],
                        'Stepwise Model': [
                            stepwise_n_vars,
                            stepwise_train_c_index_final if stepwise_train_c_index_final else None,
                            stepwise_cv_mean if stepwise_cv_mean else None,
                            stepwise_cv_std if stepwise_cv_std else None,
                            stepwise_optimism_final if stepwise_optimism_final else None,
                            stepwise_AIC_final if stepwise_AIC_final else None,
                            stepwise_BIC_final if stepwise_BIC_final else None
                        ]
                    }

                    comparison_df = pd.DataFrame(comparison_data)
                    comparison_df.to_excel(writer, sheet_name='Model_Comparison', index=False)

                    # EPV model variables
                    epv_vars_df = pd.DataFrame({
                        'Rank': range(1, len(selected_biomarkers_for_multivariate) + 1),
                        'Biomarker': selected_biomarkers_for_multivariate
                    })
                    epv_vars_df.to_excel(writer, sheet_name='EPV_Variables', index=False)

                    # Stepwise model variables
                    stepwise_vars_df = pd.DataFrame({
                        'Rank': range(1, len(stepwise_results) + 1),
                        'Biomarker': stepwise_results
                    })
                    stepwise_vars_df.to_excel(writer, sheet_name='Stepwise_Variables', index=False)

                    # Champion model info
                    champion_info = pd.DataFrame([{
                        'Champion_Model': champion_model['type'],
                        'N_Variables': champion_model['n_variables'],
                        'Training_C_Index': champion_model['train_c_index'],
                        'CV_C_Index': champion_model['cv_c_index'],
                        'CV_Std': champion_model['cv_std'],
                        'Optimism': champion_model['optimism'],
                        'AIC': champion_model['AIC'],
                        'BIC': champion_model['BIC']
                    }])
                    champion_info.to_excel(writer, sheet_name='Champion_Model', index=False)

                    # Champion biomarkers
                    champion_vars_df = pd.DataFrame({
                        'Rank': range(1, len(champion_model['biomarkers']) + 1),
                        'Biomarker': champion_model['biomarkers']
                    })
                    champion_vars_df.to_excel(writer, sheet_name='Champion_Biomarkers', index=False)

                print(f"\n✅ Model comparison results saved to:")
                print(f"   {comparison_file}")

            except Exception as e:
                print(f"\n⚠️ Error saving comparison results: {str(e)}")

        print("\n" + "=" * 120)
        print("✅ MODEL COMPARISON AND SELECTION COMPLETED")
        print("=" * 120)
        print(f"\n🏆 Champion: {champion_model['type']} Model ({champion_model['n_variables']} variables)")
        print(f"   📂 Comparison plots: {comparison_folder}")
        print(f"   Use 'champion_model', 'champion_biomarkers', 'champion_formula' in subsequent cells")
        print("=" * 120)

# 3. Enhanced Multivariate Visualizations with CV Stability Analysis

## Overview
Generates publication-quality visualizations for the multivariate Cox model with enhanced cross-validation stability metrics.

### What this cell does:
1. **Forest Plot**: Multivariate hazard ratios with 95% confidence intervals
2. **Coefficient Plot**: Standardized coefficient magnitudes
3. **UV vs MV Comparison**: Changes in hazard ratios
4. **CV Stability Analysis**: C-index stability across folds
5. **Correlation Matrix**: Inter-variable correlations

---

## ⚙️ TUNABLE PARAMETERS

| Parameter | Default | Description | Location |
|-----------|---------|-------------|----------|
| `CV_FOLDS` | `5` | Number of cross-validation folds | CV section
| `CV_REPETITIONS` | `10` |
| `BASE_SEED` | `42` |
| `figsize` | `(14, 10)` | Figure dimensions | Plot functions |

---

### Dependencies:
- Requires Cells 0-1 to be executed first


In [ ]:
# ============================================================================
# COMPLETE MULTIVARIATE MODEL VISUALIZATIONS + ENHANCED CV STABILITY (FINAL)
# Copy-ready standalone cell - Run after multivariate Cox regression
# Includes ALL original plots + enhanced CV stability analysis
# FINAL VERSION: Clean, professional summary box + LEGEND REPOSITIONED
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from lifelines import CoxPHFitter
import os

# Check if multivariate results exist
if 'multivariate_results' not in locals() or multivariate_results is None or len(multivariate_results.get('biomarkers', [])) == 0:
    print("\n" + "=" * 120)
    print("⚠️ NO MULTIVARIATE RESULTS AVAILABLE")
    print("=" * 120)
    print("\nMultivariate visualizations require completed multivariate analysis.")
    print("Please run the multivariate Cox regression analysis first.")
else:
    print("\n" + "=" * 120)
    print("📊 COMPLETE MULTIVARIATE MODEL VISUALIZATIONS + ENHANCED CV (FINAL)")
    print("=" * 120)

    # Create visualizations folder
    mv_plots_folder = os.path.join(RESULTS_FOLDER, "Multivariate_Visualizations")
    os.makedirs(mv_plots_folder, exist_ok=True)

    # Extract multivariate results
    mv_biomarkers = multivariate_results['biomarkers']
    mv_df = pd.DataFrame(mv_biomarkers)
    mv_df = mv_df.sort_values('p_value')

    # ========================================================================
    # STEP 1: PERFORM ENHANCED CROSS-VALIDATION (if not already done)
    # ========================================================================

    if 'cv_results' not in locals() or cv_results is None or 'all_cv_scores' not in cv_results:
        print("\n🔄 Performing Enhanced Cross-Validation...")

        try:
            # CV Parameters
            N_FOLDS = 5
            N_REPETITIONS = 10
            BASE_SEED = 42

            # Get biomarkers and prepare data
            selected_biomarkers_for_multivariate = [b['name'] for b in multivariate_results['biomarkers']]
            multivariate_data = df[[survival_time_col, 'event'] + selected_biomarkers_for_multivariate].dropna()

            print(f"   Dataset: {len(multivariate_data)} samples, {multivariate_data['event'].sum()} events")

            # Storage for CV results
            all_cv_scores = []
            scores_per_repetition = []
            repetition_stats = []

            # Build formula
            formula = ' + '.join([f'Q("{b}")' for b in selected_biomarkers_for_multivariate])

            # Perform repeated stratified CV
            for rep in range(N_REPETITIONS):
                seed = BASE_SEED + rep
                np.random.seed(seed)

                skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

                fold_scores = []
                for fold_idx, (train_idx, test_idx) in enumerate(skf.split(multivariate_data, multivariate_data['event'])):
                    try:
                        train_data = multivariate_data.iloc[train_idx]
                        test_data = multivariate_data.iloc[test_idx]

                        if len(train_data) < 20 or len(test_data) < 5:
                            continue
                        if train_data['event'].sum() < 5 or test_data['event'].sum() < 2:
                            continue

                        cph_cv = CoxPHFitter(penalizer=PENALIZER)
                        cph_cv.fit(train_data, duration_col=survival_time_col,
                                  event_col='event', formula=formula, robust=ROBUST_SE)

                        c_index = cph_cv.score(test_data, scoring_method="concordance_index")
                        fold_scores.append(c_index)
                        all_cv_scores.append(c_index)

                    except Exception:
                        continue

                if len(fold_scores) > 0:
                    scores_per_repetition.append(fold_scores)
                    rep_stats = {
                        'repetition': rep + 1,
                        'n_folds': len(fold_scores),
                        'mean': np.mean(fold_scores),
                        'std': np.std(fold_scores),
                        'min': np.min(fold_scores),
                        'max': np.max(fold_scores)
                    }
                    repetition_stats.append(rep_stats)
                    print(f"   Rep {rep+1:2d}: C-index = {rep_stats['mean']:.4f} ± {rep_stats['std']:.4f}")

            if len(all_cv_scores) > 0:
                overall_mean = np.mean(all_cv_scores)
                overall_std = np.std(all_cv_scores)
                train_c_index = multivariate_results['model_concordance']

                cv_results = {
                    'mean_c_index': overall_mean,
                    'std_c_index': overall_std,
                    'median_c_index': np.median(all_cv_scores),
                    'min_c_index': np.min(all_cv_scores),
                    'max_c_index': np.max(all_cv_scores),
                    'n_successful_runs': len(all_cv_scores),
                    'n_folds': N_FOLDS,
                    'n_repetitions': N_REPETITIONS,
                    'all_cv_scores': all_cv_scores,
                    'scores_per_repetition': scores_per_repetition,
                    'repetition_stats': repetition_stats,
                    'coefficient_of_variation': (overall_std / overall_mean) * 100,
                    'optimism': train_c_index - overall_mean,
                    'train_c_index': train_c_index
                }

                print(f"\n   ✅ CV Complete: {overall_mean:.4f} ± {overall_std:.4f}")
            else:
                cv_results = None

        except Exception as e:
            print(f"   ⚠️ CV failed: {str(e)}")
            cv_results = None

    # ========================================================================
    # VISUALIZATION 1: FOREST PLOT - MULTIVARIATE MODEL
    # ========================================================================

    print("\n📈 1. Generating Multivariate Forest Plot...")

    fig, ax = plt.subplots(figsize=(12, max(8, len(mv_df) * 0.8)))

    y_positions = np.arange(len(mv_df))

    for i, (idx, row) in enumerate(mv_df.iterrows()):
        color = 'darkred' if row['p_value'] < 0.001 else 'red' if row['p_value'] < 0.01 else 'orange' if row['p_value'] < 0.05 else 'gray'

        ax.plot([row['hr_lower'], row['hr_upper']], [i, i], color=color, linewidth=2, alpha=0.7)
        ax.scatter(row['hr'], i, color=color, s=100, zorder=5)

        ci_text = f"({row['hr_lower']:.2f}-{row['hr_upper']:.2f})"
        p_text = f"p={row['p_value']:.3f}" if row['p_value'] >= 0.001 else "p<0.001"
        ax.text(row['hr_upper'] + 0.05, i, f"HR={row['hr']:.2f} {ci_text}, {p_text}",
                va='center', fontsize=9, alpha=0.8)

    ax.axvline(x=1, color='black', linestyle='--', alpha=0.5, label='HR=1 (No effect)')

    ax.set_yticks(y_positions)
    ax.set_yticklabels([name[:40] if len(name) > 40 else name for name in mv_df['name']], fontsize=11)
    ax.set_xlabel('Hazard Ratio (95% CI)', fontsize=12, fontweight='bold')
    ax.set_title(f'MULTIVARIATE COX REGRESSION - FOREST PLOT\n{survival_type} Analysis (n={multivariate_results["n_samples"]}, events={multivariate_results["n_events"]})',
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    ax.set_xlim(left=0)

    legend_elements = [
        Patch(facecolor='darkred', alpha=0.7, label='p < 0.001'),
        Patch(facecolor='red', alpha=0.7, label='p < 0.01'),
        Patch(facecolor='orange', alpha=0.7, label='p < 0.05'),
        Patch(facecolor='gray', alpha=0.7, label='p ≥ 0.05')
    ]
    ax.legend(handles=legend_elements, loc='lower left', title='Significance')

    plt.tight_layout()
    forest_path = os.path.join(mv_plots_folder, "Multivariate_Forest_Plot.png")
    plt.savefig(forest_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"   ✓ Saved: Multivariate_Forest_Plot.png")

    # ========================================================================
    # VISUALIZATION 2: COEFFICIENT PLOT
    # ========================================================================

    print("\n📈 2. Generating Coefficient Plot...")

    fig, ax = plt.subplots(figsize=(10, 8))

    mv_df_sorted = mv_df.sort_values('coef')

    colors = ['green' if x > 0 else 'red' for x in mv_df_sorted['coef']]
    bars = ax.barh(range(len(mv_df_sorted)), mv_df_sorted['coef'], color=colors, alpha=0.7, edgecolor='black')

    ax.errorbar(mv_df_sorted['coef'], range(len(mv_df_sorted)),
                xerr=mv_df_sorted['se'] * 1.96, fmt='none', color='black', alpha=0.5, capsize=3)

    ax.set_yticks(range(len(mv_df_sorted)))
    ax.set_yticklabels([name[:35] if len(name) > 35 else name for name in mv_df_sorted['name']], fontsize=10)
    ax.set_xlabel('Cox Coefficient (± 95% CI)', fontsize=12, fontweight='bold')
    ax.set_title('MULTIVARIATE MODEL - COEFFICIENT PLOT', fontsize=14, fontweight='bold')
    ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax.grid(True, alpha=0.3, axis='x')

    for i, (idx, row) in enumerate(mv_df_sorted.iterrows()):
        sig_marker = '***' if row['p_value'] < 0.001 else '**' if row['p_value'] < 0.01 else '*' if row['p_value'] < 0.05 else ''
        if sig_marker:
            ax.text(row['coef'], i, f'  {sig_marker}', va='center', fontsize=12, fontweight='bold')

    plt.tight_layout()
    coef_path = os.path.join(mv_plots_folder, "Multivariate_Coefficient_Plot.png")
    plt.savefig(coef_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"   ✓ Saved: Multivariate_Coefficient_Plot.png")

    # ========================================================================
    # VISUALIZATION 3: UNIVARIATE VS MULTIVARIATE COMPARISON
    # ========================================================================

    if 'cox_df' in locals() and len(cox_df) > 0:
        print("\n📈 3. Generating Univariate vs Multivariate Comparison...")

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

        comparison_data = []
        for biomarker_result in mv_biomarkers:
            biomarker_name = biomarker_result['name']
            uv_row = cox_df[cox_df['biomarker'] == biomarker_name]

            if len(uv_row) > 0:
                comparison_data.append({
                    'biomarker': biomarker_name[:25] if len(biomarker_name) > 25 else biomarker_name,
                    'uv_hr': float(uv_row['hr'].values[0]),
                    'mv_hr': biomarker_result['hr'],
                    'uv_p': float(uv_row['p_value'].values[0]),
                    'mv_p': biomarker_result['p_value']
                })

        if comparison_data:
            comp_df = pd.DataFrame(comparison_data)
            comp_df = comp_df.sort_values('mv_p')

            x_pos = np.arange(len(comp_df))
            width = 0.35

            # Plot 1: HR Comparison
            bars1 = ax1.bar(x_pos - width/2, comp_df['uv_hr'], width, label='Univariate', alpha=0.7, color='skyblue', edgecolor='black')
            bars2 = ax1.bar(x_pos + width/2, comp_df['mv_hr'], width, label='Multivariate', alpha=0.7, color='salmon', edgecolor='black')

            ax1.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
            ax1.set_xlabel('Biomarker', fontsize=11, fontweight='bold')
            ax1.set_ylabel('Hazard Ratio', fontsize=11, fontweight='bold')
            ax1.set_title('HR: UNIVARIATE vs MULTIVARIATE', fontsize=12, fontweight='bold')
            ax1.set_xticks(x_pos)
            ax1.set_xticklabels(comp_df['biomarker'], rotation=45, ha='right', fontsize=9)
            ax1.legend(loc='upper left')
            ax1.grid(True, alpha=0.3, axis='y')

            for bar in bars1:
                height = bar.get_height()
                ax1.text(bar.get_x() + bar.get_width()/2., height + 0.05,
                        f'{height:.2f}', ha='center', va='bottom', fontsize=8)
            for bar in bars2:
                height = bar.get_height()
                ax1.text(bar.get_x() + bar.get_width()/2., height + 0.05,
                        f'{height:.2f}', ha='center', va='bottom', fontsize=8)

            # Plot 2: P-value Comparison
            bars3 = ax2.bar(x_pos - width/2, -np.log10(comp_df['uv_p']), width, label='Univariate', alpha=0.7, color='skyblue', edgecolor='black')
            bars4 = ax2.bar(x_pos + width/2, -np.log10(comp_df['mv_p']), width, label='Multivariate', alpha=0.7, color='salmon', edgecolor='black')

            ax2.axhline(y=-np.log10(0.05), color='red', linestyle='--', alpha=0.5, label='p=0.05')
            ax2.axhline(y=-np.log10(0.01), color='darkred', linestyle='--', alpha=0.5, label='p=0.01')
            ax2.set_xlabel('Biomarker', fontsize=11, fontweight='bold')
            ax2.set_ylabel('-log10(p-value)', fontsize=11, fontweight='bold')
            ax2.set_title('SIGNIFICANCE: UNIVARIATE vs MULTIVARIATE', fontsize=12, fontweight='bold')
            ax2.set_xticks(x_pos)
            ax2.set_xticklabels(comp_df['biomarker'], rotation=45, ha='right', fontsize=9)
            ax2.legend(loc='upper right')
            ax2.grid(True, alpha=0.3, axis='y')

            plt.suptitle('UNIVARIATE vs MULTIVARIATE MODEL COMPARISON', fontsize=14, fontweight='bold', y=1.02)
            plt.tight_layout()
            comparison_path = os.path.join(mv_plots_folder, "Univariate_vs_Multivariate_Comparison.png")
            plt.savefig(comparison_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"   ✓ Saved: Univariate_vs_Multivariate_Comparison.png")

    # ========================================================================
    # VISUALIZATION 4: ENHANCED C-INDEX STABILITY (FINAL - CLEAN BOX + LEGEND FIX)
    # ========================================================================

    if cv_results is not None and 'all_cv_scores' in cv_results:
        print("\n📈 4. Generating Enhanced C-Index Stability Analysis (FINAL)...")

        all_scores = cv_results['all_cv_scores']
        scores_per_rep = cv_results['scores_per_repetition']
        rep_stats = cv_results['repetition_stats']

        n_successful_reps = len(scores_per_rep)
        n_folds = cv_results['n_folds']
        overall_mean = cv_results['mean_c_index']
        overall_std = cv_results['std_c_index']
        overall_cv_pct = cv_results['coefficient_of_variation']
        train_c_index = cv_results['train_c_index']
        optimism = cv_results['optimism']

        mean_c_per_repeat = [stats['mean'] for stats in rep_stats]
        std_c_per_repeat = [stats['std'] for stats in rep_stats]
        min_c_per_repeat = [stats['min'] for stats in rep_stats]
        max_c_per_repeat = [stats['max'] for stats in rep_stats]

        fig = plt.figure(figsize=(20, 14))  # Increased width to accommodate legend
        gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.35)

        repeat_nums = np.arange(1, n_successful_reps + 1)

        # Subplot 1: Main Trend (with repositioned legend)
        ax1 = fig.add_subplot(gs[0, :])

        ax1.plot(repeat_nums, mean_c_per_repeat, 'o-', color='steelblue',
                linewidth=3, markersize=10, label='Mean C-index per repeat',
                markerfacecolor='white', markeredgewidth=2, markeredgecolor='steelblue')

        ax1.fill_between(repeat_nums,
                        np.array(mean_c_per_repeat) - np.array(std_c_per_repeat),
                        np.array(mean_c_per_repeat) + np.array(std_c_per_repeat),
                        alpha=0.3, color='steelblue', label='±1 SD within repeat')

        ax1.errorbar(repeat_nums, mean_c_per_repeat,
                    yerr=[np.array(mean_c_per_repeat) - np.array(min_c_per_repeat),
                          np.array(max_c_per_repeat) - np.array(mean_c_per_repeat)],
                    fmt='none', color='gray', alpha=0.4, capsize=4, capthick=1.5,
                    label='Min-Max range')

        ax1.axhline(y=overall_mean, color='red', linestyle='--', linewidth=2.5,
                   label=f'Overall Mean: {overall_mean:.4f}')

        se_of_means = np.std(mean_c_per_repeat) / np.sqrt(n_successful_reps)
        ax1.fill_between(repeat_nums, overall_mean - 2*se_of_means, overall_mean + 2*se_of_means,
                        alpha=0.15, color='red', label=f'±2 SE band ({2*se_of_means:.4f})')

        ax1.axhline(y=train_c_index, color='green', linestyle=':', linewidth=2.5,
                   alpha=0.7, label=f'Training C-index: {train_c_index:.4f}')
        ax1.axhline(y=0.5, color='gray', linestyle=':', linewidth=2, alpha=0.5, label='Random (0.5)')

        ax1.set_xlabel('Repetition Number', fontsize=13, fontweight='bold')
        ax1.set_ylabel('C-index', fontsize=13, fontweight='bold')
        ax1.set_title(f'C-INDEX STABILITY ACROSS {n_successful_reps} REPETITIONS ({n_folds}-Fold Stratified CV)\n' +
                     f'Overall: {overall_mean:.4f} ± {overall_std:.4f} (CV% = {overall_cv_pct:.2f}%, Optimism = {optimism:+.4f})',
                     fontsize=14, fontweight='bold')
        # LEGEND REPOSITIONED TO THE RIGHT
        ax1.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=10, framealpha=0.9)
        ax1.grid(True, alpha=0.3, linestyle='--')
        ax1.set_ylim([max(0.45, min(min_c_per_repeat) - 0.03), min(1.0, max(max_c_per_repeat) + 0.03)])
        ax1.set_xticks(repeat_nums)

        # Subplot 2: Cumulative Mean
        ax2 = fig.add_subplot(gs[1, 0])
        cumulative_means = np.cumsum(mean_c_per_repeat) / repeat_nums
        ax2.plot(repeat_nums, cumulative_means, 'o-', color='purple', linewidth=2.5, markersize=8, label='Cumulative Mean')
        ax2.axhline(y=overall_mean, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Final: {overall_mean:.4f}')
        ax2.fill_between(repeat_nums, overall_mean * 0.99, overall_mean * 1.01, alpha=0.2, color='green', label='±1% band')
        ax2.set_xlabel('Number of Repetitions', fontsize=11, fontweight='bold')
        ax2.set_ylabel('Cumulative Mean C-index', fontsize=11, fontweight='bold')
        ax2.set_title('CONVERGENCE ANALYSIS', fontsize=11, fontweight='bold')
        ax2.legend(fontsize=9)
        ax2.grid(True, alpha=0.3)
        ax2.set_xticks(repeat_nums)

        # Subplot 3: Distribution
        ax3 = fig.add_subplot(gs[1, 1])
        n_bins = min(25, len(all_scores) // 2)
        ax3.hist(all_scores, bins=n_bins, density=True, color='steelblue', alpha=0.6, edgecolor='black', label=f'n={len(all_scores)}')

        try:
            from scipy import stats as sp_stats
            kde = sp_stats.gaussian_kde(all_scores)
            x_kde = np.linspace(min(all_scores), max(all_scores), 200)
            ax3.plot(x_kde, kde(x_kde), 'r-', linewidth=2.5, label='KDE')
        except:
            pass

        ax3.axvline(x=overall_mean, color='red', linestyle='--', linewidth=2, label=f'Mean: {overall_mean:.4f}')
        ax3.axvline(x=np.median(all_scores), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(all_scores):.4f}')
        ax3.set_xlabel('C-index', fontsize=11, fontweight='bold')
        ax3.set_ylabel('Density', fontsize=11, fontweight='bold')
        ax3.set_title('DISTRIBUTION OF ALL FOLD SCORES', fontsize=11, fontweight='bold')
        ax3.legend(fontsize=8, loc='best')
        ax3.grid(True, alpha=0.3, axis='y')

        # Subplot 4: Box Plots
        ax4 = fig.add_subplot(gs[1, 2])
        bp = ax4.boxplot(scores_per_rep, patch_artist=True, notch=True, showmeans=True,
                        meanprops=dict(marker='D', markerfacecolor='red', markersize=6),
                        boxprops=dict(facecolor='lightblue', alpha=0.7),
                        medianprops=dict(color='darkblue', linewidth=2))
        ax4.axhline(y=overall_mean, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Overall: {overall_mean:.4f}')
        ax4.set_xlabel('Repetition', fontsize=11, fontweight='bold')
        ax4.set_ylabel('C-index', fontsize=11, fontweight='bold')
        ax4.set_title('VARIABILITY PER REPETITION', fontsize=11, fontweight='bold')
        ax4.legend(fontsize=8)
        ax4.grid(True, alpha=0.3, axis='y')
        ax4.set_xticklabels(repeat_nums, fontsize=9)

        # Subplot 5: Variance Decomposition
        ax5 = fig.add_subplot(gs[2, 0])
        between_var = np.var(mean_c_per_repeat, ddof=1)
        within_var = np.mean([np.var(scores, ddof=1) for scores in scores_per_rep])
        total_var = np.var(all_scores, ddof=1)

        bars = ax5.bar(['Between\nRepeats', 'Within\nRepeats'], [between_var, within_var],
                      color=['coral', 'lightblue'], alpha=0.7, edgecolor='black', linewidth=2)

        for bar, var in zip(bars, [between_var, within_var]):
            pct = (var / total_var) * 100
            height = bar.get_height()
            ax5.text(bar.get_x() + bar.get_width()/2., height,
                    f'{var:.4f}\n({pct:.1f}%)',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

        ax5.set_ylabel('Variance', fontsize=11, fontweight='bold')
        ax5.set_title(f'VARIANCE DECOMPOSITION\n(Total: {total_var:.4f})',
                     fontsize=11, fontweight='bold', pad=15)
        ax5.grid(True, alpha=0.3, axis='y')
        y_max = max(between_var, within_var)
        ax5.set_ylim([0, y_max * 1.35])

       # Subplot 6: Summary Table (REDESIGNED - INCREASED SPACING)
        ax6 = fig.add_subplot(gs[2, 1:])
        ax6.axis('off')

        icc = between_var / (between_var + within_var)
        ci_95_lower, ci_95_upper = np.percentile(all_scores, [2.5, 97.5])

        # Draw title box
        title_box = Rectangle((0.05, 0.88), 0.9, 0.10,
                              transform=ax6.transAxes,
                              facecolor='#2E5090',
                              edgecolor='black',
                              linewidth=2)
        ax6.add_patch(title_box)
        ax6.text(0.5, 0.93, 'REPEATED CROSS-VALIDATION SUMMARY',
                transform=ax6.transAxes,
                fontsize=12, fontweight='bold', color='white',
                ha='center', va='center')

        # Draw main content box
        content_box = Rectangle((0.05, 0.05), 0.9, 0.81,
                               transform=ax6.transAxes,
                               facecolor='white',
                               edgecolor='black',
                               linewidth=2)
        ax6.add_patch(content_box)

        # Draw vertical divider between columns
        ax6.plot([0.5, 0.5], [0.08, 0.83], transform=ax6.transAxes,
                color='lightgray', linestyle='-', linewidth=1.5)

        # LEFT COLUMN - Configuration & Performance
        y_start = 0.76

        # Configuration section
        ax6.text(0.10, y_start, 'CONFIGURATION', transform=ax6.transAxes,
                fontsize=10, fontweight='bold', color='#2E5090')
        y_start -= 0.08
        ax6.text(0.12, y_start, f'Strategy:', transform=ax6.transAxes,
                fontsize=9, fontweight='bold')
        ax6.text(0.32, y_start, f'{n_folds}-Fold Stratified CV', transform=ax6.transAxes,
                fontsize=9)
        y_start -= 0.05
        ax6.text(0.12, y_start, f'Repetitions:', transform=ax6.transAxes,
                fontsize=9, fontweight='bold')
        ax6.text(0.32, y_start, f'{n_successful_reps}', transform=ax6.transAxes,
                fontsize=9)
        y_start -= 0.05
        ax6.text(0.12, y_start, f'Total Fits:', transform=ax6.transAxes,
                fontsize=9, fontweight='bold')
        ax6.text(0.32, y_start, f'{len(all_scores)}', transform=ax6.transAxes,
                fontsize=9)

        # Performance section
        y_start -= 0.10
        ax6.text(0.10, y_start, 'PERFORMANCE', transform=ax6.transAxes,
                fontsize=10, fontweight='bold', color='#2E5090')
        y_start -= 0.08
        ax6.text(0.12, y_start, f'Mean C-index:', transform=ax6.transAxes,
                fontsize=9, fontweight='bold')
        ax6.text(0.32, y_start, f'{overall_mean:.4f} ± {overall_std:.4f}', transform=ax6.transAxes,
                fontsize=9)
        y_start -= 0.05
        ax6.text(0.12, y_start, f'Median:', transform=ax6.transAxes,
                fontsize=9, fontweight='bold')
        ax6.text(0.32, y_start, f'{np.median(all_scores):.4f}', transform=ax6.transAxes,
                fontsize=9)
        y_start -= 0.05
        ax6.text(0.12, y_start, f'95% CI:', transform=ax6.transAxes,
                fontsize=9, fontweight='bold')
        ax6.text(0.32, y_start, f'[{ci_95_lower:.4f}, {ci_95_upper:.4f}]', transform=ax6.transAxes,
                fontsize=9)

        # RIGHT COLUMN - Stability & Comparison
        y_start = 0.76

        # Stability section
        ax6.text(0.55, y_start, 'STABILITY METRICS', transform=ax6.transAxes,
                fontsize=10, fontweight='bold', color='#2E5090')
        y_start -= 0.08
        ax6.text(0.57, y_start, f'CV%:', transform=ax6.transAxes,
                fontsize=9, fontweight='bold')
        ax6.text(0.73, y_start, f'{overall_cv_pct:.2f}%', transform=ax6.transAxes,
                fontsize=9)
        y_start -= 0.05
        ax6.text(0.57, y_start, f'ICC:', transform=ax6.transAxes,
                fontsize=9, fontweight='bold')
        ax6.text(0.73, y_start, f'{icc:.3f}', transform=ax6.transAxes,
                fontsize=9)
        y_start -= 0.05
        ax6.text(0.57, y_start, f'Between-Rep Var:', transform=ax6.transAxes,
                fontsize=9, fontweight='bold')
        ax6.text(0.80, y_start, f'{between_var:.4f}', transform=ax6.transAxes,
                fontsize=9)
        y_start -= 0.05
        ax6.text(0.57, y_start, f'Within-Rep Var:', transform=ax6.transAxes,
                fontsize=9, fontweight='bold')
        ax6.text(0.80, y_start, f'{within_var:.4f}', transform=ax6.transAxes,
                fontsize=9)

        # Model comparison section
        y_start -= 0.10
        ax6.text(0.55, y_start, 'MODEL COMPARISON', transform=ax6.transAxes,
                fontsize=10, fontweight='bold', color='#2E5090')
        y_start -= 0.08
        ax6.text(0.57, y_start, f'Training C-index:', transform=ax6.transAxes,
                fontsize=9, fontweight='bold')
        ax6.text(0.80, y_start, f'{train_c_index:.4f}', transform=ax6.transAxes,
                fontsize=9)
        y_start -= 0.05
        ax6.text(0.57, y_start, f'CV C-index:', transform=ax6.transAxes,
                fontsize=9, fontweight='bold')
        ax6.text(0.80, y_start, f'{overall_mean:.4f}', transform=ax6.transAxes,
                fontsize=9)
        y_start -= 0.05
        ax6.text(0.57, y_start, f'Optimism:', transform=ax6.transAxes,
                fontsize=9, fontweight='bold')
        ax6.text(0.80, y_start, f'{optimism:+.4f}', transform=ax6.transAxes,
                fontsize=9)

        plt.suptitle('MULTIVARIATE MODEL: C-INDEX STABILITY ANALYSIS',
                    fontsize=16, fontweight='bold', y=0.995)

        stability_path = os.path.join(mv_plots_folder, "CV_Cindex_Stability_Complete_FINAL.png")
        plt.savefig(stability_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"   ✓ Saved: CV_Cindex_Stability_Complete_FINAL.png")

    # ========================================================================
    # VISUALIZATION 5: MODEL PERFORMANCE METRICS
    # ========================================================================

    print("\n📈 5. Generating Model Performance Metrics...")

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    # Subplot 1: C-index comparison
    if 'cox_df' in locals() and len(cox_df) > 0:
        ax = axes[0, 0]

        c_indices = []
        labels = []
        for biomarker_result in mv_biomarkers:
            biomarker_name = biomarker_result['name']
            uv_row = cox_df[cox_df['biomarker'] == biomarker_name]
            if len(uv_row) > 0:
                c_indices.append(float(uv_row['concordance'].values[0]))
                labels.append(biomarker_name[:15] if len(biomarker_name) > 15 else biomarker_name)

        c_indices.append(multivariate_results['model_concordance'])
        labels.append('MULTIVARIATE\nMODEL')

        colors = ['lightblue'] * (len(c_indices) - 1) + ['darkgreen']
        bars = ax.bar(range(len(c_indices)), c_indices, color=colors, alpha=0.7, edgecolor='black')

        max_idx = np.argmax(c_indices)
        bars[max_idx].set_color('gold')
        bars[max_idx].set_edgecolor('black')
        bars[max_idx].set_linewidth(2)

        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
        ax.set_ylabel('C-index', fontsize=11, fontweight='bold')
        ax.set_title('MODEL DISCRIMINATION (Train C-INDEX)', fontsize=12, fontweight='bold')
        ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random (0.5)')
        ax.set_ylim([0.4, 1.0])
        ax.grid(True, alpha=0.3, axis='y')

        for bar, val in zip(bars, c_indices):
            ax.text(bar.get_x() + bar.get_width()/2., val + 0.01,
                   f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    else:
        axes[0, 0].text(0.5, 0.5, 'C-index Comparison\n(Univariate results not available)',
                        ha='center', va='center', fontsize=12)
        axes[0, 0].set_xticks([])
        axes[0, 0].set_yticks([])

    # Subplot 2: Variable Importance (|z-score|)
    ax = axes[0, 1]

    mv_df_z = mv_df.copy()
    mv_df_z['abs_z'] = np.abs(mv_df_z['z'])
    mv_df_z = mv_df_z.sort_values('abs_z', ascending=True)

    colors = ['green' if p < 0.05 else 'gray' for p in mv_df_z['p_value']]
    bars = ax.barh(range(len(mv_df_z)), mv_df_z['abs_z'], color=colors, alpha=0.7, edgecolor='black')

    ax.set_yticks(range(len(mv_df_z)))
    ax.set_yticklabels([name[:25] if len(name) > 25 else name for name in mv_df_z['name']], fontsize=10)
    ax.set_xlabel('|Z-score|', fontsize=11, fontweight='bold')
    ax.set_title('VARIABLE IMPORTANCE IN MULTIVARIATE MODEL', fontsize=12, fontweight='bold')
    ax.axvline(x=1.96, color='red', linestyle='--', alpha=0.5, label='p=0.05 threshold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='x')

    # Subplot 3: P-value distribution
    ax = axes[1, 0]

    p_values_mv = mv_df['p_value'].values
    significance_levels = ['p<0.001', 'p<0.01', 'p<0.05', 'p≥0.05']
    counts = [
        sum(p_values_mv < 0.001),
        sum((p_values_mv >= 0.001) & (p_values_mv < 0.01)),
        sum((p_values_mv >= 0.01) & (p_values_mv < 0.05)),
        sum(p_values_mv >= 0.05)
    ]
    colors_pie = ['darkred', 'red', 'orange', 'gray']

    wedges, texts, autotexts = ax.pie(counts, labels=significance_levels, colors=colors_pie,
                                       autopct=lambda pct: f'{pct:.0f}%\n(n={int(pct*len(p_values_mv)/100)})' if pct > 0 else '',
                                       startangle=90)
    ax.set_title('P-VALUE DISTRIBUTION IN MULTIVARIATE MODEL', fontsize=12, fontweight='bold')

    # Subplot 4: Model Statistics Summary
    ax = axes[1, 1]
    ax.axis('off')

    summary_lines = []
    summary_lines.append("    MODEL PERFORMANCE SUMMARY")
    summary_lines.append("    " + "="*40)
    summary_lines.append("")
    summary_lines.append(f"    Sample Size: {multivariate_results['n_samples']}")
    summary_lines.append(f"    Events: {multivariate_results['n_events']} ({multivariate_results['n_events']/multivariate_results['n_samples']*100:.1f}%)")
    summary_lines.append(f"    Variables: {len(mv_biomarkers)}")
    summary_lines.append("")
    summary_lines.append(f"    C-index: {multivariate_results['model_concordance']:.3f}")
    summary_lines.append(f"    Log-likelihood: {multivariate_results['log_likelihood']:.2f}")

    if multivariate_results.get('AIC') is not None:
        summary_lines.append(f"    AIC: {multivariate_results['AIC']:.2f}")
    if multivariate_results.get('BIC') is not None:
        summary_lines.append(f"    BIC: {multivariate_results['BIC']:.2f}")

    if cv_results is not None:
        summary_lines.append("")
        summary_lines.append("    Cross-Validation:")
        summary_lines.append(f"    Repeats: {cv_results.get('n_repetitions', 'N/A')}")
        summary_lines.append(f"    Folds/repeat: {cv_results.get('n_folds', 'N/A')}")
        summary_lines.append(f"    CV C-index: {cv_results['mean_c_index']:.3f} ± {cv_results['std_c_index']:.3f}")
        summary_lines.append(f"    CV%: {cv_results.get('coefficient_of_variation', 0):.2f}%")

    sig_predictors = [b['name'] for b in mv_biomarkers if b['p_value'] < 0.05]
    summary_lines.append("")
    summary_lines.append("")
    summary_lines.append(f"    Significant Predictors (p<0.05): {len(sig_predictors)}")

    for i, pred in enumerate(sig_predictors[:5], 1):
        pred_display = pred[:30] if len(pred) > 30 else pred
        summary_lines.append(f"       {i}. {pred_display}")

    if len(sig_predictors) > 5:
        summary_lines.append(f"       ... and {len(sig_predictors)-5} more")

    summary_text = "\n".join(summary_lines)

    ax.text(0.1, 0.9, summary_text, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.suptitle('MULTIVARIATE MODEL PERFORMANCE METRICS', fontsize=14, fontweight='bold')
    plt.tight_layout()
    performance_path = os.path.join(mv_plots_folder, "Model_Performance_Metrics.png")
    plt.savefig(performance_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"   ✓ Saved: Model_Performance_Metrics.png")

    # ========================================================================
    # VISUALIZATION 6: CORRELATION HEATMAP
    # ========================================================================

    print("\n📈 6. Generating Correlation Heatmap...")

    mv_biomarker_names = [b['name'] for b in mv_biomarkers]
    available_biomarkers = [b for b in mv_biomarker_names if b in df.columns]

    if len(available_biomarkers) >= 2:
        corr_matrix = df[available_biomarkers].corr()

        fig, ax = plt.subplots(figsize=(10, 8))

        labels = [name[:20] + '...' if len(name) > 20 else name for name in available_biomarkers]

        im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label('Correlation Coefficient', rotation=270, labelpad=20)

        ax.set_xticks(np.arange(len(labels)))
        ax.set_yticks(np.arange(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha='right')
        ax.set_yticklabels(labels)

        for i in range(len(labels)):
            for j in range(len(labels)):
                text = ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                             ha='center', va='center',
                             color='black' if abs(corr_matrix.iloc[i, j]) < 0.5 else 'white',
                             fontsize=9)

        ax.set_title('CORRELATION MATRIX - MULTIVARIATE MODEL VARIABLES', fontsize=12, fontweight='bold')
        plt.tight_layout()
        corr_path = os.path.join(mv_plots_folder, "Multivariate_Correlation_Matrix.png")
        plt.savefig(corr_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"   ✓ Saved: Multivariate_Correlation_Matrix.png")

    print("\n" + "=" * 120)
    print("✅ ALL MULTIVARIATE VISUALIZATIONS COMPLETED (FINAL VERSION)!")
    print(f"   All plots saved to: {mv_plots_folder}")
    print("=" * 120)

# 4. Quick Diagnostics Check (Optional)

## Overview
A lightweight diagnostic cell that quickly assesses the champion model's status without running full QC.

### What this cell does:
1. **Auto-Detection**: Finds the champion model from globals
2. **Quick Stats**: Reports key metrics (C-index, number of variables, sample size)
3. **PH Check**: Quick proportional hazards assumption test
4. **Status Report**: Summary of model health

### Use Case:
Run this cell for a fast status check before investing time in full diagnostics.


In [ ]:
"""
ALL-IN-ONE QUICK CHECK
======================
Automatically finds your champion model and runs quick diagnostics.
Perfect for a quick status check before running full diagnostics.
"""

import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("🚀 QUICK CHAMPION MODEL CHECK")
print("="*80)

# ============================================================================
# AUTO-DETECT CHAMPION MODEL
# ============================================================================

mv_model = None
selected_biomarkers_for_multivariate = None
champion_type = None

# Priority 1: 3-variable model (TFRC, APOF, FABP4)
target_vars = {'TFRC', 'APOF', 'FABP4'}

for var_name in ['cph_multi', 'cph_cv', 'final_model', 'best_model']:
    if var_name in globals():
        candidate = globals()[var_name]
        if isinstance(candidate, CoxPHFitter) and hasattr(candidate, 'params_'):
            param_names = [p.replace('Q("', '').replace('")', '').replace("Q('", "").replace("')", "")
                          for p in candidate.params_.index]
            if set(param_names) == target_vars:
                mv_model = candidate
                selected_biomarkers_for_multivariate = list(target_vars)
                champion_type = '3-Variable Champion'
                print(f"\n✅ Found: '{var_name}' (3-variable champion)")
                break

# Priority 2: champion_model dictionary
if mv_model is None and 'champion_model' in globals():
    champion_type = champion_model.get('type', 'Unknown')
    selected_biomarkers_for_multivariate = champion_model.get('biomarkers', [])
    mv_model = champion_model.get('model_object')

    if mv_model is None:
        model_names = {'EPV-cap': 'cph_epv', 'Stepwise': 'cph_stepwise', 'All-in': 'cph_all'}
        model_var = model_names.get(champion_type)
        if model_var and model_var in globals():
            mv_model = globals()[model_var]

    if mv_model:
        print(f"\n✅ Found: champion_model ({champion_type})")

# Priority 3: Any Cox model
if mv_model is None:
    for var_name, var_val in list(globals().items()):
        try:
            if isinstance(var_val, CoxPHFitter) and hasattr(var_val, 'params_'):
                mv_model = var_val
                param_names = [p.replace('Q("', '').replace('")', '').replace("Q('", "").replace("')", "")
                              for p in var_val.params_.index]
                selected_biomarkers_for_multivariate = param_names
                champion_type = 'Auto-detected'
                print(f"\n✅ Found: '{var_name}' (auto-detected)")
                break
        except:
            pass

if mv_model is None:
    print("\n❌ No Cox model found!")
    print("Please fit a model first.")
    raise ValueError("No model found")

# Infer required variables
if 'survival_time_col' not in globals() and 'df' in globals():
    for col in ['OS', 'PFS', 'OS_MONTHS', 'PFS_days', 'survival_time']:
        if col in df.columns:
            survival_time_col = col
            break

if 'event_col' not in globals() and 'df' in globals():
    for col in ['event', 'Event', 'status', 'OS_event']:
        if col in df.columns:
            event_col = col
            break

# ============================================================================
# QUICK STATUS SUMMARY
# ============================================================================

print("\n" + "="*80)
print("📊 MODEL SUMMARY")
print("="*80)

print(f"\nType: {champion_type}")
print(f"Variables: {len(selected_biomarkers_for_multivariate)}")
print(f"  → {selected_biomarkers_for_multivariate}")
print(f"C-index: {mv_model.concordance_index_:.4f}")

# AIC
for attr in ['AIC_partial_', 'AIC_', 'AIC']:
    if hasattr(mv_model, attr):
        aic = getattr(mv_model, attr)
        if aic is not None:
            print(f"AIC: {float(aic):.1f}")
            break

# ============================================================================
# QUICK DIAGNOSTICS (IF DATA AVAILABLE)
# ============================================================================

if all(v in globals() for v in ['df', 'survival_time_col', 'event_col']):
    print("\n" + "="*80)
    print("🔍 QUICK DIAGNOSTIC CHECKS")
    print("="*80)

    # Prepare data
    data_cols = [survival_time_col, event_col] + [v for v in selected_biomarkers_for_multivariate if v in df.columns]
    clean_df = df[data_cols].dropna()

    n_total = len(clean_df)
    n_events = int(clean_df[event_col].sum())
    n_vars = len(selected_biomarkers_for_multivariate)
    epv = n_events / n_vars

    # Results tracking
    issues = []

    # 1. Sample Size
    print(f"\n1. Sample Size Check:")
    print(f"   Samples: {n_total} | Events: {n_events} ({n_events/n_total*100:.1f}%) | EPV: {epv:.1f}")
    if epv >= 10:
        print("   ✅ Adequate sample size (EPV ≥ 10)")
    elif epv >= 5:
        print("   ⚠️ Borderline sample size (EPV 5-10)")
        issues.append("Borderline EPV")
    else:
        print("   ⚠️ Low sample size (EPV < 5)")
        issues.append("Low EPV")

    # 2. Proportional Hazards
    print(f"\n2. Proportional Hazards Test:")
    try:
        ph_test = proportional_hazard_test(mv_model, clean_df, time_transform='rank')

        violations = []
        for var in ph_test.summary.index:
            if var != 'GLOBAL':
                p_val = ph_test.summary.loc[var, 'p']
                if p_val <= 0.05:
                    clean_var = var.replace('Q("', '').replace('")', '').replace("Q('", "").replace("')", "")
                    violations.append(clean_var)

        if violations:
            print(f"   ⚠️ {len(violations)} variable(s) violate PH assumption:")
            for var in violations:
                print(f"      • {var}")
            issues.append(f"PH violations: {', '.join(violations)}")
        else:
            print("   ✅ All variables satisfy PH assumption")

        if 'GLOBAL' in ph_test.summary.index:
            global_p = ph_test.summary.loc['GLOBAL', 'p']
            if global_p <= 0.05:
                print(f"   ⚠️ Global test violated (p={global_p:.4f})")
                issues.append("Global PH violated")
            else:
                print(f"   ✅ Global test passed (p={global_p:.4f})")

    except Exception as e:
        print(f"   ⚠️ Test failed: {e}")

    # 3. Collinearity
    print(f"\n3. Collinearity Check:")
    try:
        available_vars = [v for v in selected_biomarkers_for_multivariate if v in clean_df.columns]

        if len(available_vars) >= 2:
            corr_matrix = clean_df[available_vars].corr()
            high_corr = []

            for i in range(len(corr_matrix)):
                for j in range(i+1, len(corr_matrix)):
                    r = corr_matrix.iloc[i, j]
                    if pd.notna(r) and abs(r) > 0.7:
                        high_corr.append((corr_matrix.index[i], corr_matrix.columns[j], r))

            if high_corr:
                print(f"   ⚠️ {len(high_corr)} high correlation(s) detected:")
                for v1, v2, r in high_corr:
                    print(f"      • {v1} ↔ {v2}: r={r:.3f}")
                issues.append(f"{len(high_corr)} high correlations")
            else:
                print("   ✅ No high correlations (|r| < 0.7)")
        else:
            print("   ⚠️ Not enough variables to check")

    except Exception as e:
        print(f"   ⚠️ Check failed: {e}")

    # 4. Outliers
    print(f"\n4. Outlier Check:")
    outlier_detected = False

    try:
        # Try multiple methods for computing residuals
        deviance = None

        # Method 1: Direct computation
        try:
            deviance = mv_model.compute_residuals(clean_df, kind='deviance')
            outlier_detected = True
        except:
            pass

        # Method 2: With reset index
        if deviance is None:
            try:
                model_data = clean_df.copy().reset_index(drop=True)
                deviance = mv_model.compute_residuals(model_data, kind='deviance')
                outlier_detected = True
            except:
                pass

        # Method 3: Manual calculation using martingale residuals
        if deviance is None:
            try:
                # Get predictions
                risk_scores = mv_model.predict_partial_hazard(clean_df)

                # Compute martingale residuals
                cumulative_hazard = mv_model.predict_cumulative_hazard(clean_df)
                final_cumhaz = cumulative_hazard.iloc[-1].values

                martingale = clean_df[event_col].values - final_cumhaz

                # Flag extreme values
                deviance = np.abs(martingale)
                outlier_detected = True
                print("   ℹ️  Using martingale residuals (deviance unavailable)")
            except:
                pass

        if deviance is not None:
            outliers = np.where(np.abs(deviance) > 2)[0]

            if len(outliers) > 0:
                print(f"   ⚠️ {len(outliers)} potential outliers (|residual| > 2)")
                if len(outliers) <= 5:
                    for idx in outliers:
                        print(f"      • Observation {idx}: residual={deviance[idx]:.3f}")
                else:
                    print(f"      • Showing top 5...")
                    for idx in outliers[:5]:
                        print(f"      • Observation {idx}: residual={deviance[idx]:.3f}")
                issues.append(f"{len(outliers)} outliers")
            else:
                print("   ✅ No extreme outliers detected")
        else:
            raise ValueError("All outlier detection methods failed")

    except Exception as e:
        print(f"   ⚠️ Could not compute outliers")
        print(f"   → Skipping outlier check (non-critical)")

    # 5. Coefficient Summary
    print(f"\n5. Model Coefficients:")
    for var, coef in mv_model.params_.items():
        hr = np.exp(coef)
        clean_var = var.replace('Q("', '').replace('")', '').replace("Q('", "").replace("')", "")

        # Get p-value
        sig = ""
        if hasattr(mv_model, 'summary'):
            try:
                p_val = mv_model.summary.loc[var, 'p']
                if p_val < 0.001:
                    sig = "***"
                elif p_val < 0.01:
                    sig = "**"
                elif p_val < 0.05:
                    sig = "*"
                print(f"   {clean_var}: β={coef:.3f}, HR={hr:.3f}, p={p_val:.4f} {sig}")
            except:
                print(f"   {clean_var}: β={coef:.3f}, HR={hr:.3f}")
        else:
            print(f"   {clean_var}: β={coef:.3f}, HR={hr:.3f}")

    # ========================================================================
    # FINAL VERDICT
    # ========================================================================

    print("\n" + "="*80)
    print("📋 DIAGNOSTIC SUMMARY")
    print("="*80)

    if len(issues) == 0:
        print("\n✅ ALL CHECKS PASSED!")
        print("   Your model looks good. Ready for publication-quality analysis.")
    else:
        print(f"\n⚠️ {len(issues)} ISSUE(S) DETECTED:")
        for i, issue in enumerate(issues, 1):
            print(f"   {i}. {issue}")
        print("\n   Recommendation: Run comprehensive diagnostics for detailed assessment.")

    print("\n" + "="*80)
    print("💡 NEXT STEPS")
    print("="*80)
    print("\nFor comprehensive diagnostics with plots:")
    print("   %run /mnt/user-data/outputs/run_diagnostics.py")
    print("\nFor full diagnostic report:")
    print("""
exec(open('/mnt/user-data/outputs/cox_diagnostics.py').read())
diagnostic_results = run_comprehensive_diagnostics(
    model=mv_model, df=df,
    survival_time_col=survival_time_col, event_col=event_col,
    selected_vars=selected_biomarkers_for_multivariate
)
    """)
else:
    print("\n⚠️ Data variables not found. Cannot run diagnostics.")
    print("   Please ensure: df, survival_time_col, event_col exist")

print("\n" + "="*80)
print("✅ QUICK CHECK COMPLETE")
print("="*80)

# **FULL COX MODEL QC**

# 5. Comprehensive Cox Model Quality Control (QC) Suite

## Overview
Full diagnostic assessment of the Cox proportional hazards model assumptions and performance.

### What this cell does:
1. **Proportional Hazards Testing**: Schoenfeld residuals analysis
2. **Multicollinearity Assessment**: VIF calculation
3. **Model Summary**: Coefficient table with CIs
4. **Outlier Detection**: Standardized residual analysis

---

## ⚙️ TUNABLE PARAMETERS

| Parameter | Default | Description | Location |
|-----------|---------|-------------|----------|
| `VIF_THRESHOLD` | `5.0` | VIF warning threshold | VIF section |
| `VIF_CRITICAL` | `10.0` | VIF critical threshold | VIF section |
| `PH_ALPHA` | `0.05` | PH test significance level | PH testing section |
| `OUTLIER_ZSCORE` | `3.0` | Z-score threshold for outliers | Outlier section |

---

### Key Outputs:
- `RESULTS_FOLDER/Cox_QC_Diagnostics/`
  - `cox_qc_diagnostics.xlsx`
  - Multiple diagnostic PNGs

### Interpretation Guide:
- **VIF > 5**: Moderate multicollinearity concern
- **VIF > 10**: Severe multicollinearity - consider variable removal
- **PH p-value < 0.05**: Proportional hazards assumption violated


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
from scipy.stats import zscore
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
import re
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# SETUP OUTPUT FOLDER
# ============================================================================
import os
from datetime import datetime

if 'RESULTS_FOLDER' in globals() and RESULTS_FOLDER is not None:
    QC_DIR = os.path.join(RESULTS_FOLDER, "Cox_QC_Diagnostics")
else:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    QC_DIR = f"Cox_QC_Results_{timestamp}"
os.makedirs(QC_DIR, exist_ok=True)
print(f"📂 QC outputs will be saved to: {QC_DIR}")

# Set style for publication-quality plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("="*80)
print("🔍 COMPREHENSIVE COX MODEL DIAGNOSTICS SUITE")
print("="*80)

mv_model = None

# ============================================================================
# HELPER FUNCTION: Extract actual column names from Q('...') format
# ============================================================================
def extract_column_name(var_name):
    """Extract actual column name from Q('...') or similar format"""
    match = re.search(r"Q\('([^']+)'\)", str(var_name))
    if match:
        return match.group(1)
    match = re.search(r"C\('([^']+)'\)", str(var_name))
    if match:
        return match.group(1)
    return str(var_name)

# ============================================================================
# STEP 1: FIND THE CHAMPION MODEL
# ============================================================================

# First priority: Check for champion_model
if 'champion_model' in globals() and champion_model is not None:
    print("\n🏆 Found 'champion_model'!")

    champion_type = champion_model.get('type', 'Unknown')
    champion_biomarkers = champion_model.get('biomarkers', [])

    print(f"   • Type: {champion_type}")
    print(f"   • Variables: {len(champion_biomarkers)}")

    # Try to get the fitted model object
    mv_model = champion_model.get('model_object')

    if mv_model is None:
        print("   ⚠️  Champion model object not stored, checking alternatives...")

        # Check specific model names based on champion type
        if champion_type == 'Stepwise' and 'cph_stepwise' in globals():
            mv_model = cph_stepwise
            print("   ✓ Found Stepwise model as 'cph_stepwise'")
        elif champion_type == 'EPV-cap' and 'cph_epv' in globals():
            mv_model = cph_epv
            print("   ✓ Found EPV model as 'cph_epv'")
    else:
        print("   ✓ Champion model object found")

    # Store champion variables for later use
    if champion_biomarkers:
        selected_biomarkers_for_multivariate = champion_biomarkers
        print(f"   ✓ Using {len(champion_biomarkers)} champion variables")

# Second priority: Check multivariate_results
if mv_model is None and 'multivariate_results' in globals():
    mr = multivariate_results
    print("\n✅ Found 'multivariate_results' with the following keys:")
    for key in mr.keys():
        value = mr[key]
        value_type = type(value).__name__
        print(f"  • {key}: {value_type}")

        if isinstance(value, CoxPHFitter) and mv_model is None:
            print(f"    ↳ This is a Cox model! Using this for diagnostics.")
            mv_model = value
        elif isinstance(value, dict) and mv_model is None:
            for subkey, subval in value.items():
                if isinstance(subval, CoxPHFitter):
                    print(f"      ↳ Found Cox model at [{key}][{subkey}]")
                    mv_model = subval
                    break

# Third priority: Search for standalone Cox models
if mv_model is None:
    print("\n🔍 Checking for standalone Cox models...")
    cox_models_found = []
    g_snapshot = dict(globals())
    for var_name, var_val in g_snapshot.items():
        try:
            if isinstance(var_val, CoxPHFitter):
                cox_models_found.append(var_name)
                print(f"  • Found Cox model: {var_name}")
        except Exception:
            pass

    if cox_models_found:
        print(f"\n📌 Using the first Cox model found: {cox_models_found[0]}")
        mv_model = g_snapshot[cox_models_found[0]]

# ============================================================================
# STEP 2: CHECK AND INFER REQUIRED VARIABLES
# ============================================================================
print("\n🔍 Checking for required variables...")

if 'survival_time_col' not in globals() and 'df' in globals():
    possible_time_cols = ['OS', 'OS_MONTHS', 'PFS', 'PFS_days', 'survival_time', 'time', 'duration']
    for col in possible_time_cols:
        if col in df.columns:
            survival_time_col = col
            print(f"  ✅ Inferred survival_time_col: '{col}'")
            break

if 'event_col' not in globals() and 'df' in globals():
    possible_event_cols = ['event', 'status', 'Event', 'STATUS', 'censored']
    for col in possible_event_cols:
        if col in df.columns:
            event_col = col
            print(f"  ✅ Inferred event_col: '{col}'")
            break

if mv_model is not None and hasattr(mv_model, 'params_'):
    model_var_names = list(mv_model.params_.index)
    selected_biomarkers_for_multivariate = [extract_column_name(var) for var in model_var_names]
    print(f"  ✅ Extracted {len(selected_biomarkers_for_multivariate)} variables from model")
    print(f"  ✅ Actual column names: {selected_biomarkers_for_multivariate}")

# ============================================================================
# STEP 3: RUN COMPREHENSIVE DIAGNOSTICS
# ============================================================================
if mv_model is not None and all(v in globals() for v in ['df', 'survival_time_col', 'event_col']):

    print("\n" + "="*80)
    print("📊 MODEL SUMMARY")
    print("="*80)

    n_obs = len(df)
    n_events = df[event_col].sum() if event_col in df.columns else np.nan
    n_censored = n_obs - n_events if not np.isnan(n_events) else np.nan
    n_covariates = len(mv_model.params_)

    print(f"\n📈 Sample Information:")
    print(f"  • Total observations: {n_obs}")
    print(f"  • Events: {int(n_events)} ({100*n_events/n_obs:.1f}%)")
    print(f"  • Censored: {int(n_censored)} ({100*n_censored/n_obs:.1f}%)")
    print(f"  • Number of covariates: {n_covariates}")
    print(f"  • Covariates: {', '.join(list(mv_model.params_.index))}")

    epv = n_events / n_covariates
    print(f"\n📊 Events Per Variable (EPV): {epv:.2f}")
    if epv < 10:
        print("  ⚠️ WARNING: EPV < 10 may lead to overfitting and biased estimates")
    elif epv < 20:
        print("  ⚠️ CAUTION: EPV between 10-20 is acceptable but higher is better")
    else:
        print("  ✅ EPV ≥ 20 is excellent for stable estimates")

    if hasattr(mv_model, 'concordance_index_'):
        print(f"\n📈 Model Performance:")
        print(f"  • Concordance Index (C-index): {mv_model.concordance_index_:.4f}")
        print(f"  • Log-likelihood: {mv_model.log_likelihood_:.2f}")
        if hasattr(mv_model, 'AIC_partial_'):
            print(f"  • AIC (partial): {mv_model.AIC_partial_:.2f}")

    # ========================================================================
    # COEFFICIENTS AND STATISTICS
    # ========================================================================
    print("\n" + "="*80)
    print("📋 COMPREHENSIVE MODEL COEFFICIENTS")
    print("="*80)

    summary_df = mv_model.summary.copy()
    summary_df['HR'] = np.exp(summary_df['coef'])
    summary_df['HR_lower'] = np.exp(summary_df['coef lower 95%'])
    summary_df['HR_upper'] = np.exp(summary_df['coef upper 95%'])

    print("\n" + summary_df.to_string())

    export_summary = summary_df[['coef', 'exp(coef)', 'se(coef)', 'coef lower 95%',
                                   'coef upper 95%', 'z', 'p', '-log2(p)']].copy()
    export_summary.columns = ['Beta', 'HR', 'SE', 'CI_lower', 'CI_upper', 'z_score', 'p_value', 'log2_p']
    export_summary['Significant'] = export_summary['p_value'] < 0.05
    export_summary['Actual_Var'] = selected_biomarkers_for_multivariate

    # ========================================================================
    # IMPROVED VIF CALCULATION
    # ========================================================================
    print("\n" + "="*80)
    print("📊 MULTICOLLINEARITY DIAGNOSTICS")
    print("="*80)

    vif_data = None
    condition_number = None

    try:
        available_cols = [col for col in selected_biomarkers_for_multivariate if col in df.columns]

        if len(available_cols) >= 2:
            # Get complete cases only
            X_raw = df[available_cols].dropna()
            n_complete = len(X_raw)

            print(f"\n📊 Data Quality Check:")
            print(f"  • Variables: {len(available_cols)}")
            print(f"  • Complete cases: {n_complete}/{n_obs} ({100*n_complete/n_obs:.1f}%)")

            if n_complete < 30:
                print(f"  ⚠️ WARNING: Only {n_complete} complete cases - VIF may be unreliable")

            # Calculate VIF on standardized data (more stable)
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X_raw)
            X_scaled_df = pd.DataFrame(X_scaled, columns=available_cols)

            print("\n🔍 Method 1: Variance Inflation Factor (VIF)")
            print("Note: VIF calculated on standardized data for numerical stability")

            vif_data = pd.DataFrame()
            vif_data["Variable"] = available_cols

            vif_values = []
            r_squared_values = []

            for i in range(len(available_cols)):
                try:
                    vif_val = variance_inflation_factor(X_scaled, i)
                    vif_values.append(vif_val)
                    # Calculate R² for transparency
                    r_sq = 1 - (1/vif_val) if vif_val != 0 else 0
                    r_squared_values.append(r_sq)
                except:
                    vif_values.append(np.nan)
                    r_squared_values.append(np.nan)

            vif_data["VIF"] = vif_values
            vif_data["R_squared"] = r_squared_values

            # More lenient interpretation for small samples
            def interpret_vif(vif, n_samples):
                if np.isnan(vif) or np.isinf(vif):
                    return "❌ Undefined"
                if n_samples < 50:
                    # More lenient thresholds for small samples
                    if vif < 10:
                        return "✅ Acceptable"
                    elif vif < 20:
                        return "⚠️ Moderate"
                    else:
                        return "❌ High"
                else:
                    if vif < 5:
                        return "✅ Excellent"
                    elif vif < 10:
                        return "⚠️ Moderate"
                    else:
                        return "❌ High"

            vif_data["Status"] = vif_data["VIF"].apply(lambda x: interpret_vif(x, n_complete))
            vif_data = vif_data.sort_values("VIF", ascending=False)

            print("\nVIF Interpretation (adjusted for small sample, n={}):" .format(n_complete))
            print("  • VIF < 10:  Acceptable")
            print("  • VIF 10-20: Moderate concern")
            print("  • VIF > 20:  High multicollinearity\n")
            print(vif_data.to_string(index=False))

            # Condition Number (alternative diagnostic)
            print("\n🔍 Method 2: Condition Number")
            print("Alternative multicollinearity diagnostic based on eigenvalues")

            try:
                # Calculate condition number
                eigenvalues = np.linalg.eigvals(X_scaled_df.corr())
                condition_number = np.sqrt(eigenvalues.max() / eigenvalues.min())

                print(f"\nCondition Number: {condition_number:.2f}")
                if condition_number < 10:
                    print("  ✅ Excellent: No multicollinearity concerns")
                elif condition_number < 30:
                    print("  ⚠️ Moderate: Some multicollinearity present")
                elif condition_number < 100:
                    print("  ❌ High: Significant multicollinearity")
                else:
                    print("  ❌ Severe: Very high multicollinearity")

            except Exception as e:
                print(f"  ⚠️ Could not calculate condition number: {e}")

            # Tolerance (1/VIF)
            print("\n🔍 Method 3: Tolerance (1/VIF)")
            vif_data["Tolerance"] = 1 / vif_data["VIF"]
            print("\nTolerance values (higher is better, >0.1 is acceptable):")
            for idx, row in vif_data.iterrows():
                tol = row["Tolerance"]
                status = "✅" if tol > 0.1 else "⚠️" if tol > 0.05 else "❌"
                print(f"  • {row['Variable']:15s}: {tol:.4f} {status}")

            # ⚠️ IMPORTANT NOTE about VIF with small samples
            print("\n" + "="*60)
            print("⚠️  IMPORTANT NOTE ON VIF WITH SMALL SAMPLES")
            print("="*60)
            print(f"With n={n_complete} observations and {len(available_cols)} variables:")
            print("• VIF can be inflated due to small sample size")
            print("• Pairwise correlations < 0.7 suggest multicollinearity is manageable")
            print("• Condition number is often more reliable for small samples")
            print("• Model coefficients and standard errors are the ultimate test")
            print("\nRecommendation: Trust the correlation matrix and model SE more than VIF")

            # Save VIF
            vif_data.to_csv(os.path.join(QC_DIR, 'cox_model_vif.csv'), index=False)
            print(f"\n💾 VIF saved to: {os.path.join(QC_DIR, 'cox_model_vif.csv')}")
        else:
            print(f"  ⚠️ Need at least 2 columns for VIF calculation, found {len(available_cols)}")

    except Exception as e:
        print(f"⚠️ Could not calculate multicollinearity diagnostics: {e}")
        import traceback
        traceback.print_exc()

    # ========================================================================
    # PROPORTIONAL HAZARDS ASSUMPTION TEST
    # ========================================================================
    print("\n" + "="*80)
    print("🔍 PROPORTIONAL HAZARDS ASSUMPTION TEST")
    print("="*80)

    ph_test_results = {}

    try:
        for transform in ['rank', 'km', 'identity']:
            print(f"\n📊 Time transform: '{transform}'")
            try:
                ph_test = proportional_hazard_test(mv_model, df, time_transform=transform)
                ph_summary = ph_test.summary.copy()

                for var in ph_summary.index:
                    if var != 'GLOBAL':
                        test_stat = ph_summary.loc[var, 'test_statistic']
                        p_val = ph_summary.loc[var, 'p']
                        status = "✅ PASS" if p_val > 0.05 else "❌ FAIL"
                        print(f"  • {var:30s}: χ²={test_stat:8.3f}, p={p_val:.4f} {status}")

                if 'GLOBAL' in ph_summary.index:
                    global_stat = ph_summary.loc['GLOBAL', 'test_statistic']
                    global_p = ph_summary.loc['GLOBAL', 'p']
                    global_status = "✅ SATISFIED" if global_p > 0.05 else "❌ VIOLATED"
                    print(f"\n  GLOBAL TEST: χ²={global_stat:.3f}, p={global_p:.4f} {global_status}")

                ph_test_results[transform] = ph_summary

            except Exception as e:
                print(f"  ⚠️ Could not run test with '{transform}': {e}")

        if ph_test_results:
            ph_xlsx_path = os.path.join(QC_DIR, 'cox_model_ph_tests.xlsx')
            with pd.ExcelWriter(ph_xlsx_path, engine='openpyxl') as writer:
                for transform, result in ph_test_results.items():
                    result.to_excel(writer, sheet_name=f'PH_test_{transform}')
            print(f"\n💾 PH tests saved to: {ph_xlsx_path}")

    except Exception as e:
        print(f"⚠️ Could not perform PH tests: {e}")

    # ========================================================================
    # SCHOENFELD RESIDUALS PLOTS AND DATA EXPORT
    # ========================================================================
    print("\n" + "="*80)
    print("📈 SCHOENFELD RESIDUALS PLOTS AND DATA EXPORT")
    print("="*80)

    schoenfeld_residuals = None

    try:
        # Calculate Schoenfeld residuals
        print("\n📊 Calculating Schoenfeld residuals...")
        schoenfeld_residuals = mv_model.compute_residuals(df, kind='schoenfeld')

        # Also calculate scaled Schoenfeld residuals for better interpretation
        try:
            scaled_schoenfeld = mv_model.compute_residuals(df, kind='scaled_schoenfeld')
            print("✅ Scaled Schoenfeld residuals calculated")
        except:
            scaled_schoenfeld = None
            print("⚠️ Could not calculate scaled Schoenfeld residuals")

        print(f"✅ Schoenfeld residuals shape: {schoenfeld_residuals.shape}")
        print(f"   Variables: {list(schoenfeld_residuals.columns)}")

        # Export raw Schoenfeld residuals to xlsx
        schoenfeld_xlsx_path = os.path.join(QC_DIR, 'cox_model_schoenfeld_residuals_data.xlsx')
        with pd.ExcelWriter(schoenfeld_xlsx_path, engine='openpyxl') as writer:
            # Raw Schoenfeld residuals
            schoenfeld_residuals.to_excel(writer, sheet_name='Schoenfeld_Residuals')

            # Scaled Schoenfeld residuals if available
            if scaled_schoenfeld is not None:
                scaled_schoenfeld.to_excel(writer, sheet_name='Scaled_Schoenfeld')

            # Summary statistics
            summary_stats = schoenfeld_residuals.describe()
            summary_stats.to_excel(writer, sheet_name='Summary_Statistics')

        print(f"✅ Schoenfeld residuals data saved to: {schoenfeld_xlsx_path}")

        # Generate combined plot using lifelines check_assumptions
        print("\n📈 Generating combined Schoenfeld residual plot...")
        fig_combined = plt.figure(figsize=(14, 10))
        mv_model.check_assumptions(df, p_value_threshold=0.05, show_plots=True, advice=False)
        plt.tight_layout()
        schoenfeld_combined_path = os.path.join(QC_DIR, 'cox_model_schoenfeld_residuals_combined.png')
        plt.savefig(schoenfeld_combined_path, dpi=300, bbox_inches='tight')
        print(f"✅ Combined Schoenfeld plot saved to: {schoenfeld_combined_path}")
        plt.show()

        # ====================================================================
        # GENERATE INDIVIDUAL DUAL-PANEL PLOTS (RANK + KM) WITH P-VALUES
        # ====================================================================
        print("\n📈 Generating individual dual-panel Schoenfeld plots for each variable...")
        print("   (rank-transformed + km-transformed with p-values)")

        from statsmodels.nonparametric.smoothers_lowess import lowess
        from scipy.stats import rankdata
        from lifelines import KaplanMeierFitter

        individual_plots_dir = os.path.join(QC_DIR, 'Schoenfeld_Individual_Plots')
        os.makedirs(individual_plots_dir, exist_ok=True)

        # Get p-values from PH tests (already calculated)
        p_values_rank = {}
        p_values_km = {}

        if 'rank' in ph_test_results:
            for var in ph_test_results['rank'].index:
                if var != 'GLOBAL':
                    p_values_rank[var] = ph_test_results['rank'].loc[var, 'p']

        if 'km' in ph_test_results:
            for var in ph_test_results['km'].index:
                if var != 'GLOBAL':
                    p_values_km[var] = ph_test_results['km'].loc[var, 'p']

        # Get event times
        event_times = scaled_schoenfeld.index.values if scaled_schoenfeld is not None else schoenfeld_residuals.index.values
        residuals_df = scaled_schoenfeld if scaled_schoenfeld is not None else schoenfeld_residuals

        # Calculate time transformations
        # Rank transform
        rank_times = rankdata(event_times, method='ordinal')

        # KM transform (survival probability at each event time)
        kmf = KaplanMeierFitter()
        kmf.fit(df[survival_time_col], event_observed=df[event_col])
        km_times = 1 - kmf.survival_function_at_times(event_times).values

        # Generate plot for each variable
        for var_name in residuals_df.columns:
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))

            residuals = residuals_df[var_name].values
            clean_var_name = extract_column_name(var_name)

            # Get p-values for this variable
            p_rank = p_values_rank.get(var_name, np.nan)
            p_km = p_values_km.get(var_name, np.nan)

            # Remove NaN for plotting
            mask = ~np.isnan(residuals)

            # ----- LEFT PANEL: Rank-transformed time -----
            ax1 = axes[0]
            ax1.scatter(rank_times[mask], residuals[mask], alpha=0.7, s=40, c='#1f77b4', edgecolors='none')

            # Add LOWESS smoothing with multiple bootstrap samples (like lifelines)
            try:
                # Main LOWESS line
                smoothed = lowess(residuals[mask], rank_times[mask], frac=0.6)
                ax1.plot(smoothed[:, 0], smoothed[:, 1], 'k-', linewidth=2.5, zorder=10)

                # Bootstrap confidence bands (gray lines)
                n_bootstrap = 10
                for _ in range(n_bootstrap):
                    idx = np.random.choice(len(residuals[mask]), size=len(residuals[mask]), replace=True)
                    boot_smoothed = lowess(residuals[mask][idx], rank_times[mask][idx], frac=0.6)
                    ax1.plot(boot_smoothed[:, 0], boot_smoothed[:, 1], 'gray', linewidth=0.8, alpha=0.5)
            except:
                pass

            ax1.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
            ax1.set_xlabel(f'rank-transformed time\n(p={p_rank:.4f})', fontsize=11)
            ax1.set_ylabel('Scaled Schoenfeld residuals', fontsize=11)
            ax1.grid(True, alpha=0.3)

            # ----- RIGHT PANEL: KM-transformed time -----
            ax2 = axes[1]
            ax2.scatter(km_times[mask], residuals[mask], alpha=0.7, s=40, c='#1f77b4', edgecolors='none')

            # Add LOWESS smoothing with bootstrap
            try:
                smoothed_km = lowess(residuals[mask], km_times[mask], frac=0.6)
                ax2.plot(smoothed_km[:, 0], smoothed_km[:, 1], 'k-', linewidth=2.5, zorder=10)

                for _ in range(n_bootstrap):
                    idx = np.random.choice(len(residuals[mask]), size=len(residuals[mask]), replace=True)
                    boot_smoothed = lowess(residuals[mask][idx], km_times[mask][idx], frac=0.6)
                    ax2.plot(boot_smoothed[:, 0], boot_smoothed[:, 1], 'gray', linewidth=0.8, alpha=0.5)
            except:
                pass

            ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
            ax2.set_xlabel(f'km-transformed time\n(p={p_km:.4f})', fontsize=11)
            ax2.set_ylabel('Scaled Schoenfeld residuals', fontsize=11)
            ax2.grid(True, alpha=0.3)

            # Main title
            fig.suptitle(f"Scaled Schoenfeld residuals of '{var_name}'", fontsize=13, fontweight='bold', y=1.02)

            plt.tight_layout()

            # Save
            safe_var_name = re.sub(r'[^\w\-_]', '_', clean_var_name)
            plot_path = os.path.join(individual_plots_dir, f'schoenfeld_{safe_var_name}.png')
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.close(fig)

            # Print status with p-values
            status_rank = "✅" if p_rank > 0.05 else "❌"
            status_km = "✅" if p_km > 0.05 else "❌"
            print(f"  ✅ Saved: schoenfeld_{safe_var_name}.png  |  p(rank)={p_rank:.4f} {status_rank}  |  p(km)={p_km:.4f} {status_km}")

        print(f"\n📂 Schoenfeld residual files saved to:")
        print(f"  • {schoenfeld_xlsx_path} (raw data)")
        print(f"  • {schoenfeld_combined_path} (combined plot)")
        print(f"  • {individual_plots_dir}/ (individual dual-panel plots with p-values)")

    except Exception as e:
        print(f"⚠️ Could not generate Schoenfeld residual plots: {e}")
        import traceback
        traceback.print_exc()

    # ========================================================================
    # CORRELATION MATRIX
    # ========================================================================
    print("\n" + "="*80)
    print("📊 CORRELATION MATRIX")
    print("="*80)

    corr_matrix = None

    try:
        available_cols = [col for col in selected_biomarkers_for_multivariate if col in df.columns]

        if len(available_cols) >= 2:
            corr_matrix = df[available_cols].corr()

            plt.figure(figsize=(10, 8))
            mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
            sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f',
                       cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                       square=True, linewidths=1, cbar_kws={"shrink": 0.8})
            plt.title('Correlation Matrix of Covariates', fontsize=14, fontweight='bold', pad=20)
            plt.tight_layout()
            corr_png_path = os.path.join(QC_DIR, 'cox_model_correlation_matrix.png')
            plt.savefig(corr_png_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"✅ Correlation matrix saved to: {corr_png_path}")

            print("\n🔍 Pairwise Correlation Assessment:")
            high_corr = []
            for i in range(len(corr_matrix)):
                for j in range(i+1, len(corr_matrix)):
                    r = corr_matrix.iloc[i, j]
                    if pd.notna(r):
                        var1 = corr_matrix.index[i]
                        var2 = corr_matrix.columns[j]
                        high_corr.append((var1, var2, r))

            # Sort by absolute correlation
            high_corr.sort(key=lambda x: abs(x[2]), reverse=True)

            print("\nAll pairwise correlations:")
            for var1, var2, corr in high_corr:
                if abs(corr) > 0.7:
                    status = "❌ HIGH"
                elif abs(corr) > 0.5:
                    status = "⚠️ MODERATE"
                else:
                    status = "✅ LOW"
                print(f"  • {var1:10s} ↔ {var2:10s}: r={corr:7.3f} {status}")

            if high_corr:
                max_corr = max(high_corr, key=lambda x: abs(x[2]))
                print(f"\nMaximum |correlation|: {abs(max_corr[2]):.3f} ({max_corr[0]} ↔ {max_corr[1]})")

                if all(abs(c[2]) < 0.7 for c in high_corr):
                    print("✅ All pairwise correlations below 0.7 threshold")

            corr_csv_path = os.path.join(QC_DIR, 'cox_model_correlation_matrix.csv')
            corr_matrix.to_csv(corr_csv_path)
            print(f"💾 Correlation matrix saved to: {corr_csv_path}")
        else:
            print(f"  ⚠️ Need at least 2 columns for correlation, found {len(available_cols)}")

    except Exception as e:
        print(f"⚠️ Could not generate correlation matrix: {e}")

    # ========================================================================
    # FOREST PLOT (Hazard Ratios)
    # ========================================================================
    print("\n" + "="*80)
    print("📊 FOREST PLOT - HAZARD RATIOS")
    print("="*80)

    try:
        forest_data = summary_df[['HR', 'HR_lower', 'HR_upper', 'p']].copy()
        forest_data = forest_data.sort_values('HR', ascending=True)

        fig, ax = plt.subplots(figsize=(10, max(6, len(forest_data) * 0.6)))

        y_pos = np.arange(len(forest_data))

        for i, (idx, row) in enumerate(forest_data.iterrows()):
            color = 'darkgreen' if row['p'] < 0.05 else 'gray'
            marker = 'D' if row['p'] < 0.05 else 'o'
            ax.errorbar(row['HR'], i,
                       xerr=[[row['HR'] - row['HR_lower']], [row['HR_upper'] - row['HR']]],
                       fmt=marker, color=color, capsize=5, capthick=2,
                       markersize=8, linewidth=2, alpha=0.8)

            label_x = max(row['HR_upper'] * 1.1, row['HR_upper'] + 0.2)
            ax.text(label_x, i,
                   f"HR={row['HR']:.2f}\np={row['p']:.4f}",
                   va='center', fontsize=9, bbox=dict(boxstyle='round',
                                                       facecolor='white',
                                                       alpha=0.8,
                                                       edgecolor=color))

        ax.set_yticks(y_pos)
        ax.set_yticklabels(forest_data.index)
        ax.set_xlabel('Hazard Ratio (HR)', fontsize=12, fontweight='bold')
        ax.set_title('Forest Plot - Hazard Ratios with 95% CI',
                    fontsize=14, fontweight='bold', pad=20)
        ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7, label='HR = 1 (no effect)')
        ax.grid(True, alpha=0.3, axis='x')
        ax.legend(loc='best')

        max_x = forest_data['HR_upper'].max() * 1.3
        ax.set_xlim(0, max_x)

        plt.tight_layout()
        forest_path = os.path.join(QC_DIR, 'cox_model_forest_plot.png')
        plt.savefig(forest_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"✅ Forest plot saved to: {forest_path}")

    except Exception as e:
        print(f"⚠️ Could not generate forest plot: {e}")

    # ========================================================================
    # EXPORT COMPREHENSIVE RESULTS
    # ========================================================================
    print("\n" + "="*80)
    print("💾 EXPORTING COMPREHENSIVE RESULTS")
    print("="*80)

    try:
        report_path = os.path.join(QC_DIR, 'cox_model_comprehensive_report.xlsx')
        with pd.ExcelWriter(report_path, engine='openpyxl') as writer:
            export_summary.to_excel(writer, sheet_name='Model_Summary')

            if vif_data is not None:
                vif_data.to_excel(writer, sheet_name='VIF', index=False)

            if corr_matrix is not None:
                corr_matrix.to_excel(writer, sheet_name='Correlation_Matrix')

            for transform, result in ph_test_results.items():
                result.to_excel(writer, sheet_name=f'PH_Test_{transform}')

            model_info = pd.DataFrame({
                'Metric': ['Total Observations', 'Events', 'Censored',
                          'Number of Covariates', 'EPV', 'C-index', 'Log-likelihood',
                          'AIC (partial)', 'Condition Number'],
                'Value': [n_obs, int(n_events), int(n_censored), n_covariates,
                         f'{epv:.2f}', f'{mv_model.concordance_index_:.4f}',
                         f'{mv_model.log_likelihood_:.2f}',
                         f'{mv_model.AIC_partial_:.2f}' if hasattr(mv_model, 'AIC_partial_') else 'N/A',
                         f'{condition_number:.2f}' if condition_number is not None else 'N/A']
            })
            model_info.to_excel(writer, sheet_name='Model_Info', index=False)

        print(f"✅ Comprehensive report saved to: {report_path}")

        summary_csv_path = os.path.join(QC_DIR, 'cox_model_summary.csv')
        export_summary.to_csv(summary_csv_path)
        print(f"✅ Model summary saved to: {summary_csv_path}")

        print("\n📂 All exported files:")
        print(f"  • {report_path} (main report)")
        print(f"  • {summary_csv_path} (coefficients)")
        if vif_data is not None:
            print(f"  • {os.path.join(QC_DIR, 'cox_model_vif.csv')} (multicollinearity)")
        if corr_matrix is not None:
            print(f"  • {os.path.join(QC_DIR, 'cox_model_correlation_matrix.csv')} (correlations)")
            print(f"  • {os.path.join(QC_DIR, 'cox_model_correlation_matrix.png')} (heatmap)")
        print(f"  • {os.path.join(QC_DIR, 'cox_model_ph_tests.xlsx')} (proportional hazards tests)")
        print(f"  • {os.path.join(QC_DIR, 'cox_model_schoenfeld_residuals_data.xlsx')} (Schoenfeld data)")
        print(f"  • {os.path.join(QC_DIR, 'cox_model_schoenfeld_residuals_combined.png')} (combined residual plot)")
        print(f"  • {os.path.join(QC_DIR, 'Schoenfeld_Individual_Plots/')} (individual residual plots)")
        print(f"  • {os.path.join(QC_DIR, 'cox_model_forest_plot.png')} (hazard ratios)")

    except Exception as e:
        print(f"⚠️ Error during export: {e}")
        import traceback
        traceback.print_exc()

    print("\n" + "="*80)
    print("✅ COMPREHENSIVE DIAGNOSTICS COMPLETE!")
    print("="*80)

else:
    print("\n" + "="*80)
    print("❌ CANNOT RUN DIAGNOSTICS")
    print("="*80)
    if mv_model is None:
        print("\n⚠️ No Cox model found in workspace")
    if 'df' not in globals():
        print("⚠️ DataFrame 'df' not found")
    if 'survival_time_col' not in globals():
        print("⚠️ Variable 'survival_time_col' not found")
    if 'event_col' not in globals():
        print("⚠️ Variable 'event_col' not found")

# **RISK GROUP STRATIFICATION**

# 6. Risk Group Stratification and Survival Analysis

## Overview
Stratifies patients into risk groups based on the Cox model's linear predictor and performs comprehensive survival analysis.

### What this cell does:
1. **Risk Score Calculation**: Computes partial hazard for each patient
2. **Stratification**: Divides patients into risk tertiles
3. **Statistical Testing**: Log-rank tests, pairwise comparisons, hazard ratios
4. **Kaplan-Meier Analysis**: Survival curves, median survival times
5. **Visualizations**: KM curves, risk score distributions, forest plot

---

## ⚙️ TUNABLE PARAMETERS

| Parameter | Default | Description | Location |
|-----------|---------|-------------|----------|
| `n_groups` | `3` | Number of risk groups (tertiles) | Stratification section |
| `BONFERRONI_ALPHA` | `0.05` | Significance after Bonferroni | Pairwise tests |
| `group_names` | `['Low Risk', 'Intermediate Risk', 'High Risk']` | Risk group labels | Stratification section |

**To change to quartiles:** Set `n_groups = 4` and update `group_names`

---

### Key Outputs:
- `RESULTS_FOLDER/Risk_Stratification/`
  - `risk_stratification_complete.xlsx` (11 sheets)
  - KM curves, distribution plots, forest plot


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test, pairwise_logrank_test
from lifelines.plotting import add_at_risk_counts
from scipy import stats
import warnings
import os
from datetime import datetime
warnings.filterwarnings('ignore')

print("="*80)
print("🎯 RISK STRATIFICATION AND GROUP ANALYSIS (DYNAMIC TIME UNIT)")
print("="*80)

# ============================================================================
# STEP 0: VERIFY EXPORT FOLDER
# ============================================================================
print("\n📁 Verifying export folder...")

if 'RESULTS_FOLDER' in globals() and RESULTS_FOLDER is not None:
    EXPORT_DIR = RESULTS_FOLDER
    print(f"   ✅ Using pipeline output folder: {EXPORT_DIR}")
else:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    EXPORT_DIR = f"Risk_Stratification_Results_{timestamp}"
    os.makedirs(EXPORT_DIR, exist_ok=True)
    print(f"   ⚠️ RESULTS_FOLDER not found. Created: {EXPORT_DIR}")

RISK_STRAT_DIR = os.path.join(EXPORT_DIR, "Risk_Stratification")
os.makedirs(RISK_STRAT_DIR, exist_ok=True)
print(f"   📂 Risk stratification outputs: {RISK_STRAT_DIR}")

# ============================================================================
# STEP 1: EXTRACT CHAMPION MODEL
# ============================================================================
print("\n🏆 Extracting champion model...")

mv_model = None
model_source = None

if 'champion_model' in globals() and champion_model is not None:
    print("   ✅ Found 'champion_model'!")
    champion_type = champion_model.get('type', 'Unknown')
    champion_biomarkers = champion_model.get('biomarkers', [])
    print(f"   • Type: {champion_type}")
    print(f"   • Variables: {len(champion_biomarkers)}")
    mv_model = champion_model.get('model_object')
    if mv_model is None:
        if champion_type == 'Stepwise' and 'cph_stepwise' in globals():
            mv_model = cph_stepwise
            model_source = 'cph_stepwise'
        elif champion_type == 'EPV-cap' and 'cph_epv' in globals():
            mv_model = cph_epv
            model_source = 'cph_epv'
        elif 'cph' in globals():
            mv_model = cph
            model_source = 'cph'
    if mv_model is not None:
        print(f"   ✓ Champion model object found")
    if champion_biomarkers:
        selected_biomarkers_for_multivariate = champion_biomarkers
        print(f"   ✓ Using {len(champion_biomarkers)} champion variables")

if mv_model is None and 'multivariate_results' in globals():
    print("   Searching multivariate_results...")
    mr = multivariate_results
    for key in mr.keys():
        value = mr[key]
        if isinstance(value, CoxPHFitter):
            mv_model = value
            model_source = f'multivariate_results[{key}]'
            break
        elif isinstance(value, dict):
            for subkey, subval in value.items():
                if isinstance(subval, CoxPHFitter):
                    mv_model = subval
                    model_source = f'multivariate_results[{key}][{subkey}]'
                    break

if mv_model is None:
    g_snapshot = dict(globals())
    for var_name in ['cph', 'cph_epv', 'cph_stepwise', 'final_model']:
        if var_name in g_snapshot:
            try:
                if isinstance(g_snapshot[var_name], CoxPHFitter):
                    mv_model = g_snapshot[var_name]
                    model_source = var_name
                    break
            except:
                pass

if mv_model is None:
    print("   ❌ No Cox model found!")
else:
    print(f"   ✅ Champion model ready for risk stratification")

# ============================================================================
# TIME UNIT EXTRACTION
# ============================================================================
print("\n📊 Extracting time unit from previous analysis...")
if 'time_unit' in globals() and time_unit is not None:
    TIME_UNIT = time_unit
    print(f"   ✅ Found time_unit from univariate analysis: {TIME_UNIT}")
elif 'TIME_UNIT' in globals():
    TIME_UNIT = TIME_UNIT
    print(f"   ✅ Using existing TIME_UNIT: {TIME_UNIT}")
else:
    TIME_UNIT = "days"
    print(f"   ⚠️ No time_unit found, defaulting to: {TIME_UNIT}")
print(f"   ✅ All outputs will use: {TIME_UNIT}")

# ============================================================================
# MAIN ANALYSIS
# ============================================================================
if mv_model is not None and 'df' in globals():
    if 'event_col' not in globals():
        event_col = 'event'
    if 'survival_time_col' not in globals():
        survival_time_col = 'duration'

    try:
        # Calculate risk scores
        print("\n📊 Calculating risk scores for all patients...")
        X_model = df[selected_biomarkers_for_multivariate].copy()
        risk_scores = mv_model.predict_partial_hazard(X_model)
        df['risk_score'] = risk_scores
        df['log_risk_score'] = np.log(risk_scores)

        print(f"✅ Risk scores calculated for {len(risk_scores)} patients")

        # ====================================================================
        # DETAILED RISK SCORE DISTRIBUTION
        # ====================================================================
        print("\n📈 Risk Score Distribution:")
        print(f"  • Mean: {risk_scores.mean():.3f}")
        print(f"  • Median: {risk_scores.median():.3f}")
        print(f"  • Std Dev: {risk_scores.std():.3f}")
        print(f"  • Range: [{risk_scores.min():.3f}, {risk_scores.max():.3f}]")
        q1, q3 = risk_scores.quantile([0.25, 0.75])
        print(f"  • IQR: [{q1:.3f}, {q3:.3f}]")

        # ====================================================================
        # STRATIFICATION
        # ====================================================================
        print("\n" + "="*80)
        print("📊 STRATIFYING PATIENTS INTO RISK GROUPS")
        print("="*80)

        n_groups = 3
        group_names = ['Low Risk', 'Intermediate Risk', 'High Risk']
        df['risk_group'] = pd.qcut(df['risk_score'], q=3, labels=group_names, duplicates='drop')
        actual_groups = sorted(df['risk_group'].unique(), key=lambda x: df[df['risk_group']==x]['risk_score'].mean())

        print(f"\n✅ Patients stratified into {len(actual_groups)} risk groups:")

        # ====================================================================
        # DETAILED GROUP STATISTICS
        # ====================================================================
        group_stats = []
        km_medians = {}

        for group in actual_groups:
            group_data = df[df['risk_group'] == group]
            n_patients = len(group_data)
            n_events_group = int(group_data[event_col].sum())
            event_rate = 100 * n_events_group / n_patients

            # KM for median
            kmf_temp = KaplanMeierFitter()
            kmf_temp.fit(group_data[survival_time_col], group_data[event_col])
            km_median = kmf_temp.median_survival_time_
            km_medians[group] = km_median

            risk_min = group_data['risk_score'].min()
            risk_max = group_data['risk_score'].max()

            group_stats.append({
                'Group': group, 'N': n_patients, 'Events': n_events_group,
                'Event_Rate_%': round(event_rate, 1),
                'Risk_Score_Mean': group_data['risk_score'].mean(),
                'Risk_Score_Median': group_data['risk_score'].median(),
                'Risk_Score_SD': group_data['risk_score'].std(),
                'Risk_Score_Min': risk_min,
                'Risk_Score_Max': risk_max,
                f'Survival_Mean_{TIME_UNIT}': group_data[survival_time_col].mean(),
                f'Survival_Median_{TIME_UNIT}': group_data[survival_time_col].median(),
                f'KM_Median_{TIME_UNIT}': km_median,
            })

            # Detailed printout
            print(f"\n{group}:")
            print(f"  • N = {n_patients} ({100*n_patients/len(df):.1f}%)")
            print(f"  • Events = {n_events_group} ({event_rate:.1f}%)")
            print(f"  • Risk Score: {group_data['risk_score'].median():.3f} (median), {group_data['risk_score'].mean():.3f} (mean)")
            print(f"  • Risk Score Range: [{risk_min:.3f}, {risk_max:.3f}]")
            print(f"  • Survival Time: {group_data[survival_time_col].median():.1f} (median), {group_data[survival_time_col].mean():.1f} (mean) {TIME_UNIT}")
            km_str = f"{km_median:.1f}" if not np.isnan(km_median) else "Not reached"
            print(f"  • KM Median: {km_str} {TIME_UNIT}")

        group_stats_df = pd.DataFrame(group_stats)

        # ====================================================================
        # LOG-RANK TESTS
        # ====================================================================
        print("\n" + "="*80)
        print("📊 LOG-RANK TESTS FOR GROUP COMPARISONS")
        print("="*80)

        results_overall = multivariate_logrank_test(df[survival_time_col], df['risk_group'], df[event_col])

        print("\n🔍 Overall Log-Rank Test (all groups):")
        print(f"  • Test statistic: {results_overall.test_statistic:.3f}")
        print(f"  • p-value: {results_overall.p_value:.6f}")
        print(f"  • Degrees of freedom: {results_overall.degrees_of_freedom}")

        if results_overall.p_value < 0.001:
            print("  ✅ Highly significant difference (p < 0.001)")
        elif results_overall.p_value < 0.01:
            print("  ✅ Very significant difference (p < 0.01)")
        elif results_overall.p_value < 0.05:
            print("  ✅ Significant difference (p < 0.05)")
        else:
            print("  ⚠️ No significant difference (p >= 0.05)")

        pairwise_df = None
        if len(actual_groups) > 2:
            results_pairwise = pairwise_logrank_test(df[survival_time_col], df['risk_group'], df[event_col])
            pairwise_df = results_pairwise.summary.copy()
            pairwise_df['Significant'] = pairwise_df['p'] < 0.05
            pairwise_df['Bonferroni_Significant'] = pairwise_df['p'] < (0.05 / len(pairwise_df))

            print("\n🔍 Pairwise Log-Rank Tests (with Bonferroni correction):")
            print(pairwise_df.to_string())

            n_sig = pairwise_df['Significant'].sum()
            n_bonf = pairwise_df['Bonferroni_Significant'].sum()
            print(f"\n  • {n_sig}/{len(pairwise_df)} comparisons significant at p<0.05")
            print(f"  • {n_bonf}/{len(pairwise_df)} comparisons significant after Bonferroni correction")

        # ====================================================================
        # HAZARD RATIOS
        # ====================================================================
        print("\n" + "="*80)
        print("📊 HAZARD RATIOS (HR) BETWEEN RISK GROUPS")
        print("="*80)

        reference_group = actual_groups[0]
        print(f"\n🎯 Reference group: {reference_group}")

        hr_results = []
        for group in actual_groups[1:]:
            df_temp = df[df['risk_group'].isin([reference_group, group])].copy()
            df_temp['group_indicator'] = (df_temp['risk_group'] == group).astype(int)
            try:
                cph_temp = CoxPHFitter()
                cph_temp.fit(
                    df_temp[[survival_time_col, event_col, 'group_indicator']],
                    duration_col=survival_time_col,
                    event_col=event_col
                )
                hr = cph_temp.hazard_ratios_['group_indicator']
                ci = cph_temp.confidence_intervals_.loc['group_indicator']
                ci_lower = np.exp(ci['95% lower-bound'])
                ci_upper = np.exp(ci['95% upper-bound'])
                p_val = cph_temp.summary.loc['group_indicator', 'p']

                hr_results.append({
                    'Comparison': f"{group} vs {reference_group}",
                    'HR': hr,
                    'HR_lower_95': ci_lower,
                    'HR_upper_95': ci_upper,
                    'p_value': p_val
                })

                sig_marker = "✅" if p_val < 0.05 else ""
                print(f"\n{group} vs {reference_group}:")
                print(f"  • HR = {hr:.3f} (95% CI: {ci_lower:.3f}-{ci_upper:.3f})")
                print(f"  • p-value = {p_val:.6f} {sig_marker}")
                print(f"  • Interpretation: {group} has {hr:.2f}x the hazard of {reference_group}")
            except Exception as e:
                print(f"\n{group} vs {reference_group}: Could not calculate ({e})")

        hr_df = pd.DataFrame(hr_results) if hr_results else None

        # ====================================================================
        # KAPLAN-MEIER PLOT - COMPACT VERSION WITH MEDIAN LABELS
        # ====================================================================
        print("\n" + "="*80)
        print("📈 GENERATING ENHANCED KAPLAN-MEIER CURVES")
        print("="*80)

        # Calculate KM medians first
        km_medians = {}
        for group in actual_groups:
            group_data = df[df['risk_group'] == group]
            kmf_temp = KaplanMeierFitter()
            kmf_temp.fit(group_data[survival_time_col], group_data[event_col])
            km_medians[group] = kmf_temp.median_survival_time_

        print(f"\n🎯 Median Survival Times ({TIME_UNIT}):")
        for group in actual_groups:
            km_val = km_medians.get(group, np.nan)
            km_str = f"{km_val:.1f}" if not np.isnan(km_val) else "Not reached"
            print(f"  • {group}: {km_str} {TIME_UNIT}")

        # Create larger figure
        fig, ax = plt.subplots(figsize=(14, 9))

        # Color scheme
        colors_dict = {'Low Risk': '#2ecc71', 'Intermediate Risk': '#f39c12', 'High Risk': '#e74c3c'}
        color_list = [colors_dict.get(g, '#3498db') for g in actual_groups]

        kmf_list = []
        km_survival_data = []

        for i, (group, color) in enumerate(zip(actual_groups, color_list)):
            group_data = df[df['risk_group'] == group]
            kmf = KaplanMeierFitter()
            kmf.fit(
                group_data[survival_time_col],
                group_data[event_col],
                label=f'{group} (n={len(group_data)}, events={int(group_data[event_col].sum())})'
            )
            kmf.plot_survival_function(ax=ax, ci_show=True, color=color, linewidth=2.5)
            kmf_list.append((group, kmf))

            # Store survival data for export
            km_df_temp = kmf.survival_function_.copy()
            km_df_temp.columns = [f'{group}_Survival_Prob']
            km_survival_data.append(km_df_temp)

        # Draw horizontal line at 0.5 (median reference)
        ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, linewidth=1)

        # Add median dropdown lines and labels
        for i, (group, kmf) in enumerate(kmf_list):
            color = color_list[i]
            med = kmf.median_survival_time_
            if not np.isnan(med):
                # Vertical dropdown line from curve to x-axis
                ax.vlines(x=med, ymin=0, ymax=0.5, colors=color, linestyles='--', alpha=0.7, linewidth=1.5)

                # Add median label - VERTICAL orientation, staggered horizontally
                x_offset = med + (i - 1) * 0.3  # Slight horizontal stagger
                ax.text(
                    med, 0.25,
                    f'{med:.1f} {TIME_UNIT}',
                    color='white',
                    fontsize=9,
                    fontweight='bold',
                    ha='center',
                    va='center',
                    rotation=90,  # Vertical text
                    bbox=dict(
                        boxstyle='round,pad=0.3',
                        facecolor=color,
                        edgecolor='black',
                        alpha=0.95,
                        linewidth=1.5
                    )
                )

        # Add at-risk table
        add_at_risk_counts(*[kmf for _, kmf in kmf_list], ax=ax, rows_to_show=['At risk'])

        # Formatting
        ax.set_xlabel(f'Time ({TIME_UNIT})', fontsize=13, fontweight='bold')
        ax.set_ylabel('Survival Probability', fontsize=13, fontweight='bold')
        ax.set_title(
            f'Kaplan-Meier Survival Curves by Risk Group\n(Median Survival Times in {TIME_UNIT.capitalize()})',
            fontsize=15,
            fontweight='bold'
        )
        ax.set_ylim(-0.02, 1.02)
        ax.set_xlim(left=0)
        ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
        ax.legend(loc='upper right', fontsize=11, framealpha=0.95)

        # Log-rank p-value annotation
        if results_overall.p_value < 0.001:
            p_text = "Log-rank p < 0.001"
        else:
            p_text = f"Log-rank p = {results_overall.p_value:.4f}"

        ax.text(
            0.02, 0.98,
            p_text,
            transform=ax.transAxes,
            fontsize=12,
            fontweight='bold',
            va='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='black', alpha=0.95)
        )

        # Median survival summary box (bottom right)
        median_text_lines = ["Median Survival:"]
        for group in actual_groups:
            med = km_medians.get(group, np.nan)
            if not np.isnan(med):
                med_str = f"{med:.1f} {TIME_UNIT}"
            else:
                med_str = "Not reached"
            median_text_lines.append(f"  • {group}: {med_str}")

        median_text = "\n".join(median_text_lines)

        ax.text(
            0.98, 0.02,
            median_text,
            transform=ax.transAxes,
            fontsize=10,
            va='bottom',
            ha='right',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', edgecolor='gray', alpha=0.95),
            family='monospace'
        )

        plt.tight_layout()

        # Save
        km_path = os.path.join(RISK_STRAT_DIR, 'km_curves_by_risk_group.png')
        plt.savefig(km_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n✅ KM curves saved: {km_path}")

        # ====================================================================
        # DISTRIBUTION PLOTS
        # ====================================================================
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        ax = axes[0, 0]
        for group, color in zip(actual_groups, color_list):
            ax.hist(df[df['risk_group']==group]['risk_score'], bins=15, alpha=0.6, label=group, color=color)
        ax.set_xlabel('Risk Score')
        ax.set_ylabel('Frequency')
        ax.set_title('Risk Score Distribution')
        ax.legend()

        ax = axes[0, 1]
        sns.boxplot(data=df, x='risk_group', y='risk_score', palette=color_list, ax=ax, order=actual_groups)
        ax.set_title('Risk Score by Group')

        ax = axes[1, 0]
        sns.violinplot(data=df, x='risk_group', y=survival_time_col, palette=color_list, ax=ax, order=actual_groups)
        ax.set_ylabel(f'Survival Time ({TIME_UNIT})')
        ax.set_title('Survival Distribution')

        ax = axes[1, 1]
        for group, color in zip(actual_groups, color_list):
            gd = df[df['risk_group']==group]
            ax.scatter(gd['risk_score'], gd[survival_time_col], c=color, label=group, alpha=0.7)
        ax.set_xlabel('Risk Score')
        ax.set_ylabel(f'Survival Time ({TIME_UNIT})')
        ax.legend()

        plt.tight_layout()
        dist_path = os.path.join(RISK_STRAT_DIR, 'risk_score_distributions.png')
        plt.savefig(dist_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"✅ Distributions saved: {dist_path}")

        # ====================================================================
        # FOREST PLOT
        # ====================================================================
        if hr_df is not None and len(hr_df) > 0:
            fig, ax = plt.subplots(figsize=(10, 3 + len(hr_df)*0.5))
            for i, row in hr_df.iterrows():
                col = '#e74c3c' if row['p_value'] < 0.05 else '#3498db'
                ax.errorbar(
                    row['HR'], i,
                    xerr=[[row['HR']-row['HR_lower_95']], [row['HR_upper_95']-row['HR']]],
                    fmt='o', color=col, markersize=10, capsize=5
                )
            ax.axvline(x=1, color='black', linestyle='--')
            ax.set_yticks(range(len(hr_df)))
            ax.set_yticklabels(hr_df['Comparison'])
            ax.set_xlabel('Hazard Ratio (95% CI)')
            ax.set_title(f'Hazard Ratios (Reference: {reference_group})')
            plt.tight_layout()
            forest_path = os.path.join(RISK_STRAT_DIR, 'hazard_ratio_forest_plot.png')
            plt.savefig(forest_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"✅ Forest plot saved: {forest_path}")

        # ====================================================================
        # COMPREHENSIVE XLSX EXPORT
        # ====================================================================
        xlsx_path = os.path.join(RISK_STRAT_DIR, 'risk_stratification_complete.xlsx')

        with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
            # Summary
            summary_data = {
                'Analysis_Date': datetime.now().isoformat(),
                'Total_Patients': len(df),
                'Total_Events': int(df[event_col].sum()),
                'N_Risk_Groups': len(actual_groups),
                'Time_Unit': TIME_UNIT,
                'Variables': ', '.join(selected_biomarkers_for_multivariate)
            }
            pd.DataFrame([summary_data]).to_excel(writer, 'Summary', index=False)

            # Group Stats
            group_stats_df.to_excel(writer, 'Group_Statistics', index=False)

            # Patient Data
            cols = [survival_time_col, event_col, 'risk_score', 'log_risk_score', 'risk_group'] + selected_biomarkers_for_multivariate
            df[[c for c in cols if c in df.columns]].to_excel(writer, 'Patient_Risk_Scores', index=True)

            # HR
            if hr_df is not None:
                hr_df.to_excel(writer, 'Hazard_Ratios', index=False)

            # Log-Rank
            logrank_data = {
                'Test': 'Overall',
                'Statistic': results_overall.test_statistic,
                'p_value': results_overall.p_value,
                'df': results_overall.degrees_of_freedom
            }
            pd.DataFrame([logrank_data]).to_excel(writer, 'LogRank_Overall', index=False)

            if pairwise_df is not None:
                pairwise_df.to_excel(writer, 'LogRank_Pairwise')

            # KM Medians
            km_med = []
            for group in actual_groups:
                gd = df[df['risk_group']==group]
                kmf = KaplanMeierFitter()
                kmf.fit(gd[survival_time_col], gd[event_col])
                km_med.append({
                    'Group': group,
                    f'KM_Median_{TIME_UNIT}': kmf.median_survival_time_,
                    'N': len(gd),
                    'Events': int(gd[event_col].sum())
                })
            pd.DataFrame(km_med).to_excel(writer, 'KM_Medians', index=False)

            # KM Curves Data
            if km_survival_data:
                pd.concat(km_survival_data, axis=1).to_excel(writer, 'KM_Survival_Data')

            # Model Coefficients
            try:
                mv_model.summary.to_excel(writer, 'Model_Coefficients')
            except:
                pass

        print(f"\n✅ Complete XLSX: {xlsx_path}")

        print("\n" + "="*80)
        print("✅ RISK STRATIFICATION COMPLETE!")
        print(f"📂 All outputs saved to: {RISK_STRAT_DIR}")
        print("="*80)

    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()
else:
    print("\n❌ Cannot perform risk stratification - mv_model or df not found")

# **Comprehensive data export cell to xlsx**

# 7. Comprehensive Data Export

## Overview
Exports all analysis results to comprehensive Excel workbooks.

### What this cell does:
1. **Champion Model Export**: Complete model parameters
2. **Cross-Validation Data**: Fold-level metrics
3. **Diagnostic Data**: PH tests, VIF, residuals
4. **Risk Stratification Data**: Patient scores and groups
5. **Comparison Data**: UV vs MV, model comparisons

### Note:
This cell consolidates outputs from all previous analyses.


In [ ]:
# ============================================================================
# COMPREHENSIVE DATA EXPORT FOR ALL VISUALIZATIONS (UPDATED VERSION)
# Exports underlying data for CV plots, diagnostics, risk stratification
# Includes: Champion model info, time conversions, enhanced CV, model comparison
# Run after completing all analyses
# ============================================================================

import numpy as np
import pandas as pd
import os
from datetime import datetime

print("\n" + "=" * 120)
print("📊 COMPREHENSIVE VISUALIZATION DATA EXPORT (UPDATED)")
print("=" * 120)

# Create export timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
export_folder = os.path.join(RESULTS_FOLDER, "Exported_Plot_Data")
os.makedirs(export_folder, exist_ok=True)

exported_files = []

# ============================================================================
# SECTION 0: CHAMPION MODEL & TIME UNIT INFORMATION
# ============================================================================
print("\n" + "=" * 80)
print("🏆 SECTION 0: EXPORTING CHAMPION MODEL & TIME UNIT INFORMATION")
print("=" * 80)

try:
    champion_export_path = os.path.join(export_folder, f"Champion_Model_Info_{timestamp}.xlsx")

    with pd.ExcelWriter(champion_export_path, engine='openpyxl') as writer:

        # Sheet 1: Champion Model Information
        if 'champion_model' in locals() and champion_model is not None:
            champion_info = pd.DataFrame([{
                'Champion_Type': champion_model.get('type', 'Unknown'),
                'N_Variables': len(champion_model.get('biomarkers', [])),
                'Training_C_Index': champion_model.get('train_c_index', np.nan),
                'CV_C_Index': champion_model.get('cv_c_index', np.nan),
                'CV_Std': champion_model.get('cv_std', np.nan),
                'Optimism': champion_model.get('optimism', np.nan),
                'AIC': champion_model.get('AIC', np.nan),
                'BIC': champion_model.get('BIC', np.nan),
                'Formula': champion_model.get('formula', '')
            }])
            champion_info.to_excel(writer, sheet_name='Champion_Info', index=False)

            # Champion biomarkers
            champion_biomarkers_df = pd.DataFrame({
                'Rank': range(1, len(champion_model['biomarkers']) + 1),
                'Biomarker': champion_model['biomarkers']
            })
            champion_biomarkers_df.to_excel(writer, sheet_name='Champion_Biomarkers', index=False)
            print(f"  ✓ Exported champion model info: {champion_model.get('type')} with {len(champion_model['biomarkers'])} variables")
        else:
            placeholder = pd.DataFrame([{'Note': 'No champion model information available'}])
            placeholder.to_excel(writer, sheet_name='Champion_Info', index=False)
            print(f"  ⚠️ No champion model found")

        # Sheet 2: Time Unit Information
        time_unit_info = pd.DataFrame([{
            'Original_Time_Unit': globals().get('original_time_unit', globals().get('time_unit', 'Unknown')),
            'Display_Time_Unit': 'months',
            'Conversion_Factor': globals().get('conversion_factor', 1.0),
            'Conversion_Applied': globals().get('original_time_unit', 'months') != 'months',
            'Note': f"All survival times exported in months. Original unit: {globals().get('original_time_unit', globals().get('time_unit', 'Unknown'))}"
        }])
        time_unit_info.to_excel(writer, sheet_name='Time_Unit_Info', index=False)
        print(f"  ✓ Exported time unit information")

        # Sheet 3: Model Comparison (if both EPV and Stepwise exist)
        if all(v in globals() for v in ['epv_results', 'stepwise_results_dict']):
            comparison_data = pd.DataFrame({
                'Metric': [
                    'Number of Variables',
                    'Training C-index',
                    'CV Mean C-index',
                    'CV Std C-index',
                    'Optimism',
                    'AIC',
                    'BIC'
                ],
                'EPV-Cap Model': [
                    len(globals().get('epv_biomarkers_list', [])),
                    epv_results.get('model_concordance', np.nan),
                    globals().get('epv_cv', {}).get('mean_c_index', np.nan) if 'epv_cv' in globals() else np.nan,
                    globals().get('epv_cv', {}).get('std_c_index', np.nan) if 'epv_cv' in globals() else np.nan,
                    np.nan,  # Calculate if needed
                    epv_results.get('AIC', np.nan),
                    epv_results.get('BIC', np.nan)
                ],
                'Stepwise Model': [
                    len(globals().get('stepwise_biomarkers_list', [])),
                    stepwise_results_dict.get('model_concordance', np.nan),
                    globals().get('stepwise_cv', {}).get('mean_c_index', np.nan) if 'stepwise_cv' in globals() else np.nan,
                    globals().get('stepwise_cv', {}).get('std_c_index', np.nan) if 'stepwise_cv' in globals() else np.nan,
                    np.nan,  # Calculate if needed
                    stepwise_results_dict.get('AIC', np.nan),
                    stepwise_results_dict.get('BIC', np.nan)
                ]
            })
            comparison_data.to_excel(writer, sheet_name='Model_Comparison', index=False)
            print(f"  ✓ Exported model comparison data")

    exported_files.append(champion_export_path)
    print(f"\n✅ Champion & time unit info exported to: {os.path.basename(champion_export_path)}")

except Exception as e:
    print(f"\n⚠️ Error exporting champion model info: {e}")
    import traceback
    traceback.print_exc()

# ============================================================================
# SECTION 1: ENHANCED CROSS-VALIDATION DATA EXPORT
# ============================================================================
print("\n" + "=" * 80)
print("📈 SECTION 1: EXPORTING ENHANCED CROSS-VALIDATION DATA")
print("=" * 80)

if 'cv_results' in locals() and cv_results is not None:
    try:
        cv_export_path = os.path.join(export_folder, f"CV_Data_Enhanced_{timestamp}.xlsx")

        with pd.ExcelWriter(cv_export_path, engine='openpyxl') as writer:

            # Sheet 1: All fold-level scores
            if 'all_cv_scores' in cv_results:
                all_scores_df = pd.DataFrame({
                    'Fold_Index': range(1, len(cv_results['all_cv_scores']) + 1),
                    'C_Index': cv_results['all_cv_scores']
                })
                all_scores_df.to_excel(writer, sheet_name='All_Fold_Scores', index=False)
                print(f"  ✓ Exported {len(all_scores_df)} fold-level C-index scores")

            # Sheet 2: Repetition-level statistics
            if 'repetition_stats' in cv_results:
                rep_stats_df = pd.DataFrame(cv_results['repetition_stats'])
                rep_stats_df.to_excel(writer, sheet_name='Repetition_Statistics', index=False)
                print(f"  ✓ Exported {len(rep_stats_df)} repetition-level statistics")

            # Sheet 3: Scores per repetition (unpacked matrix)
            if 'scores_per_repetition' in cv_results:
                max_folds = max(len(scores) for scores in cv_results['scores_per_repetition'])
                scores_matrix = []
                for rep_idx, scores in enumerate(cv_results['scores_per_repetition'], 1):
                    row = {'Repetition': rep_idx}
                    for fold_idx, score in enumerate(scores, 1):
                        row[f'Fold_{fold_idx}'] = score
                    scores_matrix.append(row)

                scores_matrix_df = pd.DataFrame(scores_matrix)
                scores_matrix_df.to_excel(writer, sheet_name='Scores_Matrix', index=False)
                print(f"  ✓ Exported {len(scores_matrix_df)} x {max_folds} scores matrix")

            # Sheet 4: Overall CV summary (ENHANCED)
            cv_summary = pd.DataFrame({
                'Metric': [
                    'Mean C-index',
                    'Std C-index',
                    'Median C-index',
                    'Min C-index',
                    'Max C-index',
                    'CV%',
                    'Training C-index',
                    'Optimism',
                    'N_Repetitions',
                    'N_Folds_per_Rep',
                    'Total_Fits',
                    'CI_95_Lower',
                    'CI_95_Upper'
                ],
                'Value': [
                    cv_results.get('mean_c_index', np.nan),
                    cv_results.get('std_c_index', np.nan),
                    cv_results.get('median_c_index', np.nan),
                    cv_results.get('min_c_index', np.nan),
                    cv_results.get('max_c_index', np.nan),
                    cv_results.get('coefficient_of_variation', np.nan),
                    cv_results.get('train_c_index', np.nan),
                    cv_results.get('optimism', np.nan),
                    cv_results.get('n_repetitions', np.nan),
                    cv_results.get('n_folds', np.nan),
                    cv_results.get('n_successful_runs', np.nan),
                    np.percentile(cv_results['all_cv_scores'], 2.5) if 'all_cv_scores' in cv_results else np.nan,
                    np.percentile(cv_results['all_cv_scores'], 97.5) if 'all_cv_scores' in cv_results else np.nan
                ]
            })
            cv_summary.to_excel(writer, sheet_name='CV_Summary', index=False)
            print(f"  ✓ Exported enhanced CV summary")

            # Sheet 5: Variance decomposition (ENHANCED with ICC)
            if 'repetition_stats' in cv_results and 'scores_per_repetition' in cv_results:
                rep_means = [stats['mean'] for stats in cv_results['repetition_stats']]
                between_var = np.var(rep_means, ddof=1)
                within_var = np.mean([np.var(scores, ddof=1) for scores in cv_results['scores_per_repetition']])
                total_var = np.var(cv_results['all_cv_scores'], ddof=1)
                icc = between_var / (between_var + within_var) if (between_var + within_var) > 0 else np.nan

                variance_df = pd.DataFrame({
                    'Component': ['Between_Repetitions', 'Within_Repetitions', 'Total', 'ICC'],
                    'Variance': [between_var, within_var, total_var, icc],
                    'Percent_of_Total': [
                        100 * between_var / total_var if total_var > 0 else np.nan,
                        100 * within_var / total_var if total_var > 0 else np.nan,
                        100.0,
                        np.nan
                    ],
                    'Interpretation': [
                        'Variability between CV repetitions',
                        'Variability within CV repetitions (across folds)',
                        'Total variability',
                        f'Intraclass Correlation: {"High" if icc > 0.75 else "Moderate" if icc > 0.5 else "Low"} consistency'
                    ]
                })
                variance_df.to_excel(writer, sheet_name='Variance_Decomposition', index=False)
                print(f"  ✓ Exported variance decomposition with ICC={icc:.3f}")

            # Sheet 6: Cumulative convergence data
            if 'repetition_stats' in cv_results:
                rep_means = [stats['mean'] for stats in cv_results['repetition_stats']]
                cumulative_means = np.cumsum(rep_means) / np.arange(1, len(rep_means) + 1)

                convergence_df = pd.DataFrame({
                    'N_Repetitions': range(1, len(rep_means) + 1),
                    'Cumulative_Mean': cumulative_means,
                    'Distance_from_Final': np.abs(cumulative_means - cumulative_means[-1]),
                    'Within_1pct_Final': np.abs(cumulative_means - cumulative_means[-1]) < (0.01 * cumulative_means[-1])
                })
                convergence_df.to_excel(writer, sheet_name='Convergence_Analysis', index=False)
                print(f"  ✓ Exported convergence analysis")

        exported_files.append(cv_export_path)
        print(f"\n✅ Enhanced CV data exported to: {os.path.basename(cv_export_path)}")

    except Exception as e:
        print(f"\n⚠️ Error exporting CV data: {e}")
        import traceback
        traceback.print_exc()
else:
    print("\n⚠️ No CV results found to export")

# ============================================================================
# SECTION 2: ENHANCED DIAGNOSTICS/QC DATA EXPORT
# ============================================================================
print("\n" + "=" * 80)
print("📊 SECTION 2: EXPORTING ENHANCED DIAGNOSTICS/QC DATA")
print("=" * 80)

if 'mv_model' in locals() and mv_model is not None:
    try:
        qc_export_path = os.path.join(export_folder, f"Diagnostics_QC_Data_Enhanced_{timestamp}.xlsx")

        with pd.ExcelWriter(qc_export_path, engine='openpyxl') as writer:

            # Sheet 1: Model coefficients with full statistics
            if hasattr(mv_model, 'summary'):
                coef_df = mv_model.summary.copy()
                coef_df['HR'] = np.exp(coef_df['coef'])
                coef_df['HR_lower_95'] = np.exp(coef_df['coef lower 95%'])
                coef_df['HR_upper_95'] = np.exp(coef_df['coef upper 95%'])
                coef_df.to_excel(writer, sheet_name='Model_Coefficients')
                print(f"  ✓ Exported model coefficients ({len(coef_df)} variables)")

            # Sheet 2: VIF data with tolerance and interpretation
            if 'vif_data' in locals() and vif_data is not None:
                vif_enhanced = vif_data.copy()

                # Add tolerance if not present
                if 'Tolerance' not in vif_enhanced.columns and 'VIF' in vif_enhanced.columns:
                    vif_enhanced['Tolerance'] = 1 / vif_enhanced['VIF']

                # Add interpretation
                if 'Status' not in vif_enhanced.columns and 'VIF' in vif_enhanced.columns:
                    n_samples = len(df) if 'df' in locals() else 100
                    def interpret_vif(vif, n):
                        if np.isnan(vif) or np.isinf(vif):
                            return "Undefined"
                        if n < 50:
                            return "Acceptable" if vif < 10 else "Moderate" if vif < 20 else "High"
                        else:
                            return "Excellent" if vif < 5 else "Moderate" if vif < 10 else "High"
                    vif_enhanced['Status'] = vif_enhanced['VIF'].apply(lambda x: interpret_vif(x, n_samples))

                vif_enhanced.to_excel(writer, sheet_name='VIF_Multicollinearity', index=False)
                print(f"  ✓ Exported enhanced VIF/multicollinearity data")

            # Sheet 3: Correlation matrix
            if 'corr_matrix' in locals() and corr_matrix is not None:
                corr_matrix.to_excel(writer, sheet_name='Correlation_Matrix')

                # Pairwise correlations in long format with interpretation
                corr_pairs = []
                for i in range(len(corr_matrix)):
                    for j in range(i+1, len(corr_matrix)):
                        r = corr_matrix.iloc[i, j]
                        abs_r = abs(r)

                        # Interpretation
                        if abs_r > 0.7:
                            status = "High"
                        elif abs_r > 0.5:
                            status = "Moderate"
                        elif abs_r > 0.3:
                            status = "Low-Moderate"
                        else:
                            status = "Low"

                        corr_pairs.append({
                            'Variable_1': corr_matrix.index[i],
                            'Variable_2': corr_matrix.columns[j],
                            'Correlation': r,
                            'Abs_Correlation': abs_r,
                            'Status': status,
                            'Potential_Issue': 'Yes' if abs_r > 0.7 else 'No'
                        })

                if corr_pairs:
                    corr_pairs_df = pd.DataFrame(corr_pairs).sort_values('Abs_Correlation', ascending=False)
                    corr_pairs_df.to_excel(writer, sheet_name='Pairwise_Correlations', index=False)
                    print(f"  ✓ Exported correlation matrix and {len(corr_pairs_df)} pairwise correlations")

            # Sheet 4: Condition number analysis
            if 'condition_number' in locals() and condition_number is not None:
                cond_df = pd.DataFrame([{
                    'Condition_Number': condition_number,
                    'Interpretation': (
                        'Excellent' if condition_number < 10 else
                        'Moderate' if condition_number < 30 else
                        'High' if condition_number < 100 else
                        'Severe'
                    ),
                    'Multicollinearity_Risk': (
                        'None' if condition_number < 10 else
                        'Some' if condition_number < 30 else
                        'Significant' if condition_number < 100 else
                        'Very High'
                    )
                }])
                cond_df.to_excel(writer, sheet_name='Condition_Number', index=False)
                print(f"  ✓ Exported condition number analysis")

            # Sheet 5: Proportional hazards test results (all transforms)
            if 'ph_test_results' in locals() and ph_test_results:
                for transform, ph_df in ph_test_results.items():
                    sheet_name = f'PH_Test_{transform}'[:31]  # Excel sheet name limit

                    # Add interpretation column
                    ph_df_enhanced = ph_df.copy()
                    if 'p' in ph_df_enhanced.columns:
                        ph_df_enhanced['PH_Satisfied'] = ph_df_enhanced['p'].apply(
                            lambda p: 'Yes' if p > 0.05 else 'No' if p <= 0.05 else 'N/A'
                        )

                    ph_df_enhanced.to_excel(writer, sheet_name=sheet_name)
                print(f"  ✓ Exported {len(ph_test_results)} PH assumption tests")

            # Sheet 6: Model performance metrics (ENHANCED)
            model_metrics = pd.DataFrame({
                'Metric': [
                    'N_Observations',
                    'N_Events',
                    'N_Censored',
                    'Event_Rate_%',
                    'N_Covariates',
                    'Events_Per_Variable',
                    'C_Index_Training',
                    'C_Index_CV',
                    'Optimism',
                    'Log_Likelihood',
                    'AIC_partial',
                    'BIC_partial',
                    'Condition_Number',
                    'Max_VIF',
                    'Champion_Type'
                ],
                'Value': [
                    len(df) if 'df' in locals() else np.nan,
                    df[event_col].sum() if 'df' in locals() and 'event_col' in locals() else np.nan,
                    (len(df) - df[event_col].sum()) if 'df' in locals() and 'event_col' in locals() else np.nan,
                    (100 * df[event_col].sum() / len(df)) if 'df' in locals() and 'event_col' in locals() else np.nan,
                    len(mv_model.params_),
                    (df[event_col].sum() / len(mv_model.params_)) if 'df' in locals() and 'event_col' in locals() else np.nan,
                    mv_model.concordance_index_ if hasattr(mv_model, 'concordance_index_') else np.nan,
                    cv_results.get('mean_c_index', np.nan) if 'cv_results' in locals() and cv_results else np.nan,
                    cv_results.get('optimism', np.nan) if 'cv_results' in locals() and cv_results else np.nan,
                    mv_model.log_likelihood_ if hasattr(mv_model, 'log_likelihood_') else np.nan,
                    getattr(mv_model, 'AIC_partial_', np.nan) if hasattr(mv_model, 'AIC_partial_') else np.nan,
                    getattr(mv_model, 'BIC_partial_', np.nan) if hasattr(mv_model, 'BIC_partial_') else np.nan,
                    condition_number if 'condition_number' in locals() else np.nan,
                    vif_data['VIF'].max() if 'vif_data' in locals() and vif_data is not None and 'VIF' in vif_data.columns else np.nan,
                    champion_model.get('type', 'Unknown') if 'champion_model' in locals() and champion_model else 'Unknown'
                ],
                'Interpretation': [
                    'Total sample size',
                    'Number of events',
                    'Number of censored observations',
                    'Percentage of events',
                    'Number of predictor variables',
                    'EPV - should be ≥10',
                    'Training data performance',
                    'Cross-validated performance',
                    'Difference: Training - CV',
                    'Model fit quality',
                    'Akaike Information Criterion',
                    'Bayesian Information Criterion',
                    'Multicollinearity indicator',
                    'Maximum VIF across variables',
                    'Selected champion model'
                ]
            })
            model_metrics.to_excel(writer, sheet_name='Model_Metrics', index=False)
            print(f"  ✓ Exported enhanced model performance metrics")

            # Sheet 7: Sample size adequacy assessment
            if 'df' in locals() and 'event_col' in locals():
                n_obs = len(df)
                n_events = int(df[event_col].sum())
                n_vars = len(mv_model.params_)
                epv = n_events / n_vars

                adequacy_df = pd.DataFrame([{
                    'N_Observations': n_obs,
                    'N_Events': n_events,
                    'N_Variables': n_vars,
                    'EPV': epv,
                    'EPV_Status': 'Excellent' if epv >= 20 else 'Acceptable' if epv >= 10 else 'Borderline' if epv >= 5 else 'Underpowered',
                    'Recommended_Min_EPV': 10,
                    'Optimal_EPV': 20,
                    'Sample_Adequate': 'Yes' if epv >= 10 else 'Borderline' if epv >= 5 else 'No'
                }])
                adequacy_df.to_excel(writer, sheet_name='Sample_Size_Adequacy', index=False)
                print(f"  ✓ Exported sample size adequacy assessment (EPV={epv:.1f})")

        exported_files.append(qc_export_path)
        print(f"\n✅ Enhanced diagnostics/QC data exported to: {os.path.basename(qc_export_path)}")

    except Exception as e:
        print(f"\n⚠️ Error exporting diagnostics data: {e}")
        import traceback
        traceback.print_exc()
else:
    print("\n⚠️ No model found for diagnostics export")

# ============================================================================
# SECTION 3: ENHANCED RISK STRATIFICATION DATA EXPORT
# ============================================================================
print("\n" + "=" * 80)
print("📈 SECTION 3: EXPORTING ENHANCED RISK STRATIFICATION DATA")
print("=" * 80)

if 'df' in locals() and 'risk_group' in df.columns:
    try:
        risk_export_path = os.path.join(export_folder, f"Risk_Stratification_Data_Enhanced_{timestamp}.xlsx")

        with pd.ExcelWriter(risk_export_path, engine='openpyxl') as writer:

            # Sheet 1: Patient-level risk scores and assignments with time conversions
            risk_cols = ['risk_score', 'risk_group']
            if 'log_risk_score' in df.columns:
                risk_cols.append('log_risk_score')

            patient_risk_df = df[[survival_time_col, event_col] + risk_cols].copy()

            # Add time conversions if function exists
            if 'convert_to_months' in dir():
                patient_risk_df['survival_months'] = patient_risk_df[survival_time_col].apply(convert_to_months)
                patient_risk_df['original_time_unit'] = globals().get('original_time_unit', 'Unknown')

            patient_risk_df.to_excel(writer, sheet_name='Patient_Risk_Scores')
            print(f"  ✓ Exported {len(patient_risk_df)} patient risk scores with time conversions")

            # Sheet 2: Enhanced risk group statistics
            if 'group_stats_df' in locals():
                group_stats_enhanced = group_stats_df.copy()

                # Add additional derived metrics
                if 'N' in group_stats_enhanced.columns and 'Events' in group_stats_enhanced.columns:
                    group_stats_enhanced['Censored'] = group_stats_enhanced['N'] - group_stats_enhanced['Events']
                    group_stats_enhanced['Censoring_Rate_%'] = 100 * group_stats_enhanced['Censored'] / group_stats_enhanced['N']

                group_stats_enhanced.to_excel(writer, sheet_name='Group_Statistics', index=False)
                print(f"  ✓ Exported enhanced risk group statistics")

            # Sheet 3: Kaplan-Meier curve data points for each group (IN MONTHS)
            from lifelines import KaplanMeierFitter

            actual_groups = sorted(df['risk_group'].unique(),
                                 key=lambda x: df[df['risk_group']==x]['risk_score'].mean())

            km_data_all = []
            for group in actual_groups:
                group_data = df[df['risk_group'] == group]
                kmf = KaplanMeierFitter()
                kmf.fit(group_data[survival_time_col], group_data[event_col])

                # Extract survival function data
                sf = kmf.survival_function_
                km_data = sf.reset_index()
                km_data.columns = ['Time_Original_Unit', 'Survival_Probability']

                # Add time in months if conversion available
                if 'convert_to_months' in dir():
                    km_data['Time_Months'] = km_data['Time_Original_Unit'].apply(convert_to_months)

                km_data['Risk_Group'] = group
                km_data['N_at_Risk'] = len(group_data)
                km_data['N_Events'] = int(group_data[event_col].sum())
                km_data_all.append(km_data)

            if km_data_all:
                km_combined = pd.concat(km_data_all, ignore_index=True)
                km_combined.to_excel(writer, sheet_name='KM_Curve_Data', index=False)
                print(f"  ✓ Exported {len(km_combined)} KM curve data points with time conversions")

            # Sheet 4: Median survival times (both original and months)
            median_survival = []
            for group in actual_groups:
                group_data = df[df['risk_group'] == group]
                kmf = KaplanMeierFitter()
                kmf.fit(group_data[survival_time_col], group_data[event_col])

                median_original = group_data[survival_time_col].median()

                median_dict = {
                    'Risk_Group': group,
                    'Median_Survival_KM_Original': kmf.median_survival_time_,
                    'Median_Survival_Raw_Original': median_original,
                    'N': len(group_data),
                    'Events': int(group_data[event_col].sum()),
                    'Event_Rate_%': 100 * group_data[event_col].sum() / len(group_data)
                }

                # Add month conversions
                if 'convert_to_months' in dir():
                    median_dict['Median_Survival_KM_Months'] = convert_to_months(kmf.median_survival_time_) if not np.isnan(kmf.median_survival_time_) else np.nan
                    median_dict['Median_Survival_Raw_Months'] = convert_to_months(median_original)
                    median_dict['Original_Time_Unit'] = globals().get('original_time_unit', 'Unknown')

                median_survival.append(median_dict)

            median_df = pd.DataFrame(median_survival)
            median_df.to_excel(writer, sheet_name='Median_Survival_Times', index=False)
            print(f"  ✓ Exported median survival times with month conversions")

            # Sheet 5: Hazard ratios between groups
            if 'hr_df' in locals() and hr_df is not None:
                hr_df_enhanced = hr_df.copy()

                # Add clinical interpretation
                if 'HR' in hr_df_enhanced.columns:
                    def interpret_hr(hr, p_val):
                        if p_val >= 0.05:
                            return 'Not significant'
                        elif hr < 1:
                            return f'Protective ({1/hr:.2f}x lower hazard)'
                        elif hr > 2:
                            return f'High risk (>{hr:.1f}x hazard)'
                        else:
                            return f'Moderate risk ({hr:.2f}x hazard)'

                    hr_df_enhanced['Clinical_Interpretation'] = [
                        interpret_hr(row['HR'], row['p_value'])
                        for _, row in hr_df_enhanced.iterrows()
                    ]

                hr_df_enhanced.to_excel(writer, sheet_name='Hazard_Ratios', index=False)
                print(f"  ✓ Exported hazard ratios with clinical interpretation")

            # Sheet 6: Log-rank test results (overall)
            if 'results_overall' in locals() and results_overall is not None:
                logrank_df = pd.DataFrame({
                    'Test': ['Overall_LogRank'],
                    'Test_Statistic': [results_overall.test_statistic],
                    'p_value': [results_overall.p_value],
                    'Degrees_of_Freedom': [results_overall.degrees_of_freedom],
                    'Significant': ['Yes' if results_overall.p_value < 0.05 else 'No'],
                    'Interpretation': [
                        'Highly significant' if results_overall.p_value < 0.001 else
                        'Significant' if results_overall.p_value < 0.05 else
                        'Not significant'
                    ]
                })
                logrank_df.to_excel(writer, sheet_name='LogRank_Test', index=False)
                print(f"  ✓ Exported log-rank test results")

            # Sheet 7: Pairwise log-rank tests with Bonferroni correction
            if 'pairwise_df' in locals() and pairwise_df is not None:
                pairwise_enhanced = pairwise_df.copy()

                # Add Bonferroni correction
                n_comparisons = len(pairwise_enhanced)
                bonf_alpha = 0.05 / n_comparisons
                pairwise_enhanced['Bonferroni_Significant'] = pairwise_enhanced['p'] < bonf_alpha
                pairwise_enhanced['Bonferroni_Alpha'] = bonf_alpha

                pairwise_enhanced.to_excel(writer, sheet_name='Pairwise_LogRank')
                print(f"  ✓ Exported pairwise log-rank tests with Bonferroni correction")

            # Sheet 8: Risk score distribution by group (ENHANCED)
            risk_dist = []
            for group in actual_groups:
                group_data = df[df['risk_group'] == group]

                # Calculate percentiles
                percentiles = [5, 10, 25, 50, 75, 90, 95]
                perc_dict = {f'P{p}': group_data['risk_score'].quantile(p/100) for p in percentiles}

                risk_dict = {
                    'Risk_Group': group,
                    'N': len(group_data),
                    'Mean_Risk_Score': group_data['risk_score'].mean(),
                    'Median_Risk_Score': group_data['risk_score'].median(),
                    'SD_Risk_Score': group_data['risk_score'].std(),
                    'Min_Risk_Score': group_data['risk_score'].min(),
                    'Max_Risk_Score': group_data['risk_score'].max(),
                    'IQR_Risk_Score': group_data['risk_score'].quantile(0.75) - group_data['risk_score'].quantile(0.25),
                    'CV_percent': (group_data['risk_score'].std() / group_data['risk_score'].mean()) * 100
                }

                # Add percentiles
                risk_dict.update(perc_dict)

                risk_dist.append(risk_dict)

            risk_dist_df = pd.DataFrame(risk_dist)
            risk_dist_df.to_excel(writer, sheet_name='Risk_Score_Distribution', index=False)
            print(f"  ✓ Exported enhanced risk score distributions with percentiles")

            # Sheet 9: Time unit reference
            if 'original_time_unit' in globals():
                time_ref = pd.DataFrame([{
                    'Original_Unit': globals().get('original_time_unit', 'Unknown'),
                    'Display_Unit': 'months',
                    'Conversion_Factor': globals().get('conversion_factor', 1.0),
                    'Note': f'All time values provided in both {globals().get("original_time_unit", "original")} and months'
                }])
                time_ref.to_excel(writer, sheet_name='Time_Unit_Reference', index=False)
                print(f"  ✓ Exported time unit reference")

        exported_files.append(risk_export_path)
        print(f"\n✅ Enhanced risk stratification data exported to: {os.path.basename(risk_export_path)}")

    except Exception as e:
        print(f"\n⚠️ Error exporting risk stratification data: {e}")
        import traceback
        traceback.print_exc()
else:
    print("\n⚠️ No risk stratification results found to export")

# ============================================================================
# SECTION 4: MULTIVARIATE MODEL COMPARISON DATA (UV vs MV)
# ============================================================================
print("\n" + "=" * 80)
print("📊 SECTION 4: EXPORTING UNIVARIATE VS MULTIVARIATE COMPARISON")
print("=" * 80)

if 'multivariate_results' in locals() and 'cox_df' in locals():
    try:
        comparison_export_path = os.path.join(export_folder, f"UV_vs_MV_Comparison_Enhanced_{timestamp}.xlsx")

        with pd.ExcelWriter(comparison_export_path, engine='openpyxl') as writer:

            # Compile comparison data with enhanced metrics
            comparison_data = []
            for biomarker_result in multivariate_results['biomarkers']:
                biomarker_name = biomarker_result['name']
                uv_row = cox_df[cox_df['biomarker'] == biomarker_name]

                if len(uv_row) > 0:
                    uv_hr = float(uv_row['hr'].values[0])
                    mv_hr = biomarker_result['hr']
                    hr_change = mv_hr - uv_hr
                    hr_change_pct = 100 * hr_change / uv_hr

                    # Determine effect type
                    def classify_effect_change(uv_hr, mv_hr, uv_p, mv_p):
                        if uv_p >= 0.05 and mv_p >= 0.05:
                            return 'Not significant in either'
                        elif uv_p < 0.05 and mv_p >= 0.05:
                            return 'Lost significance (confounding)'
                        elif uv_p >= 0.05 and mv_p < 0.05:
                            return 'Gained significance (suppression)'
                        elif abs(hr_change_pct) < 10:
                            return 'Stable (independent effect)'
                        elif hr_change > 0:
                            return 'Effect increased (negative confounding)'
                        else:
                            return 'Effect decreased (positive confounding)'

                    uv_p = float(uv_row['p_value'].values[0])
                    mv_p = biomarker_result['p_value']

                    comparison_data.append({
                        'Biomarker': biomarker_name,
                        'UV_HR': uv_hr,
                        'UV_HR_lower': float(uv_row['hr_lower'].values[0]),
                        'UV_HR_upper': float(uv_row['hr_upper'].values[0]),
                        'UV_p_value': uv_p,
                        'UV_C_index': float(uv_row['concordance'].values[0]),
                        'MV_HR': mv_hr,
                        'MV_HR_lower': biomarker_result['hr_lower'],
                        'MV_HR_upper': biomarker_result['hr_upper'],
                        'MV_p_value': mv_p,
                        'MV_Coefficient': biomarker_result['coef'],
                        'MV_SE': biomarker_result['se'],
                        'MV_z_score': biomarker_result['z'],
                        'HR_Change': hr_change,
                        'HR_Change_Percent': hr_change_pct,
                        'Significant_UV': 'Yes' if uv_p < 0.05 else 'No',
                        'Significant_MV': 'Yes' if mv_p < 0.05 else 'No',
                        'Effect_Classification': classify_effect_change(uv_hr, mv_hr, uv_p, mv_p),
                        'Clinical_Relevance': 'High' if abs(hr_change_pct) > 25 else 'Moderate' if abs(hr_change_pct) > 10 else 'Low'
                    })

            if comparison_data:
                comparison_df = pd.DataFrame(comparison_data)
                comparison_df = comparison_df.sort_values('MV_p_value')
                comparison_df.to_excel(writer, sheet_name='UV_vs_MV_Comparison', index=False)
                print(f"  ✓ Exported enhanced UV vs MV comparison ({len(comparison_df)} biomarkers)")

                # Summary statistics
                summary_stats = pd.DataFrame([{
                    'Total_Biomarkers': len(comparison_df),
                    'Significant_UV_Only': sum((comparison_df['Significant_UV'] == 'Yes') & (comparison_df['Significant_MV'] == 'No')),
                    'Significant_MV_Only': sum((comparison_df['Significant_UV'] == 'No') & (comparison_df['Significant_MV'] == 'Yes')),
                    'Significant_Both': sum((comparison_df['Significant_UV'] == 'Yes') & (comparison_df['Significant_MV'] == 'Yes')),
                    'Significant_Neither': sum((comparison_df['Significant_UV'] == 'No') & (comparison_df['Significant_MV'] == 'No')),
                    'Mean_HR_Change_Percent': comparison_df['HR_Change_Percent'].mean(),
                    'Median_HR_Change_Percent': comparison_df['HR_Change_Percent'].median(),
                    'High_Clinical_Relevance': sum(comparison_df['Clinical_Relevance'] == 'High')
                }])
                summary_stats.to_excel(writer, sheet_name='Comparison_Summary', index=False)
                print(f"  ✓ Exported comparison summary statistics")

            # Add multivariate model summary
            mv_summary = pd.DataFrame({
                'Metric': [
                    'N_Variables',
                    'N_Samples',
                    'N_Events',
                    'Model_C_Index',
                    'Log_Likelihood',
                    'AIC',
                    'BIC',
                    'Champion_Type'
                ],
                'Value': [
                    len(multivariate_results['biomarkers']),
                    multivariate_results.get('n_samples', np.nan),
                    multivariate_results.get('n_events', np.nan),
                    multivariate_results.get('model_concordance', np.nan),
                    multivariate_results.get('log_likelihood', np.nan),
                    multivariate_results.get('AIC', np.nan),
                    multivariate_results.get('BIC', np.nan),
                    champion_model.get('type', 'Unknown') if 'champion_model' in locals() and champion_model else 'Unknown'
                ]
            })
            mv_summary.to_excel(writer, sheet_name='MV_Model_Summary', index=False)
            print(f"  ✓ Exported multivariate model summary")

        exported_files.append(comparison_export_path)
        print(f"\n✅ Enhanced UV vs MV comparison exported to: {os.path.basename(comparison_export_path)}")

    except Exception as e:
        print(f"\n⚠️ Error exporting comparison data: {e}")
        import traceback
        traceback.print_exc()
else:
    print("\n⚠️ No univariate/multivariate comparison data found")

# ============================================================================
# SECTION 5: MODEL COMPARISON DATA (EPV vs STEPWISE) - IF BOTH EXIST
# ============================================================================
print("\n" + "=" * 80)
print("📊 SECTION 5: EXPORTING EPV VS STEPWISE MODEL COMPARISON")
print("=" * 80)

if all(v in globals() for v in ['epv_results', 'stepwise_results_dict']):
    try:
        model_comp_path = os.path.join(export_folder, f"EPV_vs_Stepwise_Comparison_{timestamp}.xlsx")

        with pd.ExcelWriter(model_comp_path, engine='openpyxl') as writer:

            # Detailed model comparison
            epv_biomarkers_list = globals().get('epv_biomarkers_list', [])
            stepwise_biomarkers_list = globals().get('stepwise_biomarkers_list', [])

            comparison_metrics = pd.DataFrame({
                'Metric': [
                    'Number_of_Variables',
                    'Training_C_Index',
                    'CV_Mean_C_Index',
                    'CV_Std_C_Index',
                    'CV_Coefficient_of_Variation',
                    'Optimism',
                    'AIC',
                    'BIC',
                    'Log_Likelihood'
                ],
                'EPV_Model': [
                    len(epv_biomarkers_list),
                    epv_results.get('model_concordance', np.nan),
                    globals().get('epv_cv', {}).get('mean_c_index', np.nan) if 'epv_cv' in globals() else np.nan,
                    globals().get('epv_cv', {}).get('std_c_index', np.nan) if 'epv_cv' in globals() else np.nan,
                    globals().get('epv_cv', {}).get('coefficient_of_variation', np.nan) if 'epv_cv' in globals() else np.nan,
                    np.nan,  # Calculate from train and CV
                    epv_results.get('AIC', np.nan),
                    epv_results.get('BIC', np.nan),
                    epv_results.get('log_likelihood', np.nan)
                ],
                'Stepwise_Model': [
                    len(stepwise_biomarkers_list),
                    stepwise_results_dict.get('model_concordance', np.nan),
                    globals().get('stepwise_cv', {}).get('mean_c_index', np.nan) if 'stepwise_cv' in globals() else np.nan,
                    globals().get('stepwise_cv', {}).get('std_c_index', np.nan) if 'stepwise_cv' in globals() else np.nan,
                    globals().get('stepwise_cv', {}).get('coefficient_of_variation', np.nan) if 'stepwise_cv' in globals() else np.nan,
                    np.nan,  # Calculate from train and CV
                    stepwise_results_dict.get('AIC', np.nan),
                    stepwise_results_dict.get('BIC', np.nan),
                    stepwise_results_dict.get('log_likelihood', np.nan)
                ]
            })

            # Add difference column
            comparison_metrics['Difference'] = comparison_metrics['Stepwise_Model'] - comparison_metrics['EPV_Model']

            comparison_metrics.to_excel(writer, sheet_name='Model_Comparison', index=False)
            print(f"  ✓ Exported EPV vs Stepwise comparison metrics")

            # Variable overlap analysis
            epv_set = set(epv_biomarkers_list)
            stepwise_set = set(stepwise_biomarkers_list)

            overlap_stats = pd.DataFrame([{
                'EPV_Variables': len(epv_set),
                'Stepwise_Variables': len(stepwise_set),
                'Shared_Variables': len(epv_set & stepwise_set),
                'EPV_Only': len(epv_set - stepwise_set),
                'Stepwise_Only': len(stepwise_set - epv_set),
                'Overlap_Percent': 100 * len(epv_set & stepwise_set) / max(len(epv_set), len(stepwise_set))
            }])
            overlap_stats.to_excel(writer, sheet_name='Variable_Overlap', index=False)

            # Shared variables list
            shared_vars = pd.DataFrame({
                'Variable': sorted(list(epv_set & stepwise_set))
            })
            shared_vars.to_excel(writer, sheet_name='Shared_Variables', index=False)

            # EPV-only variables
            epv_only_vars = pd.DataFrame({
                'Variable': sorted(list(epv_set - stepwise_set))
            })
            epv_only_vars.to_excel(writer, sheet_name='EPV_Only_Variables', index=False)

            # Stepwise-only variables
            stepwise_only_vars = pd.DataFrame({
                'Variable': sorted(list(stepwise_set - epv_set))
            })
            stepwise_only_vars.to_excel(writer, sheet_name='Stepwise_Only_Variables', index=False)

            print(f"  ✓ Exported variable overlap analysis")
            print(f"    - Shared: {len(epv_set & stepwise_set)} variables")
            print(f"    - EPV only: {len(epv_set - stepwise_set)} variables")
            print(f"    - Stepwise only: {len(stepwise_set - epv_set)} variables")

        exported_files.append(model_comp_path)
        print(f"\n✅ EPV vs Stepwise comparison exported to: {os.path.basename(model_comp_path)}")

    except Exception as e:
        print(f"\n⚠️ Error exporting EPV vs Stepwise comparison: {e}")
        import traceback
        traceback.print_exc()
else:
    print("\n⚠️ Both EPV and Stepwise models not available for comparison")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 120)
print("✅ COMPREHENSIVE DATA EXPORT COMPLETE (UPDATED)!")
print("=" * 120)

if exported_files:
    print(f"\n📂 Successfully exported {len(exported_files)} files to: {export_folder}")
    print("\nExported files:")
    for i, filepath in enumerate(exported_files, 1):
        filename = os.path.basename(filepath)
        filesize = os.path.getsize(filepath) / 1024  # KB
        print(f"  {i}. {filename} ({filesize:.1f} KB)")

    print("\n💡 All plot data is now available in Excel format for:")
    print("   • Custom plotting in other software")
    print("   • Statistical verification")
    print("   • Supplementary materials")
    print("   • Data sharing and archiving")
    print("   • Time unit conversions (all data in months)")
    print("   • Champion model documentation")
    print("   • Enhanced diagnostics (VIF, condition number, ICC)")
    print("   • Model comparison (EPV vs Stepwise)")

    print("\n📋 NEW FEATURES IN THIS VERSION:")
    print("   ✓ Champion model type and selection rationale")
    print("   ✓ Time unit conversions (all outputs in months)")
    print("   ✓ Enhanced CV stability with ICC and variance decomposition")
    print("   ✓ Improved multicollinearity diagnostics (condition number, tolerance)")
    print("   ✓ EPV vs Stepwise model comparison")
    print("   ✓ Clinical interpretation for HR comparisons")
    print("   ✓ Sample size adequacy assessment")
    print("   ✓ Bonferroni-corrected pairwise comparisons")
else:
    print("\n⚠️ No files were exported - check if analyses have been completed")

print("\n" + "=" * 120)

# **OPTIONAL: ID tracked risk stratification**

# 8. Patient ID-Tracked Risk Stratification (Optional)

## Overview
Links individual patient IDs to their risk group assignments for clinical tracking.

### What this cell does:
1. **Patient Mapping**: Associates IDs with risk scores and groups
2. **Ranked Lists**: Patients sorted by risk within each group
3. **Visual Tables**: Publication-ready patient tables
4. **Export**: Patient-level data with identifiers

### Privacy Note:
Ensure appropriate data handling when exporting patient identifiers.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
import warnings
import os
from datetime import datetime
warnings.filterwarnings('ignore')

print("="*80)
print("🎯 RISK STRATIFICATION WITH PATIENT ID MAPPING (UPDATED LAYOUT)")
print("="*80)

# ============================================================================
# SETUP EXPORT FOLDER STRUCTURE
# ============================================================================
print("\n📁 Setting up export folder...")

# Create timestamp for file naming
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Setup export folder (same structure as comprehensive export)
if 'RESULTS_FOLDER' in globals():
    export_folder = os.path.join(RESULTS_FOLDER, "Exported_Plot_Data")
else:
    # If RESULTS_FOLDER not defined, use current directory
    export_folder = os.path.join(os.getcwd(), "Risk_Stratification_Results")

os.makedirs(export_folder, exist_ok=True)
print(f"   ✅ Export folder: {export_folder}")

exported_files = []

# ============================================================================
# STEP 1: VERIFY REQUIRED DATA
# ============================================================================
print("\n📊 Verifying required data...")

required_vars = ['df', 'risk_score', 'risk_group', 'survival_time_col', 'event_col']
missing = []

for var in required_vars:
    if var in ['risk_score', 'risk_group']:
        if var not in df.columns:
            missing.append(f"df['{var}']")
    elif var not in globals():
        missing.append(var)

if missing:
    print(f"❌ Missing required variables: {missing}")
    print("   Please run the risk stratification analysis first!")
else:
    print("✅ All required data available")

    # ====================================================================
    # STEP 2: EXTRACT PATIENT IDs
    # ====================================================================
    print("\n📋 Extracting patient identifiers...")

    # Check for patient ID column
    patient_id_col = None
    # FIXE<data_path> Added lowercase versions to match actual data format
    possible_id_cols = ['patient_ID', 'Patient_ID', 'patient_id', 'ID', 'id', 'PatientID',
                       'Sample_ID', 'sample_id', 'SampleID']

    for col in possible_id_cols:
        if col in df.columns:
            patient_id_col = col
            # CRITICAL: Convert to string to preserve alphanumeric IDs (e.g., "G9", "F20")
            df[patient_id_col] = df[patient_id_col].astype(str)
            print(f"   ✅ Found patient ID column: '{patient_id_col}'")
            break

    if patient_id_col is None:
        # Use dataframe index as patient ID
        df['Patient_ID'] = df.index.astype(str)
        patient_id_col = 'Patient_ID'
        print(f"   ⚠️ No ID column found, using dataframe index as Patient_ID")

    # Get time unit
    if 'time_unit' in globals():
        time_unit_display = time_unit
    else:
        time_unit_display = "days"
    print(f"   ✅ Time unit: {time_unit_display}")

    # ====================================================================
    # STEP 3: CREATE PATIENT-LEVEL RISK MAPPING
    # ====================================================================
    print("\n" + "="*80)
    print("📊 CREATING PATIENT-LEVEL RISK MAPPING")
    print("="*80)

    # Create comprehensive patient mapping
    patient_risk_mapping = df[[patient_id_col, 'risk_group', 'risk_score',
                               survival_time_col, event_col]].copy()

    # CRITICAL: Ensure patient IDs are treated as strings to preserve letters+numbers
    patient_risk_mapping[patient_id_col] = patient_risk_mapping[patient_id_col].astype(str)

    # Add risk rank within each group
    patient_risk_mapping['risk_rank_overall'] = patient_risk_mapping['risk_score'].rank(ascending=False)

    # Add risk rank within each risk group
    patient_risk_mapping['risk_rank_within_group'] = patient_risk_mapping.groupby('risk_group')['risk_score'].rank(ascending=False)

    # Sort by risk group and then by risk score
    patient_risk_mapping = patient_risk_mapping.sort_values(['risk_group', 'risk_score'],
                                                            ascending=[True, False])

    # Get unique groups
    risk_groups = sorted(patient_risk_mapping['risk_group'].unique(),
                        key=lambda x: patient_risk_mapping[patient_risk_mapping['risk_group']==x]['risk_score'].mean())

    # Count patients per group
    print("\n📈 Patient Distribution by Risk Group:")
    for group in risk_groups:
        group_data = patient_risk_mapping[patient_risk_mapping['risk_group'] == group]
        n_patients = len(group_data)
        n_events = group_data[event_col].sum()
        print(f"  • {group}: {n_patients} patients ({int(n_events)} events, {n_events/n_patients*100:.1f}%)")

    # ====================================================================
    # STEP 4: PUBLICATION-READY VISUALIZATION - PATIENT ID HEATMAP
    # ====================================================================
    print("\n" + "="*80)
    print("📊 GENERATING PUBLICATION-READY PATIENT ID VISUALIZATION")
    print("="*80)

    # Determine figure size based on number of patients
    n_patients_total = len(patient_risk_mapping)
    fig_height = max(10, n_patients_total * 0.25)
    fig_width = 16

    fig = plt.figure(figsize=(fig_width, fig_height))

    # Create grid spec for better layout control
    gs = fig.add_gridspec(1, 4, width_ratios=[3, 2, 2, 2], wspace=0.4)

    ax1 = fig.add_subplot(gs[0, 0])  # Patient IDs by group
    ax2 = fig.add_subplot(gs[0, 1])  # Risk score distribution
    ax3 = fig.add_subplot(gs[0, 2])  # Survival time
    ax4 = fig.add_subplot(gs[0, 3])  # Event status

    # ----------------------------------------------------------------
    # PANEL 1: Patient IDs organized by Risk Group
    # ----------------------------------------------------------------
    print("\n🎨 Creating Panel 1: Patient ID mapping...")

    # Color scheme
    if len(risk_groups) == 3:
        colors = {'Low Risk': '#2ecc71',
                 'Intermediate Risk': '#f39c12',
                 'High Risk': '#e74c3c'}
    elif len(risk_groups) == 4:
        colors = {'Low Risk': '#2ecc71',
                 'Intermediate Risk': '#f1c40f',
                 'High Risk': '#e67e22',
                 'Very High Risk': '#e74c3c'}
    else:
        palette = sns.color_palette("RdYlGn_r", n_colors=len(risk_groups))
        colors = {group: palette[i] for i, group in enumerate(risk_groups)}

    # Reset index for plotting
    patient_risk_mapping_plot = patient_risk_mapping.reset_index(drop=True)

    y_pos = 0
    y_ticks = []
    y_labels = []
    group_boundaries = []

    for group_idx, group in enumerate(risk_groups):
        group_data = patient_risk_mapping_plot[patient_risk_mapping_plot['risk_group'] == group]

        # Draw group header
        group_start = y_pos

        for idx, (_, row) in enumerate(group_data.iterrows()):
            patient_id = row[patient_id_col]
            risk_score = row['risk_score']
            is_event = row[event_col] == 1

            # Draw colored bar for this patient
            rect = Rectangle((0, y_pos), 1, 0.8,
                           facecolor=colors[group],
                           edgecolor='black',
                           linewidth=1.5,
                           alpha=0.7)
            ax1.add_patch(rect)

            # Add patient ID text - ENSURE FULL ID IS USED
            text_color = 'white' if group != 'Low Risk' else 'black'
            ax1.text(0.5, y_pos + 0.4, str(patient_id),
                    ha='center', va='center',
                    fontsize=8, fontweight='bold',
                    color=text_color)

            # Add event marker
            if is_event:
                ax1.plot(0.9, y_pos + 0.4, 'X',
                        color='darkred', markersize=8,
                        markeredgewidth=2, markeredgecolor='white')

            y_ticks.append(y_pos + 0.4)
            y_labels.append(f"{patient_id}")
            y_pos += 1

        group_end = y_pos
        group_boundaries.append((group_start, group_end, group))

        # Add spacing between groups
        y_pos += 0.5

    # Add group labels and boundaries
    for start, end, group in group_boundaries:
        mid = (start + end) / 2
        ax1.text(-0.3, mid, f'{group}\n({int(end-start)} pts)',
                ha='right', va='center',
                fontsize=11, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.5',
                         facecolor=colors[group],
                         edgecolor='black',
                         linewidth=2, alpha=0.8))

        # Draw boundary lines
        if end < y_pos - 0.5:
            ax1.axhline(y=end, color='black', linewidth=2, linestyle='-')

    ax1.set_xlim(-0.5, 1.2)
    ax1.set_ylim(-0.5, y_pos)
    ax1.set_title('Patient ID by Risk Group',
                 fontsize=13, fontweight='bold', pad=15)
    ax1.axis('off')

    # MODIFIE<data_path> Add legend for event marker BELOW the patient column
    # Position it at the bottom of the plot
    ax1.plot([], [], 'X', color='darkred', markersize=8,
            markeredgewidth=2, markeredgecolor='white',
            label='X = Event occurred')
    ax1.legend(loc='lower center', fontsize=9, frameon=True,
              bbox_to_anchor=(0.5, -0.05))

    # ----------------------------------------------------------------
    # PANEL 2: Risk Score by Group
    # ----------------------------------------------------------------
    print("🎨 Creating Panel 2: Risk score distribution...")

    risk_scores_by_group = []
    for group in risk_groups:
        group_scores = patient_risk_mapping[patient_risk_mapping['risk_group'] == group]['risk_score'].values
        risk_scores_by_group.append(group_scores)

    bp = ax2.boxplot(risk_scores_by_group,
                     labels=risk_groups,
                     patch_artist=True,
                     widths=0.6,
                     notch=True,
                     showmeans=True,
                     meanprops=dict(marker='D', markerfacecolor='red',
                                  markersize=8, markeredgecolor='black'))

    # Color the boxes
    for patch, group in zip(bp['boxes'], risk_groups):
        patch.set_facecolor(colors[group])
        patch.set_alpha(0.7)
        patch.set_edgecolor('black')
        patch.set_linewidth(2)

    # Add individual points
    for i, (group, scores) in enumerate(zip(risk_groups, risk_scores_by_group)):
        y = scores
        x = np.random.normal(i+1, 0.04, size=len(y))
        ax2.scatter(x, y, alpha=0.5, s=30, color='black', zorder=3)

    ax2.set_ylabel('Risk Score', fontsize=11, fontweight='bold')
    ax2.set_title('Risk Score Distribution', fontsize=12, fontweight='bold', pad=10)
    ax2.grid(True, alpha=0.3, axis='y')
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=9)

    # ----------------------------------------------------------------
    # PANEL 3: Survival Time by Group
    # ----------------------------------------------------------------
    print("🎨 Creating Panel 3: Survival time distribution...")

    survival_by_group = []
    for group in risk_groups:
        group_survival = patient_risk_mapping[patient_risk_mapping['risk_group'] == group][survival_time_col].values
        survival_by_group.append(group_survival)

    bp2 = ax3.boxplot(survival_by_group,
                      labels=risk_groups,
                      patch_artist=True,
                      widths=0.6,
                      notch=True,
                      showmeans=True,
                      meanprops=dict(marker='D', markerfacecolor='red',
                                   markersize=8, markeredgecolor='black'))

    for patch, group in zip(bp2['boxes'], risk_groups):
        patch.set_facecolor(colors[group])
        patch.set_alpha(0.7)
        patch.set_edgecolor('black')
        patch.set_linewidth(2)

    # Add individual points with event status
    for i, group in enumerate(risk_groups):
        group_data = patient_risk_mapping[patient_risk_mapping['risk_group'] == group]
        events = group_data[group_data[event_col] == 1]
        censored = group_data[group_data[event_col] == 0]

        if len(events) > 0:
            x_events = np.random.normal(i+1, 0.04, size=len(events))
            ax3.scatter(x_events, events[survival_time_col],
                       alpha=0.6, s=40, color='darkred',
                       marker='o', edgecolors='black',
                       linewidths=1, zorder=3, label='Event' if i==0 else '')

        if len(censored) > 0:
            x_censored = np.random.normal(i+1, 0.04, size=len(censored))
            ax3.scatter(x_censored, censored[survival_time_col],
                       alpha=0.6, s=40, color='lightblue',
                       marker='^', edgecolors='black',
                       linewidths=1, zorder=3, label='Censored' if i==0 else '')

    ax3.set_ylabel(f'Survival Time ({time_unit_display})', fontsize=11, fontweight='bold')
    ax3.set_title('Survival Time Distribution', fontsize=12, fontweight='bold', pad=10)
    ax3.grid(True, alpha=0.3, axis='y')
    ax3.legend(loc='best', fontsize=8)
    plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=9)

    # ----------------------------------------------------------------
    # PANEL 4: Event Status Summary
    # ----------------------------------------------------------------
    print("🎨 Creating Panel 4: Event status summary...")

    event_data = []
    for group in risk_groups:
        group_data = patient_risk_mapping[patient_risk_mapping['risk_group'] == group]
        n_events = (group_data[event_col] == 1).sum()
        n_censored = (group_data[event_col] == 0).sum()
        event_data.append([n_events, n_censored])

    event_df = pd.DataFrame(event_data,
                           columns=['Events', 'Censored'],
                           index=risk_groups)

    event_df.plot(kind='barh', stacked=True, ax=ax4,
                 color=['#e74c3c', '#3498db'],
                 edgecolor='black', linewidth=1.5, alpha=0.8)

    ax4.set_xlabel('Number of Patients', fontsize=11, fontweight='bold')
    ax4.set_title('Event Status by Group', fontsize=12, fontweight='bold', pad=10)
    ax4.legend(loc='best', fontsize=9, frameon=True)
    ax4.grid(True, alpha=0.3, axis='x')

    # Add value labels on bars
    for i, group in enumerate(risk_groups):
        n_events = event_df.loc[group, 'Events']
        n_censored = event_df.loc[group, 'Censored']
        total = n_events + n_censored

        ax4.text(n_events/2, i, f'{int(n_events)}',
                ha='center', va='center', fontsize=9,
                fontweight='bold', color='white')
        ax4.text(n_events + n_censored/2, i, f'{int(n_censored)}',
                ha='center', va='center', fontsize=9,
                fontweight='bold', color='white')

    plt.suptitle('Risk Stratification: Patient-Level Mapping and Group Characteristics',
                fontsize=15, fontweight='bold', y=0.995)

    plt.tight_layout()

    # Save to export folder with timestamp
    fig1_filename = f'risk_stratification_patient_id_mapping_{timestamp}.png'
    fig1_path = os.path.join(export_folder, fig1_filename)
    plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
    plt.show()

    exported_files.append(fig1_path)
    print(f"\n✅ Saved: {fig1_filename}")
    print(f"   Location: {fig1_path}")

    # ====================================================================
    # STEP 5: DETAILED PATIENT LIST TABLE (PUBLICATION-READY)
    # ====================================================================
    print("\n" + "="*80)
    print("📊 CREATING DETAILED PATIENT LIST TABLE (UPDATED LAYOUT)")
    print("="*80)

    fig, axes = plt.subplots(len(risk_groups), 1,
                            figsize=(16, 3.5*len(risk_groups)))

    if len(risk_groups) == 1:
        axes = [axes]

    for ax_idx, (ax, group) in enumerate(zip(axes, risk_groups)):
        group_data = patient_risk_mapping[patient_risk_mapping['risk_group'] == group].copy()

        # Prepare table data
        table_data = []
        for idx, (_, row) in enumerate(group_data.iterrows(), 1):
            patient_id = row[patient_id_col]
            risk_score = row['risk_score']
            surv_time = row[survival_time_col]
            event_status = 'Event' if row[event_col] == 1 else 'Censored'

            table_data.append([
                idx,
                patient_id,
                f"{risk_score:.4f}",
                f"{surv_time:.1f}",
                event_status
            ])

        # MODIFIE<data_path> Removed title above table as requested
        # Create table with more vertical space since title is removed
        table = ax.table(cellText=table_data,
                        colLabels=['Rank', 'Patient ID', 'Risk Score',
                                  f'Survival ({time_unit_display})', 'Status'],
                        cellLoc='center',
                        loc='center',
                        bbox=[0.05, 0.05, 0.72, 0.90],  # [left, bottom, width, height] - increased height
                        colWidths=[0.10, 0.25, 0.25, 0.25, 0.15])

        table.auto_set_font_size(False)
        table.set_fontsize(10)  # Increased from 9 to 10
        table.scale(1, 2)

        # Style header
        for i in range(5):
            cell = table[(0, i)]
            cell.set_facecolor(colors[group])
            cell.set_text_props(weight='bold', color='white', fontsize=11)  # Increased from 10 to 11
            cell.set_edgecolor('black')
            cell.set_linewidth(2)

        # Style data rows
        for i in range(1, len(table_data) + 1):
            for j in range(5):
                cell = table[(i, j)]
                if i % 2 == 0:
                    cell.set_facecolor('#f0f0f0')
                else:
                    cell.set_facecolor('white')
                cell.set_edgecolor('gray')
                cell.set_linewidth(0.5)

                # Highlight events
                if j == 4 and table_data[i-1][4] == 'Event':
                    cell.set_text_props(weight='bold', color='#e74c3c')

        # MODIFIE<data_path> Add risk group label on the RIGHT side at the same Y-position
        # Position label on the right side, vertically centered with the table
        label_x = 0.82  # Right side position
        label_y = 0.45  # Middle of subplot

        ax.text(label_x, label_y, group,
               transform=ax.transAxes,
               fontsize=14, fontweight='bold',
               ha='left', va='center',
               rotation=0,
               bbox=dict(boxstyle='round,pad=1.0',
                        facecolor=colors[group],
                        edgecolor='black', linewidth=3, alpha=0.9))

        ax.axis('off')

    plt.suptitle('Complete Patient List by Risk Group',
                fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()

    # Save to export folder with timestamp
    fig2_filename = f'risk_stratification_patient_list_tables_{timestamp}.png'
    fig2_path = os.path.join(export_folder, fig2_filename)
    plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
    plt.show()

    exported_files.append(fig2_path)
    print(f"\n✅ Saved: {fig2_filename}")
    print(f"   Location: {fig2_path}")

    # ====================================================================
    # STEP 6: EXPORT TO EXCEL
    # ====================================================================
    print("\n" + "="*80)
    print("💾 EXPORTING PATIENT-LEVEL DATA TO EXCEL")
    print("="*80)

    try:
        # Excel file with timestamp
        excel_filename = f'risk_stratification_patient_mapping_{timestamp}.xlsx'
        excel_file = os.path.join(export_folder, excel_filename)

        with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:

            # Sheet 1: Complete patient mapping
            export_full = patient_risk_mapping.copy()
            export_full = export_full.rename(columns={
                patient_id_col: 'Patient_ID',
                survival_time_col: f'Survival_Time_{time_unit_display}'
            })
            export_full.to_excel(writer, sheet_name='All_Patients', index=False)
            print(f"   ✅ Sheet 1: All_Patients ({len(export_full)} patients)")

            # Sheet 2-N: One sheet per risk group
            for group in risk_groups:
                group_data = patient_risk_mapping[patient_risk_mapping['risk_group'] == group].copy()
                group_data = group_data.rename(columns={
                    patient_id_col: 'Patient_ID',
                    survival_time_col: f'Survival_Time_{time_unit_display}'
                })

                sheet_name = group.replace(' ', '_')[:31]  # Excel limit
                group_data.to_excel(writer, sheet_name=sheet_name, index=False)
                print(f"   ✅ Sheet: {sheet_name} ({len(group_data)} patients)")

            # Summary sheet
            summary_data = []
            for group in risk_groups:
                group_data = patient_risk_mapping[patient_risk_mapping['risk_group'] == group]

                summary_data.append({
                    'Risk_Group': group,
                    'N_Patients': len(group_data),
                    'N_Events': int(group_data[event_col].sum()),
                    'N_Censored': int((group_data[event_col] == 0).sum()),
                    'Event_Rate_%': f"{group_data[event_col].mean()*100:.1f}",
                    'Mean_Risk_Score': group_data['risk_score'].mean(),
                    'Median_Risk_Score': group_data['risk_score'].median(),
                    'Risk_Score_Range': f"[{group_data['risk_score'].min():.3f}, {group_data['risk_score'].max():.3f}]",
                    f'Mean_Survival_{time_unit_display}': group_data[survival_time_col].mean(),
                    f'Median_Survival_{time_unit_display}': group_data[survival_time_col].median(),
                    'Patient_IDs': ', '.join([str(x) for x in group_data[patient_id_col].values])
                })

            summary_df = pd.DataFrame(summary_data)
            summary_df.to_excel(writer, sheet_name='Summary', index=False)
            print(f"   ✅ Sheet: Summary")

            # Metadata sheet
            metadata = pd.DataFrame([{
                'Analysis_Date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
                'Total_Patients': len(patient_risk_mapping),
                'N_Risk_Groups': len(risk_groups),
                'Time_Unit': time_unit_display,
                'Survival_Column': survival_time_col,
                'Event_Column': event_col,
                'Patient_ID_Column': patient_id_col
            }])
            metadata.to_excel(writer, sheet_name='Metadata', index=False)
            print(f"   ✅ Sheet: Metadata")

        exported_files.append(excel_file)
        print(f"\n✅ Excel file saved: {excel_filename}")
        print(f"   Location: {excel_file}")

        # Also save as CSV for easy access
        csv_filename = f'risk_stratification_patient_mapping_{timestamp}.csv'
        csv_file = os.path.join(export_folder, csv_filename)
        export_full_csv = patient_risk_mapping.copy()
        export_full_csv = export_full_csv.rename(columns={
            patient_id_col: 'Patient_ID',
            survival_time_col: f'Survival_Time_{time_unit_display}'
        })
        export_full_csv.to_csv(csv_file, index=False)

        exported_files.append(csv_file)
        print(f"✅ CSV file saved: {csv_filename}")
        print(f"   Location: {csv_file}")

    except Exception as e:
        print(f"❌ Error during export: {e}")
        import traceback
        traceback.print_exc()

    # ====================================================================
    # STEP 7: SUMMARY REPORT
    # ====================================================================
    print("\n" + "="*80)
    print("📊 PATIENT-LEVEL RISK STRATIFICATION SUMMARY")
    print("="*80)

    print(f"\n✅ Total patients analyzed: {len(patient_risk_mapping)}")
    print(f"✅ Risk groups: {len(risk_groups)}")
    print(f"✅ Time unit: {time_unit_display}")

    print(f"\n📋 Patient Distribution:")
    for group in risk_groups:
        group_data = patient_risk_mapping[patient_risk_mapping['risk_group'] == group]
        n_patients = len(group_data)
        n_events = int(group_data[event_col].sum())
        mean_risk = group_data['risk_score'].mean()
        median_surv = group_data[survival_time_col].median()

        print(f"\n{group}:")
        print(f"  • Patients: {n_patients} ({n_patients/len(patient_risk_mapping)*100:.1f}%)")
        print(f"  • Events: {n_events} ({n_events/n_patients*100:.1f}%)")
        print(f"  • Mean risk score: {mean_risk:.4f}")
        print(f"  • Median survival: {median_surv:.1f} {time_unit_display}")

        # Show first 5 patient IDs
        patient_ids = group_data[patient_id_col].values[:5]
        if len(group_data) > 5:
            print(f"  • Sample IDs: {', '.join([str(x) for x in patient_ids])}, ... (+{len(group_data)-5} more)")
        else:
            print(f"  • Patient IDs: {', '.join([str(x) for x in patient_ids])}")

    # ====================================================================
    # FINAL EXPORT SUMMARY
    # ====================================================================
    print("\n" + "="*80)
    print("✅ PATIENT ID MAPPING COMPLETE!")
    print("="*80)

    if exported_files:
        print(f"\n📂 Successfully exported {len(exported_files)} files to:")
        print(f"   {export_folder}")

        print("\n📁 Exported files:")
        for i, filepath in enumerate(exported_files, 1):
            filename = os.path.basename(filepath)
            filesize = os.path.getsize(filepath) / 1024  # KB
            print(f"  {i}. {filename} ({filesize:.1f} KB)")

        print("\n💡 Files include:")
        print("   • High-resolution visualizations (PNG, 300 DPI)")
        print("   • Patient-level risk mappings (Excel, CSV)")
        print("   • Complete risk group statistics")
        print("   • Patient ID preservation (alphanumeric)")
        print("   • Publication-ready layouts")

        print(f"\n⏰ Export timestamp: {timestamp}")
    else:
        print("\n⚠️ No files were exported")

    print("\n" + "="*80)